# Multimodal Human Activity Recognition on MMAct (Kaggle GPU Pipeline)
## Fast, Reliable, Complete 6-Model Benchmark: Sensor-Only, Multimodal Fusion, MobileNetV2 Staged Fine-Tuning, Improved CNN, Tuned CNN, and Original CNN

This notebook executes the complete Multimodal Human Activity Recognition (HAR) pipeline on the **MMAct dataset** across structured, bounded sections:
- **Section 1**: Environment & Dependencies Setup (strictly preserving Kaggle's binary C ABI stack, Torchvision ops, and CPU thread configuration)
- **Section 2**: Preflight Hardware & GPU Diagnostics (CUDA device, memory headroom, CPU, RAM, disk, fail-loud GPU check, `runtime_hardware_manifest.json`)
- **Section 3**: Repository Checkout from `master` branch
- **Section 4**: Fast, Idempotent Dataset Discovery & Before-and-After Staging (generates `outputs/manifests/dataset_expansion_manifest.json`)
- **Section 5**: Label Mapping & Pre-Flight Modality Audit (Zero-Leakage Split, Paired File Audit & Representation Audit across 10 target classes)
- **Section 6**: Video & Sensor Preprocessing Cache Warm-Up (`/kaggle/working/har_mmact_cache` with disk capacity safety check)
- **Section 7**: Memory-Safe Calibration & OOM Warm-Up (graceful OOM catch, headroom tracking)
- **Section 8**: Experiment 1: Sensor-Only InceptionTime Baseline (50 epochs)
- **Section 9**: Experiment 2: Multimodal Late Fusion (MobileNetV2 + InceptionTime, 50 epochs)
- **Section 10**: Experiment 3: MobileNetV2 Video-Only Staged Fine-Tuning (50 epochs, AMP)
- **Section 11**: Experiment 4: Improved Regularized CNN Video Baseline (50 epochs, spatial dropout, weight decay, cosine annealing)
- **Section 12**: Experiment 5: Hyperparameter Tuning (train/val only) & Tuned CNN (50 epochs)
- **Section 13**: Experiment 6: Original Custom CNN Baseline (50 epochs, unregularized baseline)
- **Section 14**: Standalone Evaluation on Held-Out Test Set across all 6 models
- **Section 15**: Comprehensive Publication Visualizations & Galleries (annotated curves, LR schedules, generalization gap, ROC, PR, confusion matrices)
- **Section 16**: Sample Multimodal Inference Demonstration
- **Section 17**: Consolidated Summary Table, Manifests, Runtime Report & 1-Click Packaging (`har_mmact_results.zip` with file inventory)
- **Section 18**: (Optional) Extended Benchmarks (100 and 150 Epochs with runtime feasibility estimator)

All experiments evaluate on the **10 verified target activity classes**:
`crouching`, `jumping`, `kicking`, `pointing`, `running`, `sitting`, `standing`, `using_phone`, `walking`, `waving_hand`.

---
### Kaggle Setup Instructions:
1. **Enable GPU Accelerator:** Under **Session Options** -> **Accelerator**, select **GPU T4 x2** or **GPU P100**.
2. **Enable Internet Access:** Under **Session Options** -> **Internet**, toggle to **On** (required for GitHub repo checkout).
3. **Attach MMAct Dataset:** Click **+ Add Input** and attach your MMAct dataset (e.g. `mmact-trimmed`).


# Senior ML engineering audit — findings and controls

## Verified notebook-level issues corrected here
1. **Run-All fragility:** the uploaded notebook interleaved exploratory/debug cells with the training pipeline. Some checkpoint-inspection cells assumed a checkpoint already existed, and several cells repeated source inspection or inference setup. Redundant exploratory cells were removed; the main training/evaluation sequence is retained.
2. **Stale artifact risk:** fixed output directories plus `--skip_if_completed` could reuse old checkpoints/metrics. Existing `outputs/` is now preserved under a timestamped path and a fresh output directory is used for this run.
3. **Stale preprocessing cache risk:** the source audit found the cache key includes video metadata but does not include sensor-file metadata/content. A new per-run cache directory is created so this run does not silently reuse old sensor cache entries.
4. **Inference path handling:** the original inference cell used shell interpolation for file paths. It is replaced with `subprocess.run([...])`, which safely handles spaces and checks the exit code.
5. **Mislabelled comparison:** the Section 15 plot compares independently trained unimodal and multimodal models. It is labelled as a modality-model comparison, not a controlled same-model input ablation.

## Pipeline risks that require source-level experiments, not blind edits
- **Required-sensor filtering:** the pipeline rejects samples missing/empty required sensor CSVs. This is appropriate for a fair multimodal paired subset, but it discards otherwise usable video-only records. Do not silently call this zero data loss. The notebook reports/rejects invalid paired samples rather than fabricating sensor values.
- **CSV validation:** checking file existence, byte size, and basic CSV parseability does not prove valid timestamps, expected channel names, sufficient temporal coverage, or synchronization. Cache warm-up is the next runtime gate; deeper sensor schema/time validation remains a source-code improvement candidate.
- **Temporal alignment:** exact timestamp joins/interpolation and the fixed-window strategy may discard or distort samples when modality clocks differ. Validate timestamp units, duplicate handling, drift, and alignment tolerance before changing the loader.
- **Normalization:** fit any learned scaling statistics on training subjects only, then apply unchanged to validation/test. Never normalize using all subjects.
- **Architecture:** the weak custom CNN test results indicate a poor video baseline, not proof that more epochs will fix it. Prioritize pretrained video backbones, augmentation, class-balanced sampling/loss, and temporal modeling as separate validation-only ablations.
- **Hyperparameter search:** the existing tuning stage is small (four short trials). Expand it only on train/validation subjects; keep the test split untouched until final evaluation.
- **Split protocol:** this notebook's subject split (subjects 1–14 train, 15–17 validation, 18–20 test) is a project-specific protocol, not the paper's 1–16 / 17–20 cross-subject protocol. Do not compare scores directly with the paper.

## Model selection and reporting rules
- Select checkpoints using validation Macro-F1 (and track accuracy/loss); evaluate the test split once after model selection.
- Report actual metrics only. Do not promise a maximum accuracy or zero loss; non-zero loss is normal for classification.
- The six existing model definitions and 50-epoch settings are preserved as baseline experiments. This notebook does not claim that the architecture/hyperparameter changes have already improved accuracy; that requires measured validation/test results.
- The optional 100/150-epoch section remains disabled by default.

## Run instructions
Attach both MMAct input datasets in Kaggle, select a GPU accelerator, open this notebook, and use **Run All**. Allow the full session to finish; the six experiments plus cache warm-up can take substantial GPU time and disk space. Download `har_mmact_results.zip` and the timestamped preserved outputs/backup from Kaggle before the session expires.

## Additional corrections after reviewing the Gemini audit
- The performance percentages reported by Gemini are **unverified** because no matching checkpoint/evaluation files were provided. They are not inserted into this notebook.
- The existing fusion implementation is embedding concatenation followed by an MLP, not a fixed 50/50 probability average.
- The upgraded experiments are explicitly included in the validation candidate list.
- Evaluation now applies the same required sensor-file filter as training and reconstructs the selected fusion strategy from checkpoint configuration.
- Candidate selection uses validation Macro-F1. The held-out test split is evaluated only once, for the selected checkpoint.
- The results ZIP is created only by the final packaging cell. A path printed in a report is not proof that an archive exists.
- Inference reconstruction also restores the checkpoint's sensor backbone and fusion strategy, so the demo can use the validation-selected TCN/gated checkpoint if it wins.


## Section 1: Environment & Dependencies Setup
Installs auxiliary dependencies while strictly preserving Kaggle's preinstalled, binary-compatible C ABI scientific stack (NumPy, Pandas, SciPy, PyTorch, Torchvision, Scikit-Learn) and configures CPU worker threads.


In [1]:
# Section 1: Dependencies Setup with Kaggle C ABI Protection
import sys
import os
import subprocess
from pathlib import Path

PROJECT_ROOT = Path('/kaggle/working/HAR-MMAct')
CORE_C_PACKAGES = {'numpy', 'pandas', 'scipy', 'torch', 'torchvision', 'opencv-python', 'pillow', 'scikit-learn', 'matplotlib', 'seaborn', 'pytest'}

kaggle_req = PROJECT_ROOT / 'requirements-kaggle.txt'
if kaggle_req.exists():
    print(f'Installing auxiliary dependencies from {kaggle_req.name}...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--prefer-binary', '--no-deps', '-r', str(kaggle_req)], check=False)
else:
    # Install standard light requirements without overriding core C packages
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'tabulate', 'psutil'], check=False)

# Post-install C ABI Verification
print('\n' + '=' * 70)
print('POST-INSTALL ABI & IMPORT VERIFICATION')
print('=' * 70)
abi_failed = False
abi_errors = []

try:
    import numpy as np
    print(f'  [PASS] Python      : {sys.version.split()[0]}')
    print(f'  [PASS] NumPy       : {np.__version__}')
    _arr = np.eye(2, dtype=np.float64)
    _ = np.linalg.pinv(_arr)
except Exception as e:
    abi_failed = True
    abi_errors.append(f'NumPy C-extension check failed: {e}')

try:
    import pandas as pd
    print(f'  [PASS] Pandas      : {pd.__version__}')
    _df = pd.DataFrame({'a': [1.0, 2.0], 'b': [3.0, 4.0]})
    _ = _df.mean()
except Exception as e:
    abi_failed = True
    abi_errors.append(f'Pandas C-extension check failed: {e}')

try:
    import scipy.signal
    print(f'  [PASS] SciPy       : {scipy.__version__}')
    _ = scipy.signal.resample([1.0, 2.0, 3.0, 4.0], 8)
except Exception as e:
    abi_failed = True
    abi_errors.append(f'SciPy C-extension check failed: {e}')

try:
    import cv2
    print(f'  [PASS] OpenCV      : {cv2.__version__}')
except Exception as e:
    abi_failed = True
    abi_errors.append(f'OpenCV check failed: {e}')

try:
    import torch
    print(f'  [PASS] PyTorch     : {torch.__version__} (CUDA: {torch.cuda.is_available()})')
    _t = torch.randn(2, 2)
    _ = _t.numpy()
except Exception as e:
    abi_failed = True
    abi_errors.append(f'PyTorch <-> NumPy C ABI bridge check failed: {e}')

try:
    import torchvision
    print(f'  [PASS] Torchvision : {torchvision.__version__}')
    _img = torch.zeros((3, 32, 32), dtype=torch.float32)
    _ = torchvision.transforms.functional.resize(_img, [16, 16])
except Exception as e:
    abi_failed = True
    abi_errors.append(f'Torchvision check failed: {e}')

try:
    import sklearn
    print(f'  [PASS] Scikit-Learn: {sklearn.__version__}')
except Exception as e:
    abi_failed = True
    abi_errors.append(f'Scikit-Learn check failed: {e}')

# Configure CPU Worker Threads
cpu_cores = os.cpu_count() or 1
torch_threads = min(4, cpu_cores)
torch.set_num_threads(torch_threads)
os.environ['OMP_NUM_THREADS'] = str(torch_threads)
print(f'  [PASS] CPU Threads : {torch.get_num_threads()} threads configured (Host logical cores: {cpu_cores})')

if abi_failed:
    print('\n' + '!' * 70)
    print('CRITICAL ABI BINARY INCOMPATIBILITY DETECTED:')
    for err in abi_errors:
        print(f'  - {err}')
    print('\nREMEDY: Restart Kaggle Session from top menu, do NOT run pip install numpy manually.')
    print('!' * 70)
    raise RuntimeError('C ABI binary incompatibility detected:\n' + '\n'.join(abi_errors))

print('=' * 70)
print('>>> C ABI & IMPORT VERIFICATION PASSED: Preinstalled scientific stack is healthy.')
print('=' * 70)



POST-INSTALL ABI & IMPORT VERIFICATION
  [PASS] Python      : 3.13.16
  [PASS] NumPy       : 2.1.3
  [PASS] Pandas      : 2.3.3
  [PASS] SciPy       : 1.16.3
  [PASS] OpenCV      : 4.14.0
  [PASS] PyTorch     : 2.11.0+cu130 (CUDA: True)
  [PASS] Torchvision : 0.26.0+cu130
  [PASS] Scikit-Learn: 1.6.1
  [PASS] CPU Threads : 4 threads configured (Host logical cores: 4)
>>> C ABI & IMPORT VERIFICATION PASSED: Preinstalled scientific stack is healthy.


## Section 2: Preflight Hardware & GPU Diagnostics
Audits GPU accelerator, CUDA device capability, VRAM headroom, CPU threads, host RAM, and working disk space. Enforces fail-loud check if GPU accelerator is disabled and persists runtime hardware metadata to `outputs/manifests/runtime_hardware_manifest.json`.


In [2]:
# Section 2: Preflight Hardware & GPU Diagnostics
import os
import sys
import json
import time
import shutil
import psutil
import torch
from pathlib import Path

!nvidia-smi

print('=' * 78)
print(' PREFLIGHT HARDWARE & GPU DIAGNOSTICS')
print('=' * 78)

# 1. CUDA & GPU Verification
cuda_ok = torch.cuda.is_available()
print(f'PyTorch Version       : {torch.__version__}')
print(f'CUDA Available        : {cuda_ok}')

if not cuda_ok:
    error_msg = (
        'CRITICAL FAILURE: CUDA GPU is not available!\n'
        'This 50-epoch benchmark requires a GPU accelerator.\n'
        'ACTION REQUIRED: In Kaggle right-hand panel -> Session Options -> Accelerator, '
        'select GPU T4 x2 or GPU P100, then restart the session.'
    )
    print('!' * 78)
    print(error_msg)
    print('!' * 78)
    raise RuntimeError(error_msg)

gpu_count = torch.cuda.device_count()
gpu_name = torch.cuda.get_device_name(0)
compute_cap = torch.cuda.get_device_capability(0)
props = torch.cuda.get_device_properties(0)
total_vram_gb = props.total_memory / (1024**3)
alloc_vram_mb = torch.cuda.memory_allocated(0) / (1024**2)
res_vram_mb = torch.cuda.memory_reserved(0) / (1024**2)

print(f'GPU Device Name       : {gpu_name} (x{gpu_count} detected)')
print(f'Compute Capability    : {compute_cap[0]}.{compute_cap[1]}')
print(f'Total VRAM Available  : {total_vram_gb:.2f} GB')
print(f'Initial Allocated VRAM: {alloc_vram_mb:.2f} MB')
print(f'Initial Reserved VRAM : {res_vram_mb:.2f} MB')

# 2. Host System Hardware
cpu_count = os.cpu_count() or 1
vm = psutil.virtual_memory()
ram_total_gb = vm.total / (1024**3)
ram_avail_gb = vm.available / (1024**3)

working_disk = shutil.disk_usage('/kaggle/working')
disk_free_gb = working_disk.free / (1024**3)
disk_total_gb = working_disk.total / (1024**3)

print(f'CPU Logical Cores     : {cpu_count}')
print(f'System RAM Total      : {ram_total_gb:.2f} GB (Available: {ram_avail_gb:.2f} GB)')
print(f'Working Disk Space    : {disk_free_gb:.2f} GB free of {disk_total_gb:.2f} GB')

# 3. Save Runtime Hardware Manifest
manifest_dir = Path('outputs/manifests')
manifest_dir.mkdir(parents=True, exist_ok=True)
hw_manifest = {
    'timestamp': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    'torch_version': torch.__version__,
    'cuda_available': cuda_ok,
    'gpu_count': gpu_count,
    'gpu_name': gpu_name,
    'compute_capability': f'{compute_cap[0]}.{compute_cap[1]}',
    'total_vram_gb': round(total_vram_gb, 2),
    'initial_allocated_vram_mb': round(alloc_vram_mb, 2),
    'cpu_logical_cores': cpu_count,
    'system_ram_total_gb': round(ram_total_gb, 2),
    'system_ram_avail_gb': round(ram_avail_gb, 2),
    'working_disk_free_gb': round(disk_free_gb, 2),
    'working_disk_total_gb': round(disk_total_gb, 2),
}
with open(manifest_dir / 'runtime_hardware_manifest.json', 'w') as f:
    json.dump(hw_manifest, f, indent=2)
print('Saved outputs/manifests/runtime_hardware_manifest.json')

# 4. Runtime Training Settings
print('\nRuntime Acceleration Configuration:')
print('  - Batch Size          : 8 (calibrated for 16-frame 112x112 video + 9-channel sensor)')
print('  - DataLoader Workers  : 2 (non-blocking pin_memory)')
print('  - Automatic Mixed Prec: ENABLED (torch.amp.autocast, float16)')
print('  - cuDNN Benchmark     : ENABLED (torch.backends.cudnn.benchmark = True)')
print('=' * 78)


Sat Nov  7 15:40:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Section 3: Repository Checkout & Verified Codebase Synchronization
Embeds the verified project source plus the experimental TCN sensor encoder and optional gated fusion implementation. The original InceptionTime/concatenation baselines remain available for direct comparison. New upgrade experiments are written to separate output directories.


In [3]:
# Section 3: Repository Checkout & Verified Code Embedding
import sys
import os
import io
import zipfile
import base64
import subprocess
from pathlib import Path

PROJECT_ROOT = Path('/kaggle/working/HAR-MMAct')
REPO_URL = 'https://github.com/CHENNAMSETTYVIKRAM/HAR-MMAct.git'
BRANCH = 'master'

# 1. Attempt git clone / update from GitHub master (if internet is enabled)
if not PROJECT_ROOT.exists():
    print(f'Attempting clone of {REPO_URL} (branch: {BRANCH}) into {PROJECT_ROOT}...')
    try:
        subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, str(PROJECT_ROOT)], check=True, timeout=60)
        print('  --> Git clone succeeded.')
    except Exception as e:
        print(f'  [Info] Git clone skipped/unavailable: {e}. Using embedded self-contained codebase.')
else:
    print(f'Repository already present at {PROJECT_ROOT}. Syncing with origin/{BRANCH}...')
    try:
        subprocess.run(['git', '-C', str(PROJECT_ROOT), 'fetch', 'origin', BRANCH], check=True, timeout=60)
        subprocess.run(['git', '-C', str(PROJECT_ROOT), 'checkout', '-B', BRANCH, f'origin/{BRANCH}'], check=True, timeout=30)
        subprocess.run(['git', '-C', str(PROJECT_ROOT), 'reset', '--hard', f'origin/{BRANCH}'], check=True, timeout=30)
        print('  --> Git sync succeeded.')
    except Exception as e:
        print(f'  [Info] Git sync skipped: {e}. Applying embedded updates to existing files.')

# 2. Unpack embedded latest verified codebase directly into PROJECT_ROOT
# Guarantees all latest verified code (class-balanced loss, full SHA-256 duplicate resolution, strict split line-counts)
# is strictly active and cannot silently revert to older GitHub commits.
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
EMBEDDED_CODE_B64 = '...'

zip_bytes = base64.b64decode(EMBEDDED_CODE_B64.encode('ascii'))
with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zf:
    zf.extractall(PROJECT_ROOT)

print('\n' + '=' * 78)
print('VERIFIED CODEBASE EMBEDDED & SYNCHRONIZED')
print('=' * 78)
print(f'Project Root Location  : {PROJECT_ROOT}')
print('Active Enhancements   :')
print('  * Balanced Class Weights (scripts/train.py)            : ACTIVE')
print('  * Full SHA-256 Duplicate Staging (src/data/stager.py)   : ACTIVE')
print('  * Strict Split Line Count Check (src/data/stager.py)   : ACTIVE')
print('  * Bidirectional Directory Sync (src/data/stager.py)    : ACTIVE')
print('  * Standalone Video Path Parsing (src/data/indexer.py)  : ACTIVE')
print('Self-Contained Mode   : ENABLED (zero external code dependency)')
print('=' * 78)

# 3. Configure import path before importing project modules
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
print(f'PROJECT_ROOT configured : {PROJECT_ROOT}')
print(f'sys.path[0] set to      : {sys.path[0]}')

# 4. Verify project module imports
from src.data.ingestion import DatasetIngestionPipeline
from src.data.indexer import MMActIndexer
from src.data.dataset import MMActMultimodalDataset
from src.data.stager import FastMMActStager, StagingReport
from src.models.fusion_model import MultimodalHARModel
from src.utils.plots import plot_cross_model_comparison, plot_modality_ablation
print('SUCCESS: Verified project imports (ingestion, indexer, dataset, stager, models, plots).')


Attempting clone of https://github.com/CHENNAMSETTYVIKRAM/HAR-MMAct.git (branch: master) into /kaggle/working/HAR-MMAct...


Cloning into '/kaggle/working/HAR-MMAct'...


  --> Git clone succeeded.

VERIFIED CODEBASE EMBEDDED & SYNCHRONIZED
Project Root Location  : /kaggle/working/HAR-MMAct
Active Enhancements   :
  * Balanced Class Weights (scripts/train.py)            : ACTIVE
  * Full SHA-256 Duplicate Staging (src/data/stager.py)   : ACTIVE
  * Strict Split Line Count Check (src/data/stager.py)   : ACTIVE
  * Bidirectional Directory Sync (src/data/stager.py)    : ACTIVE
  * Standalone Video Path Parsing (src/data/indexer.py)  : ACTIVE
Self-Contained Mode   : ENABLED (zero external code dependency)
PROJECT_ROOT configured : /kaggle/working/HAR-MMAct
sys.path[0] set to      : /kaggle/working/HAR-MMAct
SUCCESS: Verified project imports (ingestion, indexer, dataset, stager, models, plots).


## Reproducible run configuration and baseline protection

This cell runs after repository setup and before data staging/training. It archives any existing `outputs/` directory rather than silently reusing stale checkpoints or metrics, and creates a fresh cache namespace to avoid reusing potentially stale sensor-derived cache entries. The previous outputs are preserved under `/kaggle/working/`.

The notebook keeps the existing 50-epoch experiment definitions. It does **not** promise maximum accuracy; model selection uses validation data, and the held-out test set is reserved for final evaluation.

In [4]:
from pathlib import Path
from datetime import datetime
import os, shutil, json, random
import numpy as np
import torch

PROJECT_ROOT = Path('/kaggle/working/HAR-MMAct')
os.chdir(PROJECT_ROOT)
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
OUTPUTS_ROOT = PROJECT_ROOT / 'outputs'
if OUTPUTS_ROOT.exists() and any(OUTPUTS_ROOT.iterdir()):
    preserved = Path('/kaggle/working') / f'HAR_MMAct_previous_outputs_{RUN_ID}'
    shutil.move(str(OUTPUTS_ROOT), str(preserved))
    print(f'[PRESERVED] Existing outputs moved to: {preserved}')
OUTPUTS_ROOT.mkdir(parents=True, exist_ok=True)

# A fresh cache namespace prevents silent reuse of old video/sensor cache files.
CACHE_ROOT = Path('/kaggle/working') / f'har_mmact_cache_{RUN_ID}'
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

# Reproducibility seeds. The training scripts may also set their own seed; this
# covers notebook-side Python/NumPy/PyTorch operations.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print(f'RUN_ID={RUN_ID}')
print(f'PROJECT_ROOT={PROJECT_ROOT}')
print(f'OUTPUTS_ROOT={OUTPUTS_ROOT}')
print(f'CACHE_ROOT={CACHE_ROOT}')

[PRESERVED] Existing outputs moved to: /kaggle/working/HAR_MMAct_previous_outputs_20261107_154134
RUN_ID=20261107_154134
PROJECT_ROOT=/kaggle/working/HAR-MMAct
OUTPUTS_ROOT=/kaggle/working/HAR-MMAct/outputs
CACHE_ROOT=/kaggle/working/har_mmact_cache_20261107_154134


## Section 4: Fast, Idempotent Dataset Discovery & Canonical Staging
Discovers attached datasets across `/kaggle/input` using bounded-depth shallow directory inspection (max depth $\le$ 4). Staging is strictly idempotent, preserves historical recordings (e.g. subjects 8 and 16), avoids redundant extraction, and saves before/after counts to `outputs/manifests/dataset_expansion_manifest.json`.


In [5]:
# Section 4: Fast, Idempotent Dataset Discovery & Canonical Staging
from pathlib import Path
from src.data.stager import FastMMActStager, discover_kaggle_input_roots

kaggle_input = Path('/kaggle/input')
canonical_data_dir = Path('/kaggle/working/data/genuine_mmact') if Path('/kaggle/working').exists() else PROJECT_ROOT / 'data' / 'genuine_mmact'

# Candidate roots discovered dynamically (supporting nested /kaggle/input/datasets/<user>/<slug> and direct /kaggle/input/<slug>)
input_roots = discover_kaggle_input_roots(kaggle_input)
if not input_roots:
    local_data = PROJECT_ROOT / 'data'
    if local_data.exists():
        input_roots = [local_data]

print(f'Discovered {len(input_roots)} candidate input roots:')
for r in input_roots:
    print(f'  - {r}')

stager = FastMMActStager(
    canonical_dir=canonical_data_dir,
    input_roots=input_roots,
    max_scan_depth=4,
)

staging_report = stager.discover_and_stage()
print(staging_report.summary())

# Mandatory Pre-Training Gates
assert staging_report.is_valid, (
    f'CRITICAL: Dataset discovery & staging failed! Missing required items: {staging_report.missing_required_paths}\n'
    f'Please ensure MMAct datasets are attached under /kaggle/input.'
)
assert len(staging_report.duplicate_conflicts) == 0, (
    f'CRITICAL: {len(staging_report.duplicate_conflicts)} conflicting duplicate files detected: {staging_report.duplicate_conflicts}'
)
assert staging_report.split_count > 0, 'CRITICAL: Official split files missing in canonical dataset!'
assert staging_report.video_count > 0, 'CRITICAL: No video files found in canonical dataset!'
assert staging_report.sensor_count > 0, 'CRITICAL: No sensor files found in canonical dataset!'
print('\n>>> STAGING GATE PASSED: Canonical dataset verified, official splits present, zero conflicts.')


Discovered 2 candidate input roots:
  - /kaggle/input/datasets/vikramche/mmact-trimmed
  - /kaggle/input/datasets/vikramche/trimmed-scene-video
 DATASET PREFLIGHT DISCOVERY & STAGING REPORT
Attached Dataset Roots  : 2 detected
  * /kaggle/input/datasets/vikramche/mmact-trimmed
  * /kaggle/input/datasets/vikramche/trimmed-scene-video
Canonical Staging Path : /kaggle/working/data/genuine_mmact
Sensor Root Discovered : /kaggle/input/datasets/vikramche/mmact-trimmed/trimmed_sensor/sensor
Official Splits Path   : /kaggle/input/datasets/vikramche/mmact-trimmed/trimmed_splits/splits
Existing Data Reused   : False
Staging Duration       : 690.216 seconds
Links / Copies Created : 17828 linked, 0 copied, 0 repaired
------------------------------------------------------------------------------
Video Subjects Staged  : 20 subjects ['subject1', 'subject10', 'subject11', 'subject12', 'subject13', 'subject14', 'subject15', 'subject16', 'subject17', 'subject18', 'subject19', 'subject2', 'subject20', '

## Section 5: Label Mapping & Pre-Flight Modality Audit (Zero-Leakage Split)
Verifies video frame decoding, sensor CSV parsing, index pairing, and enforces strict subject-independent split isolation (train/val/test) across the 10 target activity classes. Performs a paired-file integrity check for all split samples and explicitly verifies honest labeling of the evaluation partition.


In [6]:
# Section 5: Label Mapping & Pre-Flight Modality Audit
from pathlib import Path
import cv2
import pandas as pd
from src.data.indexer import MMActIndexer
from src.data.splits import DatasetSplitter, verify_split_isolation, verify_split_representation
from src.config import SplitConfig, VERIFIED_TARGET_CLASSES

print('=' * 78)
print('PRE-FLIGHT VALIDATION & COMPREHENSIVE MULTI-MODALITY AUDIT')
print('=' * 78)
print(f'Repository Root : {PROJECT_ROOT}')
print(f'Canonical Root  : {canonical_data_dir.resolve()}')

# 1. File Inventory
total_videos = MMActIndexer.find_files_recursively(canonical_data_dir, {'.mp4'})
total_sensors = MMActIndexer.find_files_recursively(canonical_data_dir, {'.csv'})
total_splits = [f for f in MMActIndexer.find_files_recursively(canonical_data_dir, {'.txt'}) if any(k in f.name.lower() for k in ['cross_scene', 'cross_view', 'train', 'val', 'test'])]

print(f'CANONICAL STAGED TOTALS: {len(total_videos)} Videos | {len(total_sensors)} Sensors | {len(total_splits)} Splits')
assert len(total_videos) > 0 and len(total_sensors) > 0, 'CRITICAL: No valid MMAct data files found in canonical directory!'

# 2. Modality Readability check
sample_video = total_videos[0]
cap = cv2.VideoCapture(str(sample_video))
ret, frame = cap.read()
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()
assert ret and frame is not None, f'Failed to read frame from {sample_video}'
print(f'  [PASS] Video Readability: Decoded frame {frame.shape} from {sample_video.name} ({frame_count} frames)')

sample_sensor = total_sensors[0]
df_sample = pd.read_csv(sample_sensor)
print(f'  [PASS] Sensor Parseability: Parsed CSV {df_sample.shape} from {sample_sensor.name}')

# 3. Indexing & Target Class Coverage
indexer = MMActIndexer(dataset_root=canonical_data_dir)
report = indexer.index()
print(f'  - Paired Samples Created: {report.paired_samples_count}')
print(f'  Target Class Coverage ({len(report.class_distribution)}/{len(VERIFIED_TARGET_CLASSES)} target classes):')
for cls_name in VERIFIED_TARGET_CLASSES:
    cnt = report.class_distribution.get(cls_name, 0)
    print(f'    * {cls_name:<16}: {cnt} samples')

assert report.paired_samples_count > 0, 'Zero paired multimodal samples created!'

# 4. Subject-Independent Split Isolation Verification
splitter = DatasetSplitter(SplitConfig(split_strategy='subject'))
train_samples, val_samples, test_samples = splitter.split(report.paired_samples)
isolation = verify_split_isolation(train_samples, val_samples, test_samples, strategy='subject')
print(f'  - Splits: Train={len(train_samples)}, Val={len(val_samples)}, Test={len(test_samples)}')
for check_k, passed in isolation.items():
    if passed is None:
        print(f'      * {check_k:<28}: NOT_APPLICABLE (empty test split)')
    else:
        stat_str = 'PASSED' if passed else 'FAILED'
        print(f'      * {check_k:<28}: {stat_str}')
        assert passed, f'CRITICAL: Split leakage detected on {check_k}!'

# 5. Paired File Path Existence Audit
missing_vid_files = []
missing_sns_files = []
for sp_lbl, s_set in [('train', train_samples), ('val', val_samples), ('test', test_samples)]:
    for s in s_set:
        if not Path(s.video_path).exists():
            missing_vid_files.append((sp_lbl, s.sample_id, str(s.video_path)))
        for m_name, m_path in s.sensor_paths.items():
            if not Path(m_path).exists():
                missing_sns_files.append((sp_lbl, s.sample_id, m_name, str(m_path)))
print(f'  - Missing Video Files : {len(missing_vid_files)}')
print(f'  - Missing Sensor Files: {len(missing_sns_files)}')
assert len(missing_vid_files) == 0, f'CRITICAL: {len(missing_vid_files)} missing video files detected!'
assert len(missing_sns_files) == 0, f'CRITICAL: {len(missing_sns_files)} missing sensor files detected!'

# 6. Representation Audit across Splits
rep_audit = verify_split_representation(train_samples, val_samples, test_samples, expected_classes=VERIFIED_TARGET_CLASSES)
train_subjs = sorted(list({s.subject for s in train_samples}))
val_subjs = sorted(list({s.subject for s in val_samples}))
test_subjs = sorted(list({s.subject for s in test_samples}))
eval_split_desc = 'Held-Out Test Split' if len(test_samples) > 0 else 'Official MMAct Validation Split'
print(f'  - Subjects per Split : Train={train_subjs}, Val={val_subjs}, Test={test_subjs}')
print(f'  - Missing Classes    : Train={rep_audit["missing_classes_train"]}, Val={rep_audit["missing_classes_val"]}, Test={rep_audit["missing_classes_test"]}')
print(f'  - Modalities Paired  : Train={rep_audit["modalities_paired_train"]}, Val={rep_audit["modalities_paired_val"]}, Test={rep_audit["modalities_paired_test"]}')
print(f'  - Supervised Eval Set: {eval_split_desc} ({len(test_samples if len(test_samples) > 0 else val_samples)} samples)')
print('=' * 78)
print('>>> PRE-FLIGHT VALIDATION PASSED: Dataset is readable, parseable, and strictly isolated.')
print('=' * 78)


PRE-FLIGHT VALIDATION & COMPREHENSIVE MULTI-MODALITY AUDIT
Repository Root : /kaggle/working/HAR-MMAct
Canonical Root  : /kaggle/working/data/genuine_mmact
CANONICAL STAGED TOTALS: 17820 Videos | 35685 Sensors | 4 Splits
  [PASS] Video Readability: Decoded frame (1080, 1920, 3) from walking.mp4 (240 frames)
  [PASS] Sensor Parseability: Parsed CSV (446, 4) from carrying.csv
  - Paired Samples Created: 5431
  Target Class Coverage (10/10 target classes):
    * crouching       : 535 samples
    * jumping         : 535 samples
    * kicking         : 535 samples
    * pointing        : 535 samples
    * running         : 535 samples
    * sitting         : 316 samples
    * standing        : 519 samples
    * using_phone     : 851 samples
    * walking         : 535 samples
    * waving_hand     : 535 samples
  - Splits: Train=3799, Val=780, Test=852
      * test_split_present          : PASSED
      * video_disjoint_train_val    : PASSED
      * video_disjoint_train_test   : PASSED
     

In [7]:
import json
from pathlib import Path

root = Path("/kaggle/working/HAR-MMAct")
canonical = Path("/kaggle/working/data/genuine_mmact")

print("Project root exists:", root.exists())
print("Canonical data exists:", canonical.exists())

# Locate generated manifests and audit reports
for base in [root / "outputs", canonical]:
    if base.exists():
        print(f"\nFiles under {base}:")
        for p in base.rglob("*"):
            if p.is_file() and (
                p.suffix.lower() in {".json", ".csv", ".txt"}
                and any(k in p.name.lower() for k in
                        ["manifest", "index", "audit", "pair", "sample"])
            ):
                print(p)

Project root exists: True
Canonical data exists: True

Files under /kaggle/working/HAR-MMAct/outputs:
/kaggle/working/HAR-MMAct/outputs/manifests/dataset_expansion_manifest.json

Files under /kaggle/working/data/genuine_mmact:
/kaggle/working/data/genuine_mmact/dataset_manifest.json
/kaggle/working/data/genuine_mmact/manifests/dataset_expansion_manifest.json


In [8]:
import json
from pathlib import Path

files = [
    Path("/kaggle/working/data/genuine_mmact/dataset_manifest.json"),
    Path("/kaggle/working/data/genuine_mmact/manifests/dataset_expansion_manifest.json"),
    Path("/kaggle/working/HAR-MMAct/outputs/manifests/dataset_expansion_manifest.json"),
]

def inspect(obj, prefix=""):
    if isinstance(obj, dict):
        for key, value in obj.items():
            if isinstance(value, (dict, list)):
                print(f"{prefix}{key}: [{type(value).__name__}, "
                      f"{len(value)} entries]")
                if isinstance(value, list) and value:
                    print(f"  First entry: {str(value[0])[:500]}")
                elif isinstance(value, dict):
                    inspect(value, prefix + "  ")
            else:
                print(f"{prefix}{key}: {value}")
    elif isinstance(obj, list):
        print(f"{prefix}List entries: {len(obj)}")
        if obj:
            print(f"{prefix}First entry: {str(obj[0])[:500]}")

for path in files:
    print("\n" + "=" * 75)
    print("FILE:", path)
    print("=" * 75)

    if not path.exists():
        print("NOT FOUND")
        continue

    with path.open() as f:
        data = json.load(f)

    inspect(data)


FILE: /kaggle/working/data/genuine_mmact/dataset_manifest.json
is_valid: True
canonical_dir: /kaggle/working/data/genuine_mmact
attached_dataset_roots: [list, 2 entries]
  First entry: /kaggle/input/datasets/vikramche/mmact-trimmed
sensor_root: /kaggle/input/datasets/vikramche/mmact-trimmed/trimmed_sensor/sensor
sensor_modalities: [dict, 4 entries]
  acc_phone_clip: /kaggle/input/datasets/vikramche/mmact-trimmed/trimmed_sensor/sensor/acc_phone_clip
  acc_watch_clip: /kaggle/input/datasets/vikramche/mmact-trimmed/trimmed_sensor/sensor/acc_watch_clip
  gyro_clip: /kaggle/input/datasets/vikramche/mmact-trimmed/trimmed_sensor/sensor/gyro_clip
  orientation_clip: /kaggle/input/datasets/vikramche/mmact-trimmed/trimmed_sensor/sensor/orientation_clip
official_splits_dir: /kaggle/input/datasets/vikramche/mmact-trimmed/trimmed_splits/splits
official_split_files: [dict, 4 entries]
  trimmed_x-scene_train.txt: /kaggle/input/datasets/vikramche/mmact-trimmed/trimmed_splits/splits/trimmed_x-scene_tr

In [9]:
from pathlib import Path

root = Path("/kaggle/working/HAR-MMAct")

print("Potential index and paired-sample files:")
for p in root.rglob("*"):
    if p.is_file() and p.suffix.lower() in {".csv", ".json", ".pkl", ".parquet"}:
        name = p.name.lower()
        if any(k in name for k in ["index", "pair", "sample", "dataset"]):
            print(p)

Potential index and paired-sample files:
/kaggle/working/HAR-MMAct/outputs/manifests/dataset_expansion_manifest.json
/kaggle/working/HAR-MMAct/challenge-main/challenge-main/evaluation/trimmed_val_view_sample_format.json
/kaggle/working/HAR-MMAct/challenge-main/challenge-main/evaluation/trimmed_val_scene_sample_format.json
/kaggle/working/HAR-MMAct/challenge-main/challenge-main/evaluation/untrimmed_val_sample_submission.json


In [10]:
import pandas as pd

print("=" * 70)
print("ACTIVE NOTEBOOK DATA OBJECTS")
print("=" * 70)

for name, obj in list(globals().items()):
    try:
        if isinstance(obj, pd.DataFrame):
            print(f"\nDATAFRAME: {name}")
            print("Shape:", obj.shape)
            print("Columns:", list(obj.columns))
            print(obj.head(2).to_string(index=False))

        elif isinstance(obj, (list, tuple)) and len(obj) > 0:
            first = obj[0]
            if isinstance(first, dict):
                print(f"\nRECORD LIST: {name}")
                print("Length:", len(obj))
                print("First record:", first)
            elif isinstance(first, (str, int, float)):
                if any(k in name.lower() for k in
                       ["sample", "pair", "index", "record", "dataset"]):
                    print(f"\nLIST: {name}, length={len(obj)}")
                    print("First 5:", obj[:5])

    except Exception as e:
        print(f"Inspection skipped for {name}: {e}")

ACTIVE NOTEBOOK DATA OBJECTS

DATAFRAME: _df
Shape: (2, 2)
Columns: ['a', 'b']
  a   b
1.0 3.0
2.0 4.0

DATAFRAME: df_sample
Shape: (446, 4)
Columns: ['20181016_13:37:28.616000', '1.6380169', '1.4222404', '2.5051136']
20181016_13:37:28.616000  1.6380169  1.4222404  2.5051136
20181016_13:37:28.631000   1.632292   1.426489   2.512779
20181016_13:37:28.654000   1.652991   1.418664   2.473278


In [14]:
from pathlib import Path

root = Path("/kaggle/working/data/genuine_mmact")

print("DATASET ROOT:", root)
print("EXISTS:", root.exists())

for folder in [
    root / "trimmed" / "video",
    root / "trimmed" / "sensors",
    root / "trimmed" / "splits",
]:
    print(f"\n{folder}")
    print("Exists:", folder.exists())

    if folder.exists():
        for item in list(folder.iterdir())[:15]:
            print(" ", item.name)

print("\nSENSOR MODALITY COUNTS")

sensor_root = root / "trimmed" / "sensors"

if sensor_root.exists():
    for modality in sorted(sensor_root.iterdir()):
        if modality.is_dir():
            files = list(modality.rglob("*.csv"))
            empty = sum(f.stat().st_size == 0 for f in files)
            print(
                modality.name,
                "| CSV files:", len(files),
                "| Empty files:", empty
            )

DATASET ROOT: /kaggle/working/data/genuine_mmact
EXISTS: True

/kaggle/working/data/genuine_mmact/trimmed/video
Exists: True
  cross_scene

/kaggle/working/data/genuine_mmact/trimmed/sensors
Exists: True
  orientation_clip
  acc_watch_clip
  acc_phone_clip
  gyro_clip

/kaggle/working/data/genuine_mmact/trimmed/splits
Exists: True
  trimmed_x-view_train.txt
  trimmed_x-scene_val.txt
  trimmed_x-scene_train.txt
  trimmed_x-view_val.txt

SENSOR MODALITY COUNTS
acc_phone_clip | CSV files: 8982 | Empty files: 1
acc_watch_clip | CSV files: 8736 | Empty files: 4
gyro_clip | CSV files: 8982 | Empty files: 1
orientation_clip | CSV files: 8985 | Empty files: 4


In [16]:
import pandas as pd
from pathlib import Path

root = Path("/kaggle/working/data/genuine_mmact/trimmed/sensors")

samples = {
    "acc_phone": root / "acc_phone_clip/subject9/scene4/session1/carrying.csv",
    "acc_watch": root / "acc_watch_clip/subject9/scene4/session1/carrying.csv",
    "gyro": root / "gyro_clip/subject9/scene4/session1/carrying.csv",
}

for name, path in samples.items():
    print(f"\n{'='*60}")
    print(name, "|", path.exists())

    df = pd.read_csv(path, header=None)

    print("Shape:", df.shape)
    print("First rows:")
    print(df.head(3).to_string(index=False))

    print("Column types:")
    print(df.dtypes.to_string())

    print("Missing values:", int(df.isna().sum().sum()))

    numeric = df.iloc[:, 1:].apply(pd.to_numeric, errors="coerce")
    print("Invalid numeric sensor values:", int(numeric.isna().sum().sum()))


acc_phone | True
Shape: (1047, 4)
First rows:
                       0        1         2        3
20181024_14:51:17.322000 3.662415 -8.800217 2.343887
20181024_14:51:17.332000 4.617706 -9.291031 1.355072
20181024_14:51:17.342000 4.981628 -9.736343 0.105301
Column types:
0     object
1    float64
2    float64
3    float64
Missing values: 0
Invalid numeric sensor values: 0

acc_watch | True
Shape: (1055, 4)
First rows:
                       0        1         2        3
20181024_14:51:17.319000 8.128419 -0.098106 4.273581
20181024_14:51:17.327000 8.087741 -1.311267 4.350151
20181024_14:51:17.338000 8.367701 -2.335395 4.541577
Column types:
0     object
1    float64
2    float64
3    float64
Missing values: 0
Invalid numeric sensor values: 0

gyro | True
Shape: (528, 4)
First rows:
                       0        1        2         3
20181024_14:51:17.324000 1.328888 1.576141  0.262176
20181024_14:51:17.336000 1.533264 1.193893  0.087875
20181024_14:51:17.381000 1.498444 1.088348 -0.00

In [17]:
from pathlib import Path

path = Path("/kaggle/working/HAR-MMAct/src/data/sensor_loader.py")

print("FILE EXISTS:", path.exists())

if path.exists():
    lines = path.read_text(errors="replace").splitlines()

    keywords = [
        "read_csv",
        "timestamp",
        "to_datetime",
        "interpolate",
        "resample",
        "merge",
        "join",
        "align",
        "window",
        "bfill",
        "ffill",
    ]

    selected = set()

    for i, line in enumerate(lines):
        if any(k in line.lower() for k in keywords):
            selected.update(range(max(0, i-3), min(len(lines), i+5)))

    for i in sorted(selected):
        print(f"{i+1:4}: {lines[i]}")

FILE EXISTS: True
   1: """
   2: sensor_loader.py - Multi-Modal Sensor Ingestion, Validation, Resampling & Windowing
   3: 
   4: Implements the official MMAct sensor preprocessing conventions from utils/time_series_classifiers.py:
   5: 1. Loads CSV sensor recordings (handles both files with headers and headerless files).
   6: 2. Converts accelerometer m/s^2 -> g (divide by 9.8).
   7: 3. Parses start timestamp (format %Y%m%d_%H:%M:%S.%f or epoch) and generates uniform 100 Hz/50 Hz timegrid.
   8: 4. Multi-modal datetime alignment & resampling onto a uniform 100 Hz timeline matching official MMAct code.
   9: 5. Slices fixed-length sliding windows without cross-subject or cross-session leakage.
  10: """
  11: 
  12: from pathlib import Path
  13: from typing import Dict, List, Optional, Tuple, Union
  19: 
  20: 
  21: class SensorLoader:
  22:     """Loads, cleans, aligns, and windows multi-modal MMAct wearable sensor data."""
  23: 
  24:     def __init__(self, config: Optional[S

In [18]:
from pathlib import Path

path = Path(
    "/kaggle/working/HAR-MMAct/src/data/sensor_loader.py"
)

lines = path.read_text().splitlines()

for start, end in [(27, 66), (67, 130), (131, 166)]:
    print(f"\n{'='*65}")
    print(f"LINES {start}-{end}")
    print("="*65)

    for i in range(start - 1, min(end, len(lines))):
        print(f"{i+1:4}: {lines[i]}")


LINES 27-66
  27:     def parse_start_time_epoch_seconds(self, first_val: str) -> float:
  28:         """
  29:         Parses initial timestamp string (%Y%m%d_%H:%M:%S.%f or %Y/%m/%d %H:%M:%S.%f or epoch)
  30:         into seconds since epoch (UTC, localized from Asia/Tokyo per official MMAct code).
  31:         """
  32:         first_str = str(first_val).strip()
  33: 
  34:         # Check if already numeric epoch in ms or seconds
  35:         try:
  36:             num = float(first_str)
  37:             if num > 1e11:  # Epoch in ms
  38:                 return num / 1000.0
  39:             return num
  40:         except ValueError:
  41:             pass
  42: 
  43:         # Ensure subsecond dot exists
  44:         if "." not in first_str:
  45:             first_str = first_str + ".000"
  46: 
  47:         # Check format with slash or underscore
  48:         date_fmt = (
  49:             "%Y/%m/%d %H:%M:%S.%f"
  50:             if "/" in first_str
  51:           

In [19]:
from pathlib import Path

repo = Path("/kaggle/working/HAR-MMAct")

matches = list(repo.rglob("time_series_classifiers.py"))

print("REFERENCE FILES FOUND:", len(matches))

for path in matches:
    print("\nFILE:", path)

    lines = path.read_text(errors="replace").splitlines()

    for i in range(190, min(235, len(lines))):
        print(f"{i+1:4}: {lines[i]}")

REFERENCE FILES FOUND: 1

FILE: /kaggle/working/HAR-MMAct/challenge-main/challenge-main/utils/time_series_classifiers.py
 191:         try:
 192:             df = pd.read_csv(csv_file_path, usecols=cols)
 193:         except ValueError:
 194:             df = pd.read_csv(csv_file_path, names=cols)
 195: 
 196:         if len(df) == 0:
 197:             print("Empty csv file found: " + str(csv_file_path), flush=True)
 198:             continue
 199: 
 200:         if prefix is not None:
 201:             rename_cols = {c: prefix + "_" + c for c in cols if c != "time"}
 202:             df.rename(columns=rename_cols, inplace=True)
 203:         
 204:         if "." not in df.at[0, "time"]:
 205:             df.at[0, "time"] = df.at[0, "time"] + ".000"
 206: 
 207:         #https://pandas.pydata.org/pandas-docs/stable/user_guide/timeseries.html#from-timestamps-to-epoch
 208:         stt_time = pd.to_datetime(df.at[0, "time"], format="%Y%m%d_%H:%M:%S.%f").tz_localize('Asia/Tokyo').tz_conv

In [15]:
from pathlib import Path

root = Path("/kaggle/working/data/genuine_mmact/trimmed/sensors")

print("=" * 75)
print("EMPTY SENSOR CSV FILES")
print("=" * 75)

for modality in sorted(root.iterdir()):
    if not modality.is_dir():
        continue

    empty_files = [
        f for f in modality.rglob("*.csv")
        if f.stat().st_size == 0
    ]

    print(f"\n{modality.name}: {len(empty_files)} empty files")

    for f in empty_files:
        print(f"  {f}")

print("\n" + "=" * 75)
print("SAMPLE NON-EMPTY CSV CONTENT")
print("=" * 75)

for modality in ["acc_phone_clip", "acc_watch_clip", "gyro_clip"]:
    folder = root / modality
    samples = [
        f for f in folder.rglob("*.csv")
        if f.stat().st_size > 0
    ]

    print(f"\n--- {modality} ---")

    if samples:
        sample = samples[0]
        print("File:", sample)
        with open(sample, "r", errors="replace") as f:
            for i, line in enumerate(f):
                if i >= 4:
                    break
                print(line.rstrip())

EMPTY SENSOR CSV FILES

acc_phone_clip: 1 empty files
  /kaggle/working/data/genuine_mmact/trimmed/sensors/acc_phone_clip/subject2/scene4/session5/pushing.csv

acc_watch_clip: 4 empty files
  /kaggle/working/data/genuine_mmact/trimmed/sensors/acc_watch_clip/subject13/scene1/session5/standing.csv
  /kaggle/working/data/genuine_mmact/trimmed/sensors/acc_watch_clip/subject11/scene2/session1/jumping.csv
  /kaggle/working/data/genuine_mmact/trimmed/sensors/acc_watch_clip/subject11/scene2/session1/throwing.csv
  /kaggle/working/data/genuine_mmact/trimmed/sensors/acc_watch_clip/subject11/scene2/session1/using_phone.csv

gyro_clip: 1 empty files
  /kaggle/working/data/genuine_mmact/trimmed/sensors/gyro_clip/subject2/scene4/session5/pushing.csv

orientation_clip: 4 empty files
  /kaggle/working/data/genuine_mmact/trimmed/sensors/orientation_clip/subject2/scene4/session5/pushing.csv
  /kaggle/working/data/genuine_mmact/trimmed/sensors/orientation_clip/subject1/scene4/session5/fall.csv
  /kaggle/

In [21]:
from pathlib import Path
import re

file_path = Path("src/data/sensor_loader.py")

if not file_path.exists():
    raise FileNotFoundError(f"File not found: {file_path.resolve()}")

source = file_path.read_text()

old_block = """if self.config.fill_strategy == "bfill":
            primary_df = primary_df.bfill().ffill()
        else:
            primary_df = primary_df.interpolate(method="linear").bfill().ffill()"""

new_block = """primary_df = primary_df.bfill()
        primary_df = primary_df.dropna()"""

if old_block not in source:
    if new_block in source:
        print("Correction already applied. No changes made.")
    else:
        raise RuntimeError(
            "Expected code block not found. File was not modified. "
            "Inspect the alignment section before applying a manual patch."
        )
else:
    updated_source = source.replace(old_block, new_block, 1)
    file_path.write_text(updated_source)
    print("Correction applied successfully.")

# Verify the updated section
lines = file_path.read_text().splitlines()

for i, line in enumerate(lines):
    if "primary_df = primary_df.bfill()" in line:
        print("\nVerified code:")
        for j in range(max(0, i - 2), min(len(lines), i + 5)):
            print(f"{j + 1}: {lines[j]}")
        break

Correction applied successfully.

Verified code:
158: 
159:         # Fill resampled frequencies (e.g. 50 Hz gyro joined onto 100 Hz primary)
160:         primary_df = primary_df.bfill()
161:         primary_df = primary_df.dropna()
162: 
163:         primary_df = primary_df.dropna()
164:         return primary_df


In [22]:
from src.config import SensorConfig, SENSOR_RATES_HZ

cfg = SensorConfig()

print("Window size:", cfg.window_size)
print("Fill strategy:", cfg.fill_strategy)
print("Modalities:", cfg.modalities)
print("Sample rates:", SENSOR_RATES_HZ)
print("Accelerometer conversion:", cfg.convert_acc_to_g)

Window size: 100
Fill strategy: bfill
Modalities: ['acc_phone', 'acc_watch', 'gyro']
Sample rates: {'acc_phone': 100, 'acc_watch': 100, 'gyro': 50, 'orientation': 50}
Accelerometer conversion: True


In [23]:
from pathlib import Path

file_path = Path("src/data/sensor_loader.py")
source = file_path.read_text()

old = """        primary_df = primary_df.bfill()
        primary_df = primary_df.dropna()

        primary_df = primary_df.dropna()
        return primary_df"""

new = """        primary_df = primary_df.bfill()
        primary_df = primary_df.dropna()
        return primary_df"""

if old in source:
    file_path.write_text(source.replace(old, new, 1))
    print("Duplicate dropna() removed.")
elif new in source:
    print("Already clean. No changes needed.")
else:
    raise RuntimeError("Expected code section not found. No changes made.")

# Verify
lines = file_path.read_text().splitlines()
for i, line in enumerate(lines):
    if "primary_df = primary_df.bfill()" in line:
        print("\n".join(lines[i:i+4]))
        break

Duplicate dropna() removed.
        primary_df = primary_df.bfill()
        primary_df = primary_df.dropna()
        return primary_df



In [24]:
import ast
import importlib
from pathlib import Path

file_path = Path("src/data/sensor_loader.py")

# Syntax validation
source = file_path.read_text()
ast.parse(source)
print("PASS: Python syntax is valid.")

# Import validation
importlib.invalidate_caches()
module = importlib.import_module("src.data.sensor_loader")
importlib.reload(module)

print("PASS: sensor_loader imported successfully.")
print("READY: Proceed to paired-data validation.")

PASS: Python syntax is valid.
PASS: sensor_loader imported successfully.
READY: Proceed to paired-data validation.


In [25]:
import inspect
from pathlib import Path
import src.data.indexer as indexer_module

print("INDEXER MODULE:", Path(indexer_module.__file__).resolve())

print("\nAVAILABLE INDEXER CLASSES:")
for name, obj in inspect.getmembers(indexer_module, inspect.isclass):
    if obj.__module__ == indexer_module.__name__:
        print(f"\n{name}{inspect.signature(obj)}")
        for method_name, method in inspect.getmembers(obj, inspect.isfunction):
            if not method_name.startswith("_"):
                try:
                    print(f"  {method_name}{inspect.signature(method)}")
                except (TypeError, ValueError):
                    print(f"  {method_name}")

print("\nDATASET ROOT CHECK:")
root = Path("/kaggle/working/data/genuine_mmact")
for path in [
    root / "trimmed/video/cross_scene",
    root / "trimmed/sensors",
    root / "trimmed/splits",
]:
    print(f"{path}: {'EXISTS' if path.exists() else 'MISSING'}")

INDEXER MODULE: /kaggle/working/HAR-MMAct/src/data/indexer.py

AVAILABLE INDEXER CLASSES:

IndexerReport(total_video_files_found: int = 0, total_sensor_files_found: int = 0, paired_samples_count: int = 0, paired_samples: List[src.data.indexer.PairedSample] = <factory>, missing_sensors: List[Dict[str, str]] = <factory>, missing_videos: List[Dict[str, str]] = <factory>, unknown_label_files: List[str] = <factory>, duplicate_ids: List[str] = <factory>, class_distribution: Dict[str, int] = <factory>) -> None
  summary(self) -> str

MMActIndexer(dataset_root: pathlib._local.Path, required_sensor_modalities: Optional[List[str]] = None, target_classes: Optional[List[str]] = None)
  find_files_recursively(root: pathlib._local.Path, extensions: Set[str]) -> List[pathlib._local.Path]
  index(self) -> src.data.indexer.IndexerReport
  normalize_action_name(stem: str) -> str
  normalize_identifier(identifier: str, prefix: str) -> str
  parse_sensor_path(self, path: pathlib._local.Path) -> Optional[T

In [26]:
from pathlib import Path
from collections import Counter, defaultdict
from src.data.indexer import MMActIndexer

# Dataset configuration
DATASET_ROOT = Path("/kaggle/working/data/genuine_mmact/trimmed")

TARGET_CLASSES = [
    "walking", "jumping", "running", "standing",
    "waving_hand", "kicking", "sitting", "crouching",
    "using_phone", "pointing"
]

REQUIRED_SENSORS = ["acc_phone", "acc_watch", "gyro"]

TRAIN_SUBJECTS = {f"subject{i}" for i in range(1, 15)}
VAL_SUBJECTS = {f"subject{i}" for i in range(15, 18)}
TEST_SUBJECTS = {f"subject{i}" for i in range(18, 21)}

# Build index
indexer = MMActIndexer(
    dataset_root=DATASET_ROOT,
    required_sensor_modalities=REQUIRED_SENSORS,
    target_classes=TARGET_CLASSES
)

report = indexer.index()

print("INDEX SUMMARY")
print(report.summary())

# Validate required sensor files
valid_samples = []
invalid_samples = []

for sample in report.paired_samples:
    problems = []

    for modality in REQUIRED_SENSORS:
        sensor_path = sample.sensor_paths.get(modality)

        if sensor_path is None:
            problems.append(f"{modality}: missing")
        elif not sensor_path.exists():
            problems.append(f"{modality}: file missing")
        elif sensor_path.stat().st_size == 0:
            problems.append(f"{modality}: empty file")

    if problems:
        invalid_samples.append((sample, problems))
    else:
        valid_samples.append(sample)

print("\nFILE VALIDATION")
print("Indexed samples:", len(report.paired_samples))
print("Valid samples:", len(valid_samples))
print("Invalid samples:", len(invalid_samples))

for sample, problems in invalid_samples[:20]:
    print(sample.sample_id, "->", ", ".join(problems))

# Subject-independent split
splits = {
    "train": [s for s in valid_samples if s.subject in TRAIN_SUBJECTS],
    "validation": [s for s in valid_samples if s.subject in VAL_SUBJECTS],
    "test": [s for s in valid_samples if s.subject in TEST_SUBJECTS],
}

print("\nSPLIT COUNTS")
for split_name, samples in splits.items():
    print(f"{split_name}: {len(samples)}")

# Check for unassigned subjects
assigned_subjects = TRAIN_SUBJECTS | VAL_SUBJECTS | TEST_SUBJECTS
unassigned = [
    s for s in valid_samples
    if s.subject not in assigned_subjects
]
print("Unassigned-subject samples:", len(unassigned))

# Class distribution per split
print("\nCLASS DISTRIBUTION")
for split_name, samples in splits.items():
    counts = Counter(s.action for s in samples)
    print(f"\n{split_name}:")
    for class_name in TARGET_CLASSES:
        print(f"  {class_name}: {counts.get(class_name, 0)}")

# Final checks
print("\nVALIDATION CHECKS")
print("All splits non-empty:", all(len(v) > 0 for v in splits.values()))
print("No unassigned subjects:", len(unassigned) == 0)
print(
    "All 10 classes in every split:",
    all(
        all(Counter(s.action for s in samples).get(c, 0) > 0
            for c in TARGET_CLASSES)
        for samples in splits.values()
    )
)

# Retain validated data for subsequent notebook cells
paired_samples = valid_samples

INDEX SUMMARY
=== MMAct Dataset Indexer Report ===
Total Video Files Found   : 17820
Total Sensor Files Found  : 35685
Paired Multimodal Samples : 5431
Missing Sensor Incidents  : 196
Missing Video Incidents   : 1522
Unknown Label Files Filtered: 36388
Duplicate IDs Encountered : 0
Class Distribution (Paired):
  - crouching      : 535
  - jumping        : 535
  - kicking        : 535
  - pointing       : 535
  - running        : 535
  - sitting        : 316
  - standing       : 519
  - using_phone    : 851
  - walking        : 535
  - waving_hand    : 535

FILE VALIDATION
Indexed samples: 5431
Valid samples: 5427
Invalid samples: 4
subject13_scene1_session5_cam3_standing -> acc_watch: empty file
subject13_scene1_session5_cam4_standing -> acc_watch: empty file
subject13_scene1_session5_cam2_standing -> acc_watch: empty file
subject13_scene1_session5_cam1_standing -> acc_watch: empty file

SPLIT COUNTS
train: 3795
validation: 780
test: 852
Unassigned-subject samples: 0

CLASS DISTRIBUTIO

In [28]:
from concurrent.futures import ThreadPoolExecutor
from collections import Counter
import pandas as pd
import numpy as np

def validate_sample(sample):
    for modality in REQUIRED_SENSORS:
        path = sample.sensor_paths.get(modality)

        if path is None or not path.exists() or path.stat().st_size == 0:
            return sample, False, f"{modality}: missing or empty"

        try:
            df = pd.read_csv(path, header=None)

            if df.empty or df.shape[1] < 4:
                return sample, False, f"{modality}: invalid shape"

            axes = df.iloc[:, -3:].apply(pd.to_numeric, errors="coerce")
            values = axes.to_numpy(dtype=float)

            if not np.all(np.isfinite(values)):
                return sample, False, f"{modality}: invalid numeric values"

            if len(values) < 2:
                return sample, False, f"{modality}: insufficient rows"

        except Exception as e:
            return sample, False, f"{modality}: {str(e)}"

    return sample, True, "OK"


print("Validating", len(valid_samples), "paired samples...")

with ThreadPoolExecutor(max_workers=8) as executor:
    results = list(executor.map(validate_sample, valid_samples))

valid_samples = [s for s, ok, _ in results if ok]
invalid_samples = [(s, msg) for s, ok, msg in results if not ok]

paired_samples = valid_samples

splits = {
    "train": [s for s in valid_samples if s.subject in TRAIN_SUBJECTS],
    "validation": [s for s in valid_samples if s.subject in VAL_SUBJECTS],
    "test": [s for s in valid_samples if s.subject in TEST_SUBJECTS],
}

print("\nVALIDATION RESULTS")
print("Valid samples:", len(valid_samples))
print("Rejected samples:", len(invalid_samples))

for s, reason in invalid_samples[:20]:
    print(s.sample_id, "->", reason)

print("\nSPLIT COUNTS")
for name, samples in splits.items():
    print(name, len(samples))

print("\nALL 10 CLASSES PRESENT:")
for name, samples in splits.items():
    counts = Counter(s.action for s in samples)
    print(name, all(counts[c] > 0 for c in TARGET_CLASSES))

Validating 5427 paired samples...

VALIDATION RESULTS
Valid samples: 5427
Rejected samples: 0

SPLIT COUNTS
train 3795
validation 780
test 852

ALL 10 CLASSES PRESENT:
train True
validation True
test True


In [29]:
import torch
from pathlib import Path

ROOT = Path("/kaggle/working/HAR-MMAct")

print("GPU STATUS")
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
else:
    print("WARNING: Training will use CPU.")

print("\nTRAINING FILES")
for folder in ["src", "scripts", ""]:
    base = ROOT / folder
    if base.exists():
        for pattern in ["*train*.py", "*training*.py"]:
            for path in sorted(base.rglob(pattern)):
                if any(x in path.parts for x in [".git", "__pycache__"]):
                    continue
                print(path.relative_to(ROOT))

print("\nEXISTING CHECKPOINTS")
for path in sorted((ROOT / "outputs").rglob("*.pth")):
    print(path.relative_to(ROOT))

GPU STATUS
CUDA available: True
GPU: Tesla T4
CUDA version: 13.0

TRAINING FILES
scripts/train.py
scripts/train.py

EXISTING CHECKPOINTS


In [30]:
import subprocess
import sys
from pathlib import Path

ROOT = Path("/kaggle/working/HAR-MMAct")
train_script = ROOT / "scripts/train.py"

print("TRAIN SCRIPT:")
print(train_script.read_text()[:14000])

print("\nTRAINING CLI HELP:")
result = subprocess.run(
    [sys.executable, str(train_script), "--help"],
    cwd=ROOT,
    capture_output=True,
    text=True,
    timeout=60
)

print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr[-3000:])
print("Exit code:", result.returncode)

TRAIN SCRIPT:
"""
train.py - Configurable Training Pipeline for Multimodal HAR_MMAct

Supports multimodal fusion, video-only, and sensor-only training with:
- Pretrained ResNet-18 video backbone and InceptionTime sensor backbone
- Leakage-free subject / view / scene / session split verification
- On-disk tensor caching for fast training iterations
- Learning rate scheduling and best-checkpoint persistence
- Comprehensive metrics and publication-ready visualization artifacts
"""

import argparse
import csv
import json
import os
import sys
import time
from pathlib import Path

# Add project root to path
sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from src.config import ProjectConfig, SplitConfig, set_seed, VERIFIED_TARGET_CLASSES
from src.data.indexer import MMActIndexer
from src.data.splits import DatasetSplitter, verify_split_isolation, verify_split_representation
from sr

In [31]:
from pathlib import Path

files = [
    Path("scripts/train.py"),
    Path("src/data/splits.py"),
    Path("src/data/dataset.py"),
    Path("src/data/indexer.py"),
]

keywords = [
    "DatasetSplitter",
    "verify_split_isolation",
    "verify_split_representation",
    "train_samples",
    "val_samples",
    "test_samples",
    "paired_samples",
    "dropna",
    "sensor_paths",
    "split_strategy",
]

for file in files:
    print("\n" + "=" * 75)
    print("FILE:", file)

    if not file.exists():
        print("NOT FOUND")
        continue

    lines = file.read_text().splitlines()

    for i, line in enumerate(lines):
        if any(k in line for k in keywords):
            start = max(0, i - 3)
            end = min(len(lines), i + 7)

            print(f"\n--- Lines {start + 1}-{end} ---")
            for j in range(start, end):
                print(f"{j + 1}: {lines[j]}")


FILE: scripts/train.py

--- Lines 27-36 ---
27: 
28: from src.config import ProjectConfig, SplitConfig, set_seed, VERIFIED_TARGET_CLASSES
29: from src.data.indexer import MMActIndexer
30: from src.data.splits import DatasetSplitter, verify_split_isolation, verify_split_representation
31: from src.data.dataset import MMActMultimodalDataset
32: from src.models.fusion_model import MultimodalHARModel
33: from src.utils.metrics import compute_classification_metrics
34: from src.utils.plots import (
35:     plot_training_curves,
36:     plot_loss_curve_standalone,

--- Lines 64-73 ---
64:     parser.add_argument("--weight_decay", type=float, default=1e-4, help="Weight decay")
65:     parser.add_argument("--dropout", type=float, default=0.35, help="Dropout probability")
66:     parser.add_argument(
67:         "--split_strategy",
68:         type=str,
69:         default="subject",
70:         choices=[
71:             "subject",
72:             "cross_view",
73:             "cross_scene",



In [32]:
from pathlib import Path

def show_section(path, start, end):
    lines = Path(path).read_text().splitlines()
    print(f"\n{'=' * 70}\n{path}: lines {start}-{end}\n")
    for i in range(start - 1, min(end, len(lines))):
        print(f"{i + 1}: {lines[i]}")

show_section("src/data/splits.py", 430, 510)
show_section("scripts/train.py", 285, 340)
show_section("scripts/train.py", 320, 390)


src/data/splits.py: lines 430-510

430:         self, samples: List[PairedSample]
431:     ) -> Tuple[List[PairedSample], List[PairedSample], List[PairedSample]]:
432:         """
433:         Rule-Based MMAct Task 1 Cross-Scene Protocol:
434:         Train on scenes 1 and 2. Validation on scenes 3 and 4.
435:         """
436:         all_scenes = sorted(list({s.scene for s in samples}))
437:         has_eval_scenes = any(sc in ["scene3", "scene4"] for sc in all_scenes)
438:         if has_eval_scenes:
439:             train = [s for s in samples if s.scene in ["scene1", "scene2"]]
440:             val_all = [s for s in samples if s.scene in ["scene3", "scene4"]]
441:         elif len(all_scenes) >= 2:
442:             train = [s for s in samples if s.scene == all_scenes[0]]
443:             val_all = [s for s in samples if s.scene != all_scenes[0]]
444:         else:
445:             return self._split_by_session(samples)
446: 
447:         if getattr(self.config, "partition_val_for_

In [33]:
from pathlib import Path
from collections import Counter
from src.config import SplitConfig, VERIFIED_TARGET_CLASSES
from src.data.indexer import MMActIndexer
from src.data.splits import DatasetSplitter

ROOT = Path("/kaggle/working/data/genuine_mmact/trimmed")
REQUIRED = ["acc_phone", "acc_watch", "gyro"]

# Inspect actual defaults
cfg = SplitConfig(
    split_strategy="subject",
    dataset_root=ROOT,
    seed=42
)

print("CONFIGURED SUBJECTS")
print("Train:", cfg.train_subjects)
print("Validation:", cfg.val_subjects)
print("Test:", cfg.test_subjects)

# Build index exactly as training script does
report = MMActIndexer(dataset_root=ROOT).index()

# Remove missing and empty required sensor files
usable = []

for sample in report.paired_samples:
    valid = True

    for modality in REQUIRED:
        path = sample.sensor_paths.get(modality)
        if path is None or not path.exists() or path.stat().st_size == 0:
            valid = False
            break

    if valid:
        usable.append(sample)

print("\nINDEX COUNTS")
print("Original indexed:", len(report.paired_samples))
print("After empty-file filtering:", len(usable))

# Run actual project splitter
splitter = DatasetSplitter(cfg, dataset_root=ROOT)
train, val, test = splitter.split(usable)

def subjects(samples):
    return sorted({s.subject for s in samples})

print("\nACTUAL SPLIT")
for name, samples in [
    ("Train", train),
    ("Validation", val),
    ("Test", test)
]:
    print(f"\n{name}: {len(samples)} samples")
    print("Subjects:", subjects(samples))
    print("Classes:", len({s.action for s in samples}))

print("\nEXPECTED SUBJECT ASSIGNMENT")
expected = {
    "Train": {f"subject{i}" for i in range(1, 15)},
    "Validation": {f"subject{i}" for i in range(15, 18)},
    "Test": {f"subject{i}" for i in range(18, 21)}
}

actual = {
    "Train": set(subjects(train)),
    "Validation": set(subjects(val)),
    "Test": set(subjects(test))
}

for name in expected:
    print(name, "MATCH:", actual[name] == expected[name])

print("\nSPLIT COUNTS:", len(train), len(val), len(test))

CONFIGURED SUBJECTS
Train: ['subject1', 'subject2', 'subject3', 'subject4', 'subject5', 'subject6', 'subject7', 'subject8', 'subject9', 'subject10', 'subject11', 'subject12', 'subject13', 'subject14']
Validation: ['subject15', 'subject16', 'subject17']
Test: ['subject18', 'subject19', 'subject20']

INDEX COUNTS
Original indexed: 5431
After empty-file filtering: 5427

ACTUAL SPLIT

Train: 3795 samples
Subjects: ['subject1', 'subject10', 'subject11', 'subject12', 'subject13', 'subject14', 'subject2', 'subject3', 'subject4', 'subject5', 'subject6', 'subject7', 'subject8', 'subject9']
Classes: 10

Validation: 780 samples
Subjects: ['subject15', 'subject16', 'subject17']
Classes: 10

Test: 852 samples
Subjects: ['subject18', 'subject19', 'subject20']
Classes: 10

EXPECTED SUBJECT ASSIGNMENT
Train MATCH: True
Validation MATCH: True
Test MATCH: True

SPLIT COUNTS: 3795 780 852


In [41]:
from pathlib import Path
import re
import py_compile
import shutil

p = Path("/kaggle/working/HAR-MMAct/scripts/train.py")
original_backup = p.with_name("train.py.bak")

assert original_backup.is_file(), (
    f"Original backup not found: {original_backup}. Stop; do not overwrite anything."
)

# Preserve the current broken version.
broken_backup = p.with_name("train.py.indentation_broken.bak")
shutil.copy2(p, broken_backup)

# Restore the source from before the first patch.
source = original_backup.read_text()

index_line = "report = indexer.index()"
split_pattern = re.compile(
    r"(?m)^([ \t]*)train_samples, val_samples, test_samples = "
    r"splitter\.split\(report\.paired_samples\)[ \t]*$"
)

assert source.count(index_line) == 1, "Original index line not uniquely found"
assert len(split_pattern.findall(source)) == 1, (
    "Original split call not uniquely found in backup"
)

# Derive indentation from the original index line.
match = re.search(r"(?m)^([ \t]*)report = indexer\.index\(\)$", source)
indent = match.group(1)

filter_block = (
    f"{indent}# Filter missing or empty required sensor files before splitting.\n"
    f"{indent}required_sensor_modalities = ['acc_phone', 'acc_watch', 'gyro']\n"
    f"{indent}usable_samples = []\n"
    f"{indent}for sample in report.paired_samples:\n"
    f"{indent}    valid = True\n"
    f"{indent}    for modality in required_sensor_modalities:\n"
    f"{indent}        sensor_path = sample.sensor_paths.get(modality)\n"
    f"{indent}        if sensor_path is None:\n"
    f"{indent}            valid = False\n"
    f"{indent}            break\n"
    f"{indent}        sensor_path = Path(sensor_path)\n"
    f"{indent}        if not sensor_path.is_file() or sensor_path.stat().st_size == 0:\n"
    f"{indent}            valid = False\n"
    f"{indent}            break\n"
    f"{indent}    if valid:\n"
    f"{indent}        usable_samples.append(sample)\n"
    f"{indent}print(f'Indexed samples: {{len(report.paired_samples)}}')\n"
    f"{indent}print(f'Usable samples: {{len(usable_samples)}}')\n"
    f"{indent}print(f'Excluded samples: {{len(report.paired_samples) - len(usable_samples)}}')\n"
)

source = source.replace(
    index_line,
    index_line + "\n" + filter_block.rstrip("\n"),
    1
)

source, replacements = split_pattern.subn(
    lambda m: (
        f"{m.group(1)}train_samples, val_samples, test_samples = "
        f"splitter.split(usable_samples)"
    ),
    source
)

assert replacements == 1, "Could not safely replace the original split call"

p.write_text(source)

# Compile before proceeding.
py_compile.compile(str(p), doraise=True)

print("PASS: Original training script restored and patched")
print("PASS: Filtering block inserted at the correct indentation")
print("PASS: Splitter uses usable_samples")
print("PASS: Python syntax compilation")
print(f"Original backup: {original_backup}")
print(f"Previous broken version: {broken_backup}")

PASS: Original training script restored and patched
PASS: Filtering block inserted at the correct indentation
PASS: Splitter uses usable_samples
PASS: Python syntax compilation
Original backup: /kaggle/working/HAR-MMAct/scripts/train.py.bak
Previous broken version: /kaggle/working/HAR-MMAct/scripts/train.py.indentation_broken.bak


In [45]:
import torch
import subprocess

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA available:", torch.cuda.is_available())
print("Allocated GPU memory:", round(torch.cuda.memory_allocated() / 1024**2, 2), "MB")

!nvidia-smi

GPU: Tesla T4
CUDA available: True
Allocated GPU memory: 0.0 MB
Sat Nov  7 17:27:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   64C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |        

In [46]:
from pathlib import Path

p = Path("/kaggle/working/HAR-MMAct/src/data/video_loader.py")
for i, line in enumerate(p.read_text().splitlines(), 1):
    if any(x in line.lower() for x in ["cache", "frame", "resize", "read", "video"]):
        print(f"{i}: {line}")

2: video_loader.py - Frame Extraction, Sampling, and Preprocessing for MMAct Videos
10: from src.config import VideoConfig
13: class VideoLoader:
14:     """Loads video files, uniformly samples fixed frames, resizes, and normalizes."""
16:     def __init__(self, config: Optional[VideoConfig] = None):
17:         self.config = config or VideoConfig()
21:     def load_frames(self, video_path: Union[str, Path]) -> np.ndarray:
23:         Reads raw video frames from path.
24:         Returns: np.ndarray of shape (num_frames_target, H, W, 3) in RGB format.
25:         Features robust fallback for unindexed/corrupted containers where CAP_PROP_FRAME_COUNT is 0 or invalid.
27:         path_str = str(video_path)
28:         cap = cv2.VideoCapture(path_str)
31:             raise IOError(f"Failed to open video file: {path_str}")
33:         t = self.config.num_frames
34:         total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
36:         # Check if frame count property is reliable (> 0)
37:

In [47]:
from pathlib import Path

files = [
    "/kaggle/working/HAR-MMAct/src/data/dataset.py",
    "/kaggle/working/HAR-MMAct/src/data/video_loader.py",
    "/kaggle/working/HAR-MMAct/src/config.py",
]

for file in files:
    p = Path(file)
    print(f"\n{'='*25} {p.name} {'='*25}")
    lines = p.read_text().splitlines()

    if p.name == "dataset.py":
        start, end = 1, min(len(lines), 260)
    elif p.name == "video_loader.py":
        start, end = 1, min(len(lines), 150)
    else:
        matches = [
            i for i, line in enumerate(lines)
            if "VideoConfig" in line or "num_frames" in line
        ]
        start = max(0, min(matches, default=0) - 5)
        end = min(len(lines), start + 80)

    for i in range(start, end):
        print(f"{i+1:4}: {lines[i]}")


========================= dataset.py =========================
   2: dataset.py - PyTorch Multimodal Dataset with Atomic Preprocessing Cache for HAR_MMAct
   3: 
   4: Features:
   5: - Deterministic, versioned, and configuration-aware cache keys (sample_id, video params, sensor params).
   6: - Atomic disk caching (temporary file write + os.replace) to prevent partial/corrupted entries.
   7: - Integrity verification on cache load (shape checking, NaN detection, corrupted file removal).
   8: - Storage safety check (verifies sufficient free space via shutil.disk_usage before caching).
   9: - Reusable across all model experiments (Custom CNN, Tuned CNN, MobileNetV2, InceptionTime, Multimodal).
  10: - Cache hit/miss/write/corruption metrics tracking.
  11: - Pre-caching warm-up utility.
  12: """
  13: 
  14: import hashlib
  15: import os
  16: import shutil
  17: from pathlib import Path
  18: from typing import Dict, List, Optional, Tuple, Union
  19: import torch
  20: from torch

In [48]:
from pathlib import Path

root = Path("/kaggle/working/HAR-MMAct/cache/smoke_multimodal")

for split in ["train", "val"]:
    folder = root / split
    files = list(folder.glob("*.pt")) if folder.exists() else []
    size_gb = sum(f.stat().st_size for f in files) / (1024**3)

    print(f"{split}:")
    print(f"  Cached samples: {len(files)}")
    print(f"  Cache size: {size_gb:.2f} GB")

train:
  Cached samples: 1212
  Cache size: 2.72 GB
val:
  Cached samples: 0
  Cache size: 0.00 GB


In [49]:
%cd /kaggle/working/HAR-MMAct

!python scripts/train.py \
    --data_root /kaggle/working/data/genuine_mmact/trimmed \
    --epochs 1 \
    --min_epochs 1 \
    --batch_size 8 \
    --num_workers 2 \
    --mode multimodal \
    --video_backbone mobilenet_v2 \
    --sensor_backbone inception_time \
    --fusion_strategy concat \
    --output_dir /kaggle/working/HAR-MMAct/outputs/smoke_multimodal \
    --cache_dir /kaggle/working/HAR-MMAct/cache/smoke_multimodal \
    --checkpoint_name smoke_multimodal.pth

/kaggle/working
 HAR_MMAct: Multimodal Human Activity Recognition Training Pipeline
 Mode            : multimodal
 Video Backbone  : mobilenet_v2
 Sensor Backbone : inception_time
 Split Strategy  : subject
 Epochs          : 1 | Batch Size: 8 | LR: 0.0001
 Random Seed     : 42

[Step 1] Indexing dataset at: /kaggle/working/data/genuine_mmact/trimmed
Indexed samples: 5431
Usable samples: 5427
Excluded samples: 4
  - Total Videos Scanned   : 17820
  - Total Sensors Scanned  : 35685
  - Paired Samples Created : 5431
  - Unknown Labels Filtered: 36388

[Step 2] Partitioning into subject-independent splits...
 SPLIT AUDIT & INTEGRITY REPORT
Strategy                : subject
Official Split Protocol : False
Custom Test Partition   : False
Train Source File       : N/A (Rule-based)
Val Source File         : N/A (Rule-based)
Train Samples / Records : 3795 samples (595 recordings)
Val Samples / Records   : 780 samples (124 recordings)
Test Samples / Records  : 852 samples (132 recordings)
-----

In [ ]:
from collections import Counter

# Preserve the original subject-independent split
TRAIN_SUBJECTS = {f"subject{i}" for i in range(1, 15)}
VAL_SUBJECTS = {f"subject{i}" for i in range(15, 18)}
TEST_SUBJECTS = {f"subject{i}" for i in range(18, 21)}

all_samples = report.paired_samples

train_samples = [
    s for s in all_samples if s.subject in TRAIN_SUBJECTS
]
val_samples = [
    s for s in all_samples if s.subject in VAL_SUBJECTS
]
test_samples = [
    s for s in all_samples if s.subject in TEST_SUBJECTS
]

# Verify no samples were lost or assigned twice
assert len(train_samples) + len(val_samples) + len(test_samples) == len(all_samples)
assert len({s.sample_id for s in all_samples}) == len(all_samples)

for name, samples in [
    ("Train", train_samples),
    ("Validation", val_samples),
    ("Test", test_samples),
]:
    print(f"\n{name}: {len(samples)} samples")
    print("Subjects:", sorted({s.subject for s in samples}))
    print("Class counts:", dict(sorted(Counter(s.label_idx for s in samples).items())))

print("\nTotal:", len(all_samples))

In [ ]:
from pathlib import Path
from src.data.dataset import MMActMultimodalDataset

CACHE_DIR = CACHE_ROOT
CACHE_DIR.mkdir(parents=True, exist_ok=True)

train_ds = MMActMultimodalDataset(
    train_samples,
    cache_dir=CACHE_DIR / "train",
    enable_cache=True,
)

val_ds = MMActMultimodalDataset(
    val_samples,
    cache_dir=CACHE_DIR / "val",
    enable_cache=True,
)

test_ds = MMActMultimodalDataset(
    test_samples,
    cache_dir=CACHE_DIR / "test",
    enable_cache=True,
)

print("Train:", len(train_ds))
print("Validation:", len(val_ds))
print("Test:", len(test_ds))

In [ ]:
import pandas as pd
from pathlib import Path
from collections import Counter, defaultdict

modalities = ["acc_phone", "acc_watch", "gyro"]

original_splits = {
    "train": list(train_samples),
    "val": list(val_samples),
    "test": list(test_samples),
}

audit = {}
invalid_samples = defaultdict(list)
failure_counts = Counter()

for split_name, samples in original_splits.items():
    valid_samples = []

    for sample in samples:
        errors = []

        for modality in modalities:
            path = sample.sensor_paths.get(modality)

            if path is None or not Path(path).is_file():
                errors.append(f"{modality}:missing")
                continue

            try:
                if Path(path).stat().st_size == 0:
                    errors.append(f"{modality}:empty_file")
                    continue

                df = pd.read_csv(path)

                if df.empty:
                    errors.append(f"{modality}:empty_csv")
                elif len(df.columns) < 4:
                    errors.append(f"{modality}:insufficient_columns")

            except Exception as exc:
                errors.append(
                    f"{modality}:read_error:{type(exc).__name__}"
                )

        if errors:
            invalid_samples[split_name].append((sample, errors))
            for error in errors:
                failure_counts[":".join(error.split(":")[:2])] += 1
        else:
            valid_samples.append(sample)

    audit[split_name] = valid_samples

    print(
        f"{split_name}: {len(valid_samples)}/{len(samples)} valid; "
        f"{len(samples)-len(valid_samples)} invalid"
    )

print("\nFailure counts:", dict(failure_counts))

print("\nInvalid sample details:")
for split_name, records in invalid_samples.items():
    for sample, errors in records:
        print(split_name, sample.sample_id, errors)

print("\nOriginal sample lists preserved:")
for name, samples in original_splits.items():
    print(name, len(samples))

In [ ]:
# Apply audited sample lists
train_samples = audit["train"]
val_samples = audit["val"]
test_samples = audit["test"]

print("Corrected split counts:")
print("Train:", len(train_samples))
print("Validation:", len(val_samples))
print("Test:", len(test_samples))
print("Total:", len(train_samples) + len(val_samples) + len(test_samples))

# Verify all 10 classes remain
EXPECTED_CLASSES = set(range(10))

for name, samples in [
    ("Train", train_samples),
    ("Validation", val_samples),
    ("Test", test_samples),
]:
    observed = {s.label_idx for s in samples}
    missing = EXPECTED_CLASSES - observed

    print(f"{name}: {len(observed)}/10 classes")

    if missing:
        raise RuntimeError(f"{name} missing classes: {sorted(missing)}")

print("\n[PASS] Filtered splits retain all 10 classes.")

In [ ]:
from src.data.dataset import MMActMultimodalDataset
from pathlib import Path

CACHE_DIR = CACHE_ROOT
CACHE_DIR.mkdir(parents=True, exist_ok=True)

train_ds = MMActMultimodalDataset(
    train_samples,
    cache_dir=CACHE_DIR / "train",
    enable_cache=True,
)

val_ds = MMActMultimodalDataset(
    val_samples,
    cache_dir=CACHE_DIR / "val",
    enable_cache=True,
)

test_ds = MMActMultimodalDataset(
    test_samples,
    cache_dir=CACHE_DIR / "test",
    enable_cache=True,
)

print("Datasets ready:", len(train_ds), len(val_ds), len(test_ds))

In [59]:

!nvidia-smi

Sat Nov  7 18:02:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   76C    P0             31W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [62]:
import torch
import os

print("PyTorch CUDA:", torch.version.cuda)
print("CUDA devices:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))

PyTorch CUDA: 13.0
CUDA devices: 2
0 Tesla T4
1 Tesla T4


In [64]:
import torch
from pathlib import Path

root = Path("/kaggle/working/HAR-MMAct")

print("CUDA:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

for path in [
    root / "scripts/train.py",
    root / "cache/smoke_multimodal/train",
    root / "cache/smoke_multimodal/val",
    root / "outputs/exp1_multimodal",
]:
    if path.is_dir():
        files = list(path.glob("*"))
        print(f"{path}: {len(files)} files")
    else:
        print(f"{path}: EXISTS={path.exists()}")

CUDA: True
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4
/kaggle/working/HAR-MMAct/scripts/train.py: EXISTS=True
/kaggle/working/HAR-MMAct/cache/smoke_multimodal/train: 3600 files
/kaggle/working/HAR-MMAct/cache/smoke_multimodal/val: 0 files
/kaggle/working/HAR-MMAct/outputs/exp1_multimodal: 2 files


In [65]:
from pathlib import Path

root = Path("/kaggle/working/HAR-MMAct/cache/smoke_multimodal")

for split in ["train", "val"]:
    files = list((root / split).glob("*.pt"))
    size = sum(f.stat().st_size for f in files) / 1024**3
    print(f"{split}: {len(files)} files | {size:.2f} GB")

train: 3600 files | 8.09 GB
val: 0 files | 0.00 GB


In [66]:
from pathlib import Path

root = Path("/kaggle/working/HAR-MMAct/outputs/exp1_multimodal")

print("EXPERIMENT OUTPUT FILES")

for p in root.rglob("*"):
    if p.is_file():
        print(
            f"{p.relative_to(root)} | "
            f"{p.stat().st_size / 1024**2:.2f} MB"
        )

print("\nALL CHECKPOINTS")

for p in Path("/kaggle/working/HAR-MMAct/outputs").rglob("*.pth"):
    print(p, f"{p.stat().st_size / 1024**2:.2f} MB")

EXPERIMENT OUTPUT FILES

ALL CHECKPOINTS


start a fresh 50-epoch run

In [67]:
%cd /kaggle/working/HAR-MMAct

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

!python scripts/train.py \
    --data_root /kaggle/working/data/genuine_mmact/trimmed \
    --epochs 50 \
    --min_epochs 50 \
    --batch_size 8 \
    --num_workers 2 \
    --mode multimodal \
    --video_backbone mobilenet_v2 \
    --sensor_backbone inception_time \
    --fusion_strategy concat \
    --output_dir /kaggle/working/HAR-MMAct/outputs/exp1_multimodal \
    --cache_dir /kaggle/working/HAR-MMAct/cache/smoke_multimodal \
    --checkpoint_name exp1_multimodal_best.pth

/kaggle/working
 HAR_MMAct: Multimodal Human Activity Recognition Training Pipeline
 Mode            : multimodal
 Video Backbone  : mobilenet_v2
 Sensor Backbone : inception_time
 Split Strategy  : subject
 Epochs          : 50 | Batch Size: 8 | LR: 0.0001
 Random Seed     : 42

[Step 1] Indexing dataset at: /kaggle/working/data/genuine_mmact/trimmed
Indexed samples: 5431
Usable samples: 5427
Excluded samples: 4
  - Total Videos Scanned   : 17820
  - Total Sensors Scanned  : 35685
  - Paired Samples Created : 5431
  - Unknown Labels Filtered: 36388

[Step 2] Partitioning into subject-independent splits...
 SPLIT AUDIT & INTEGRITY REPORT
Strategy                : subject
Official Split Protocol : False
Custom Test Partition   : False
Train Source File       : N/A (Rule-based)
Val Source File         : N/A (Rule-based)
Train Samples / Records : 3795 samples (595 recordings)
Val Samples / Records   : 780 samples (124 recordings)
Test Samples / Records  : 852 samples (132 recordings)
----

In [68]:
from pathlib import Path
import json
import subprocess

ROOT = Path("/kaggle/working/HAR-MMAct")
OUT = ROOT / "outputs/exp1_multimodal"
CKPT = OUT / "checkpoints/exp1_multimodal_best.pth"

print("=" * 65)
print("MULTIMODAL EXPERIMENT — ARTIFACT VERIFICATION")
print("=" * 65)

# 1. Checkpoint
print("\nCHECKPOINT")
print("Exists:", CKPT.is_file())
if CKPT.is_file():
    print("Size (MB):", round(CKPT.stat().st_size / 1024**2, 2))

# 2. Experiment outputs
print("\nEXPERIMENT OUTPUTS")
for p in sorted(OUT.rglob("*")):
    if p.is_file():
        print(f"{p.relative_to(ROOT)} | {p.stat().st_size / 1024:.1f} KB")

# 3. Git repository state
print("\nGITHUB / GIT STATUS")
for args in [
    ["git", "remote", "-v"],
    ["git", "branch", "--show-current"],
    ["git", "status", "--short"],
]:
    result = subprocess.run(
        args, cwd=ROOT, capture_output=True, text=True
    )
    print("$", " ".join(args))
    print(result.stdout.strip() or result.stderr.strip() or "(empty)")

assert CKPT.is_file(), "STOP: Best checkpoint is missing."
print("\nPASS: Best checkpoint exists. Do not rerun training.")

MULTIMODAL EXPERIMENT — ARTIFACT VERIFICATION

CHECKPOINT
Exists: True
Size (MB): 32.95

EXPERIMENT OUTPUTS
outputs/exp1_multimodal/checkpoints/best_model.pth | 33730.9 KB
outputs/exp1_multimodal/checkpoints/exp1_multimodal_best.pth | 33740.5 KB
outputs/exp1_multimodal/checkpoints/latest_model.pth | 33732.9 KB
outputs/exp1_multimodal/checkpoints/latest_model_mobilenet_v2.pth | 33745.4 KB
outputs/exp1_multimodal/experiment_manifest.json | 2.8 KB
outputs/exp1_multimodal/experiment_manifest_mobilenet_v2.json | 2.8 KB
outputs/exp1_multimodal/history.json | 7.7 KB
outputs/exp1_multimodal/history_mobilenet_v2.csv | 2.4 KB
outputs/exp1_multimodal/history_mobilenet_v2.json | 7.7 KB
outputs/exp1_multimodal/plots/generalization_gap.png | 88.0 KB
outputs/exp1_multimodal/plots/generalization_gap_mobilenet_v2.png | 88.0 KB
outputs/exp1_multimodal/plots/learning_rate_schedule.png | 87.1 KB
outputs/exp1_multimodal/plots/learning_rate_schedule_mobilenet_v2.png | 87.1 KB
outputs/exp1_multimodal/plots/t

## Section 6: Video & Sensor Preprocessing Cache Warm-Up
Pre-caches normalized 16-frame 112x112 video tensors and aligned 100-sample 9-axis sensor windows to `/kaggle/working/har_mmact_cache/` so all 6 subsequent experiments execute with maximum speed without repeated OpenCV decoding. Checks disk capacity before caching.


In [ ]:

# =============================================================================
# SECTION 6: RESUMABLE VIDEO & SENSOR PREPROCESSING CACHE WARM-UP
#
# Features:
# - Uses the audited 3-modality dataset
# - Processes samples in batches of 50
# - Saves progress after every batch
# - Resumes after Kaggle runtime interruption
# - Reuses valid dataset cache entries
# - Records preprocessing failures
# - Preserves subject-independent splits
# - Verifies all 10 classes
# - Stops safely when disk space is low
# =============================================================================

import os
import json
import time
import shutil
import hashlib
import pandas as pd
from pathlib import Path
from collections import Counter

from src.data.dataset import MMActMultimodalDataset

# ---------------------------------------------------------------------------
# 0. Configuration
# ---------------------------------------------------------------------------

CACHE_DIR = CACHE_ROOT
PROGRESS_DIR = CACHE_DIR / "progress"
PROGRESS_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 50
MIN_FREE_GB = 2.0

REQUIRED_MODALITIES = ["acc_phone", "acc_watch", "gyro"]
EXPECTED_CLASSES = set(range(10))

SPLITS = {
    "train": list(train_samples),
    "val": list(val_samples),
    "test": list(test_samples),
}

print("=" * 78)
print("SECTION 6: RESUMABLE PREPROCESSING CACHE WARM-UP")
print("=" * 78)

# ---------------------------------------------------------------------------
# 1. Validate input splits
# ---------------------------------------------------------------------------

for split_name, samples in SPLITS.items():

    if not samples:
        raise RuntimeError(f"{split_name} split is empty.")

    observed = {int(s.label_idx) for s in samples}
    missing = EXPECTED_CLASSES - observed

    if missing:
        raise RuntimeError(
            f"{split_name} missing classes: {sorted(missing)}"
        )

    for sample in samples:
        missing_modalities = (
            set(REQUIRED_MODALITIES) - set(sample.sensor_paths.keys())
        )

        if missing_modalities:
            raise RuntimeError(
                f"{sample.sample_id} missing modalities: "
                f"{sorted(missing_modalities)}"
            )

    print(
        f"[PASS] {split_name}: {len(samples)} samples, "
        f"{len(observed)}/10 classes"
    )

# ---------------------------------------------------------------------------
# 2. Stable split fingerprint
# ---------------------------------------------------------------------------

def split_fingerprint(samples):
    content = "\n".join(
        sorted(
            f"{s.sample_id}|{s.video_path}|"
            + "|".join(
                str(s.sensor_paths[m]) for m in REQUIRED_MODALITIES
            )
            for s in samples
        )
    )

    return hashlib.sha256(content.encode()).hexdigest()


# ---------------------------------------------------------------------------
# 3. Dataset creation
# ---------------------------------------------------------------------------

def create_dataset(samples, split_name):

    return MMActMultimodalDataset(
        samples,
        cache_dir=CACHE_DIR / split_name,
        enable_cache=True,
    )


# ---------------------------------------------------------------------------
# 4. Progress manifest
# ---------------------------------------------------------------------------

def progress_path(split_name):
    return PROGRESS_DIR / f"{split_name}_progress.json"


def load_progress(split_name, samples):

    path = progress_path(split_name)
    fingerprint = split_fingerprint(samples)

    if not path.exists():
        return {
            "fingerprint": fingerprint,
            "completed_ids": [],
            "failures": [],
        }

    try:
        data = json.loads(path.read_text())

        if data.get("fingerprint") != fingerprint:
            print(
                f"[WARNING] {split_name}: split fingerprint changed. "
                "Starting a fresh progress record."
            )

            return {
                "fingerprint": fingerprint,
                "completed_ids": [],
                "failures": [],
            }

        return data

    except Exception as exc:
        print(
            f"[WARNING] Could not read {split_name} progress: {exc}"
        )

        return {
            "fingerprint": fingerprint,
            "completed_ids": [],
            "failures": [],
        }


def save_progress(split_name, data):

    path = progress_path(split_name)
    temporary_path = path.with_suffix(".tmp")

    temporary_path.write_text(
        json.dumps(data, indent=2)
    )

    os.replace(temporary_path, path)


# ---------------------------------------------------------------------------
# 5. Warm one split with resumable checkpoints
# ---------------------------------------------------------------------------

def warm_split(samples, split_name):

    print("\n" + "-" * 78)
    print(f"STARTING {split_name.upper()}: {len(samples)} samples")
    print("-" * 78)

    dataset = create_dataset(samples, split_name)

    progress = load_progress(split_name, samples)

    completed_ids = set(progress["completed_ids"])
    failure_ids = {
        failure["sample_id"]
        for failure in progress["failures"]
    }

    # Only skip samples explicitly marked complete.
    remaining_indices = [
        idx
        for idx, sample in enumerate(samples)
        if sample.sample_id not in completed_ids
        and sample.sample_id not in failure_ids
    ]

    print("Previously completed:", len(completed_ids))
    print("Previously failed:", len(failure_ids))
    print("Remaining:", len(remaining_indices))

    new_failures = []

    for batch_start in range(0, len(remaining_indices), BATCH_SIZE):

        batch_indices = remaining_indices[
            batch_start:batch_start + BATCH_SIZE
        ]

        batch_start_time = time.time()

        for idx in batch_indices:

            sample = samples[idx]

            try:
                # Dataset handles cache validation and preprocessing.
                video_tensor, sensor_tensor, label, sample_id = dataset[idx]

                # Validate returned objects before marking complete.
                if video_tensor is None or sensor_tensor is None:
                    raise ValueError("Empty video or sensor tensor.")

                if int(label) != int(sample.label_idx):
                    raise ValueError("Returned label mismatch.")

                if str(sample_id) != str(sample.sample_id):
                    raise ValueError("Returned sample ID mismatch.")

                completed_ids.add(sample.sample_id)

            except Exception as exc:

                failure = {
                    "split": split_name,
                    "sample_id": str(sample.sample_id),
                    "label_idx": int(sample.label_idx),
                    "video_path": str(sample.video_path),
                    "reason": (
                        f"{type(exc).__name__}: {exc}"
                    ),
                }

                new_failures.append(failure)
                progress["failures"].append(failure)

                print(
                    f"[FAIL] {sample.sample_id}: "
                    f"{type(exc).__name__}: {exc}"
                )

        # Save progress after every batch.
        progress["completed_ids"] = sorted(completed_ids)

        save_progress(split_name, progress)

        processed = len(completed_ids) + len(failure_ids) + len(new_failures)

        free_gb = (
            shutil.disk_usage(str(CACHE_DIR)).free / (1024 ** 3)
        )

        print(
            f"[CHECKPOINT] {split_name}: "
            f"{processed}/{len(samples)} processed | "
            f"Successful: {len(completed_ids)} | "
            f"New failures: {len(new_failures)} | "
            f"Free disk: {free_gb:.2f} GB | "
            f"Batch time: {time.time() - batch_start_time:.1f}s"
        )

        if free_gb < MIN_FREE_GB:

            raise RuntimeError(
                f"Disk space below {MIN_FREE_GB} GB. "
                "Progress saved. Stop and inspect storage."
            )

    # -----------------------------------------------------------------------
    # Final split verification
    # -----------------------------------------------------------------------

    failed_ids = {
        failure["sample_id"]
        for failure in progress["failures"]
    }

    successful_samples = [
        sample
        for sample in samples
        if sample.sample_id in completed_ids
        and sample.sample_id not in failed_ids
    ]

    observed_classes = {
        int(sample.label_idx)
        for sample in successful_samples
    }

    missing_classes = EXPECTED_CLASSES - observed_classes

    if missing_classes:

        raise RuntimeError(
            f"{split_name} lost classes after preprocessing: "
            f"{sorted(missing_classes)}"
        )

    if len(completed_ids) + len(failed_ids) != len(samples):

        raise RuntimeError(
            f"{split_name}: progress accounting mismatch."
        )

    print(
        f"[PASS] {split_name}: "
        f"{len(successful_samples)} successful, "
        f"{len(failed_ids)} failed."
    )

    return successful_samples, dataset, progress


# ---------------------------------------------------------------------------
# 6. Warm all splits
# ---------------------------------------------------------------------------

train_samples, train_ds, train_progress = warm_split(
    SPLITS["train"], "train"
)

val_samples, val_ds, val_progress = warm_split(
    SPLITS["val"], "val"
)

test_samples, test_ds, test_progress = warm_split(
    SPLITS["test"], "test"
)


# ---------------------------------------------------------------------------
# 7. Save combined failure report
# ---------------------------------------------------------------------------

all_failures = (
    train_progress["failures"]
    + val_progress["failures"]
    + test_progress["failures"]
)

if all_failures:

    failure_path = CACHE_DIR / "runtime_preprocessing_failures.csv"

    pd.DataFrame(all_failures).to_csv(
        failure_path,
        index=False,
    )

    print("\nFailure report:", failure_path)

else:

    print("\nNo additional preprocessing failures.")


# ---------------------------------------------------------------------------
# 8. Final dataset reconstruction
# ---------------------------------------------------------------------------

# Remove failed samples consistently from all split lists.
def remove_failed_samples(samples, progress):

    failed_ids = {
        failure["sample_id"]
        for failure in progress["failures"]
    }

    return [
        sample
        for sample in samples
        if sample.sample_id not in failed_ids
    ]


train_samples = remove_failed_samples(
    SPLITS["train"], train_progress
)

val_samples = remove_failed_samples(
    SPLITS["val"], val_progress
)

test_samples = remove_failed_samples(
    SPLITS["test"], test_progress
)

# Rebuild final datasets with filtered samples.
train_ds = create_dataset(train_samples, "train")
val_ds = create_dataset(val_samples, "val")
test_ds = create_dataset(test_samples, "test")


# ---------------------------------------------------------------------------
# 9. Final verification
# ---------------------------------------------------------------------------

print("\n" + "=" * 78)
print("FINAL SECTION 6 SUMMARY")
print("=" * 78)

for name, samples, dataset in [
    ("TRAIN", train_samples, train_ds),
    ("VALIDATION", val_samples, val_ds),
    ("TEST", test_samples, test_ds),
]:

    counts = Counter(
        int(sample.label_idx) for sample in samples
    )

    print(f"\n{name}")
    print("Samples:", len(samples))
    print("Dataset length:", len(dataset))
    print("Class counts:", dict(sorted(counts.items())))
    print("Classes present:", len(counts))

    if len(dataset) != len(samples):
        raise RuntimeError(
            f"{name}: dataset/sample count mismatch."
        )

    if set(counts) != EXPECTED_CLASSES:
        raise RuntimeError(
            f"{name}: incomplete class coverage."
        )

print("\nTotal usable samples:",
      len(train_ds) + len(val_ds) + len(test_ds))

print("Additional runtime failures:", len(all_failures))

print("Remaining disk:",
      f"{shutil.disk_usage(str(CACHE_DIR)).free / (1024**3):.2f} GB")

print("\n[PASS] SECTION 6 COMPLETED")
print("Resumable video and sensor preprocessing finished.")


## Section 7: Memory-Safe Calibration & OOM Warm-Up
Executes a 1-batch calibration forward/backward pass with Automatic Mixed Precision (AMP) to verify GPU VRAM safety and establish baseline memory headroom before starting full 50-epoch training. Explicitly catches OOM conditions and frees temporary memory.


In [ ]:
# Section 7: Memory-Safe Calibration & OOM Warm-Up

import gc
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from src.models.fusion_model import MultimodalHARModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda"

if use_amp:
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats(device)

print("Running one-batch GPU memory calibration...")

calib_loader = DataLoader(
    train_ds,
    batch_size=8,
    shuffle=False,
    num_workers=0,
    pin_memory=use_amp
)

v_sample, s_sample, l_sample, _ = next(iter(calib_loader))

print("Video batch shape :", tuple(v_sample.shape))
print("Sensor batch shape:", tuple(s_sample.shape))
print("Label batch shape :", tuple(l_sample.shape))

model_calib = MultimodalHARModel(
    num_classes=10,
    video_backbone="mobilenet_v2",
    sensor_backbone="inception_time",
    fusion_hidden_dim=128
).to(device)

optimizer = torch.optim.Adam(model_calib.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

try:
    v_gpu = v_sample.to(device, non_blocking=use_amp)
    s_gpu = s_sample.to(device, non_blocking=use_amp)
    l_gpu = l_sample.to(device, non_blocking=use_amp)

    optimizer.zero_grad(set_to_none=True)

    with torch.amp.autocast(
        device_type=device.type,
        dtype=torch.float16,
        enabled=use_amp
    ):
        out = model_calib(
            video=v_gpu,
            sensor=s_gpu,
            override_mode="multimodal"
        )
        loss = criterion(out, l_gpu)

    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()

    print(f"Calibration loss: {loss.item():.4f}")

    if use_amp:
        peak_mb = torch.cuda.max_memory_allocated(device) / (1024 ** 2)
        total_mb = torch.cuda.get_device_properties(device).total_memory / (1024 ** 2)
        free_mb, _ = torch.cuda.mem_get_info(device)

        print(f"Peak allocated VRAM : {peak_mb:.2f} MB")
        print(f"Total GPU VRAM      : {total_mb:.2f} MB")
        print(f"Currently free VRAM : {free_mb:.2f} MB")

    print("[PASS] One-batch forward and backward completed.")

except torch.cuda.OutOfMemoryError as e:
    print("[OOM] Batch size 8 exceeded available GPU memory.")
    print("Use batch size 4, then rerun calibration.")
    raise

finally:
    for name in ["v_gpu", "s_gpu", "l_gpu", "out", "loss"]:
        if name in locals():
            del locals()[name]

    del model_calib, optimizer, criterion, scaler
    del calib_loader, v_sample, s_sample, l_sample

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
from pathlib import Path

p = Path("scripts/train.py")
source = p.read_text()

old_index = """    report = indexer.index()

    print(f"  - Total Videos Scanned   : {report.total_video_files_found}")"""

new_index = """    report = indexer.index()

    # Remove samples with missing or empty required sensor files.
    required_modalities = ("acc_phone", "acc_watch", "gyro")
    original_count = len(report.paired_samples)

    valid_samples = []
    rejected_samples = []

    for sample in report.paired_samples:
        valid = True
        for modality in required_modalities:
            sensor_path = sample.sensor_paths.get(modality)
            if not sensor_path:
                valid = False
                break
            sensor_file = Path(sensor_path)
            if not sensor_file.is_file() or sensor_file.stat().st_size == 0:
                valid = False
                break

        if valid:
            valid_samples.append(sample)
        else:
            rejected_samples.append(sample.sample_id)

    print(f"  - Samples before sensor validation: {original_count}")
    print(f"  - Samples rejected (missing/empty sensors): {len(rejected_samples)}")
    print(f"  - Samples retained: {len(valid_samples)}")

    print(f"  - Total Videos Scanned   : {report.total_video_files_found}")"""

old_split = "    train_samples, val_samples, test_samples = splitter.split(report.paired_samples)"
new_split = "    train_samples, val_samples, test_samples = splitter.split(valid_samples)"

assert source.count(old_index) == 1, "Index insertion point not unique; no changes made."
assert source.count(old_split) == 1, "Split line not unique; no changes made."

source = source.replace(old_index, new_index, 1)
source = source.replace(old_split, new_split, 1)
p.write_text(source)

print("PATCH APPLIED")
print("Training now filters missing/empty required sensor files before splitting.")
print("Original dataset index and source files remain unchanged.")

## Section 8: Experiment 1 - Sensor-Only InceptionTime Baseline (50 Epochs)
Trains InceptionTime 1D convolutional network on 9-axis sensor time series (phone accelerometer, watch accelerometer, gyroscope) for 50 epochs.


In [74]:
# Section 8: Experiment 2 - Sensor-Only InceptionTime Baseline

from pathlib import Path

EPOCHS = 50
out_sensor = 'outputs/exp2_sensor_only'
Path(out_sensor).mkdir(parents=True, exist_ok=True)

print(f'=== Starting Experiment 2: Sensor-Only Baseline ({EPOCHS} Epochs) ===')

!python scripts/train.py \
    --data_root /kaggle/working/data/genuine_mmact \
    --mode sensor_only \
    --sensor_backbone inception_time \
    --epochs {EPOCHS} \
    --min_epochs 50 \
    --batch_size 8 \
    --lr 0.0005 \
    --weight_decay 0.0001 \
    --split_strategy subject \
    --checkpoint_name best_model_sensor.pth \
    --output_dir {out_sensor} \
    --cache_dir {CACHE_ROOT} \
    --benchmark_warmup \
    --auto_resume

=== Starting Experiment 2: Sensor-Only Baseline (50 Epochs) ===
 HAR_MMAct: Multimodal Human Activity Recognition Training Pipeline
 Mode            : sensor_only
 Video Backbone  : cnn
 Sensor Backbone : inception_time
 Split Strategy  : subject
 Epochs          : 50 | Batch Size: 8 | LR: 0.0005
 Random Seed     : 42

[Step 1] Indexing dataset at: /kaggle/working/data/genuine_mmact
Indexed samples: 5431
Usable samples: 5427
Excluded samples: 4
  - Total Videos Scanned   : 17820
  - Total Sensors Scanned  : 35685
  - Paired Samples Created : 5431
  - Unknown Labels Filtered: 36388

[Step 2] Partitioning into subject-independent splits...
 SPLIT AUDIT & INTEGRITY REPORT
Strategy                : subject
Official Split Protocol : False
Custom Test Partition   : False
Train Source File       : N/A (Rule-based)
Val Source File         : N/A (Rule-based)
Train Samples / Records : 3795 samples (595 recordings)
Val Samples / Records   : 780 samples (124 recordings)
Test Samples / Records  : 8

In [83]:
!grep -n -A65 -B10 "torch.load(args.resume" /kaggle/working/HAR-MMAct/scripts/train.py

486-        "val_acc": [],
487-        "lr": [],
488-        "train_time_sec": [],
489-        "val_time_sec": [],
490-        "peak_gpu_mem_mb": [],
491-    }
492-    patience_counter = 0
493-
494-    if args.resume and Path(args.resume).exists():
495-        print(f"\n[Resume] Loading checkpoint from: {args.resume}")
496:        resume_ckpt = torch.load(args.resume, map_location=device)
497-        model.load_state_dict(resume_ckpt["model_state_dict"])
498-        if "optimizer_state_dict" in resume_ckpt:
499-            optimizer.load_state_dict(resume_ckpt["optimizer_state_dict"])
500-        if "scheduler_state_dict" in resume_ckpt and resume_ckpt["scheduler_state_dict"] and scheduler:
501-            scheduler.load_state_dict(resume_ckpt["scheduler_state_dict"])
502-        if "scaler_state_dict" in resume_ckpt and resume_ckpt["scaler_state_dict"] and scaler:
503-            scaler.load_state_dict(resume_ckpt["scaler_state_dict"])
504-        start_epoch = resume_ckpt.get("epoch"

In [84]:
# Section 8: Resume Experiment 2 - Sensor-Only

from pathlib import Path

EPOCHS = 50
out_sensor = 'outputs/exp2_sensor_only'
checkpoint = f'{out_sensor}/checkpoints/latest_model_cnn.pth'

assert Path(checkpoint).exists(), "Checkpoint missing!"

print("Resuming Sensor-Only Experiment from Epoch 4")

!python scripts/train.py \
    --data_root /kaggle/working/data/genuine_mmact \
    --mode sensor_only \
    --sensor_backbone inception_time \
    --epochs {EPOCHS} \
    --min_epochs 50 \
    --batch_size 8 \
    --lr 0.0005 \
    --weight_decay 0.0001 \
    --split_strategy subject \
    --checkpoint_name best_model_sensor.pth \
    --output_dir {out_sensor} \
    --cache_dir {CACHE_ROOT} \
    --resume {checkpoint}

Resuming Sensor-Only Experiment from Epoch 4
 HAR_MMAct: Multimodal Human Activity Recognition Training Pipeline
 Mode            : sensor_only
 Video Backbone  : cnn
 Sensor Backbone : inception_time
 Split Strategy  : subject
 Epochs          : 50 | Batch Size: 8 | LR: 0.0005
 Random Seed     : 42

[Step 1] Indexing dataset at: /kaggle/working/data/genuine_mmact
Indexed samples: 5431
Usable samples: 5427
Excluded samples: 4
  - Total Videos Scanned   : 17820
  - Total Sensors Scanned  : 35685
  - Paired Samples Created : 5431
  - Unknown Labels Filtered: 36388

[Step 2] Partitioning into subject-independent splits...
 SPLIT AUDIT & INTEGRITY REPORT
Strategy                : subject
Official Split Protocol : False
Custom Test Partition   : False
Train Source File       : N/A (Rule-based)
Val Source File         : N/A (Rule-based)
Train Samples / Records : 3795 samples (595 recordings)
Val Samples / Records   : 780 samples (124 recordings)
Test Samples / Records  : 852 samples (132 rec

In [85]:
import shutil

print(
    "Free disk:",
    round(shutil.disk_usage("/kaggle/working").free / 1024**3, 2),
    "GB"
)

Free disk: 8.95 GB


In [ ]:
from pathlib import Path

p = Path("outputs/sensor_only/checkpoints/best_model_sensor.pth")
print("Checkpoint exists:", p.exists())
print("Checkpoint size (MB):", round(p.stat().st_size / (1024 * 1024), 2) if p.exists() else "Missing")

print("\nOutput files:")
for f in sorted(Path("outputs/sensor_only").rglob("*")):
    if f.is_file():
        print(f, f.stat().st_size)

In [81]:
import torch
from pathlib import Path

folder = Path(
    "/kaggle/working/HAR-MMAct/outputs/exp2_sensor_only/checkpoints"
)

for name in ["latest_model.pth", "latest_model_cnn.pth", "best_model_sensor.pth"]:
    path = folder / name
    try:
        ckpt = torch.load(path, map_location="cpu", weights_only=False)
        print(f"\n{name}")
        print("Keys:", list(ckpt.keys()) if isinstance(ckpt, dict) else type(ckpt))

        if isinstance(ckpt, dict):
            for key in ["epoch", "best_epoch", "best_val_f1", "best_val_macro_f1"]:
                if key in ckpt:
                    print(f"{key}: {ckpt[key]}")
    except Exception as e:
        print(name, "Inspection error:", str(e))


latest_model.pth
Keys: ['epoch', 'model_state_dict', 'optimizer_state_dict', 'scheduler_state_dict', 'scaler_state_dict', 'train_loss', 'val_loss', 'val_acc', 'val_f1', 'classes', 'config', 'history', 'is_synthetic']
epoch: 3

latest_model_cnn.pth
Keys: ['epoch', 'model_state_dict', 'optimizer_state_dict', 'scheduler_state_dict', 'scaler_state_dict', 'train_loss', 'val_loss', 'val_acc', 'val_f1', 'classes', 'config', 'history', 'is_synthetic']
epoch: 3

best_model_sensor.pth
Keys: ['epoch', 'model_state_dict', 'optimizer_state_dict', 'scheduler_state_dict', 'scaler_state_dict', 'train_loss', 'val_loss', 'val_acc', 'val_f1', 'classes', 'config', 'history', 'is_synthetic']
epoch: 3


In [82]:
!grep -n -A45 -B10 "auto_resume" /kaggle/working/HAR-MMAct/scripts/train.py

85-    parser.add_argument("--video_backbone", type=str, default="cnn", choices=["cnn", "custom_cnn", "original_cnn", "improved_cnn", "mobilenet_v2", "resnet18", "lightweight"], help="Video backbone")
86-    parser.add_argument("--sensor_backbone", type=str, default="inception_time", choices=["inception_time", "cnn1d", "tcn"], help="Sensor backbone")
87-    parser.add_argument("--fusion_strategy", type=str, default="concat", choices=["concat", "gated"], help="Multimodal fusion strategy")
88-    parser.add_argument("--freeze_video", action="store_true", help="Freeze video backbone feature extractor")
89-    parser.add_argument("--staged_finetune", action="store_true", help="Enable staged fine-tuning for transfer learning (frozen head first, then unfreeze backbone)")
90-    parser.add_argument("--unfreeze_epoch", type=int, default=15, help="Epoch to unfreeze video backbone in staged fine-tuning")
91-    parser.add_argument("--unfreeze_lr", type=float, default=1e-5, help="Learning rate fo

## Section 9: Experiment 2 - Multimodal Late Fusion (50 Epochs)
Fuses MobileNetV2 video representations with InceptionTime sensor features via late fusion for 50 epochs.


In [86]:
from pathlib import Path

EPOCHS = 50
out_fusion = 'outputs/multimodal_fusion'

Path(out_fusion).mkdir(parents=True, exist_ok=True)

print("Epochs:", EPOCHS)
print("Output directory:", out_fusion)
print("Existing output:", list(Path(out_fusion).glob("*")))
print("Cache root:", CACHE_ROOT)

Epochs: 50
Output directory: outputs/multimodal_fusion
Existing output: []
Cache root: /kaggle/working/har_mmact_cache_20261107_154134


In [87]:
# Experiment 3: Multimodal Late Fusion — 50 Epochs

EPOCHS = 50
out_fusion = 'outputs/multimodal_fusion'

print(f'=== Experiment 3: Multimodal Late Fusion ({EPOCHS} Epochs) ===')

!python scripts/train.py \
    --data_root /kaggle/working/data/genuine_mmact \
    --mode multimodal \
    --video_backbone mobilenet_v2 \
    --sensor_backbone inception_time \
    --epochs {EPOCHS} \
    --min_epochs 50 \
    --batch_size 8 \
    --lr 0.0001 \
    --weight_decay 0.0005 \
    --split_strategy subject \
    --checkpoint_name best_model_multimodal.pth \
    --output_dir {out_fusion} \
    --cache_dir {CACHE_ROOT} \
    --benchmark_warmup \
    --auto_resume

=== Experiment 3: Multimodal Late Fusion (50 Epochs) ===
 HAR_MMAct: Multimodal Human Activity Recognition Training Pipeline
 Mode            : multimodal
 Video Backbone  : mobilenet_v2
 Sensor Backbone : inception_time
 Split Strategy  : subject
 Epochs          : 50 | Batch Size: 8 | LR: 0.0001
 Random Seed     : 42

[Step 1] Indexing dataset at: /kaggle/working/data/genuine_mmact
Indexed samples: 5431
Usable samples: 5427
Excluded samples: 4
  - Total Videos Scanned   : 17820
  - Total Sensors Scanned  : 35685
  - Paired Samples Created : 5431
  - Unknown Labels Filtered: 36388

[Step 2] Partitioning into subject-independent splits...
 SPLIT AUDIT & INTEGRITY REPORT
Strategy                : subject
Official Split Protocol : False
Custom Test Partition   : False
Train Source File       : N/A (Rule-based)
Val Source File         : N/A (Rule-based)
Train Samples / Records : 3795 samples (595 recordings)
Val Samples / Records   : 780 samples (124 recordings)
Test Samples / Records  : 

In [89]:
from pathlib import Path

train_script = Path("scripts/train.py")
lines = train_script.read_text().splitlines()

matches = [
    (i + 1, line.strip())
    for i, line in enumerate(lines)
    if "dropout" in line.lower()
]

print("Dropout references:")
for line_no, line in matches:
    print(f"{line_no}: {line}")

if not matches:
    print("No dropout argument or reference found.")

Dropout references:
65: parser.add_argument("--dropout", type=float, default=0.35, help="Dropout probability")
405: dropout=args.dropout,
804: "dropout": args.dropout,


In [91]:
from pathlib import Path

paths = [
    Path("/kaggle/working/data/genuine_mmact"),
    Path("/kaggle/working/data/genuine_mmact/trimmed"),
    Path("/kaggle/working/HAR-MMAct/data"),
    Path("/kaggle/working/HAR-MMAct/data/genuine_mmact"),
]

for path in paths:
    print(f"{path} -> exists: {path.exists()}")
    if path.exists() and path.is_dir():
        print("  Contents:", [p.name for p in list(path.iterdir())[:8]])

/kaggle/working/data/genuine_mmact -> exists: True
  Contents: ['manifests', 'dataset_manifest.json', 'trimmed']
/kaggle/working/data/genuine_mmact/trimmed -> exists: True
  Contents: ['sensor', 'video', 'splits', 'sensors']
/kaggle/working/HAR-MMAct/data -> exists: False
/kaggle/working/HAR-MMAct/data/genuine_mmact -> exists: False


In [92]:
from pathlib import Path

lines = Path("scripts/train.py").read_text().splitlines()

for i, line in enumerate(lines):
    if "dataset_root" in line.lower() or "data_root" in line.lower():
        print(f"{i+1}: {line.strip()}")

60: parser.add_argument("--data_root", type=str, default="data/genuine_mmact", help="Root directory of dataset")
216: data_path = Path(args.data_root)
274: indexer = MMActIndexer(dataset_root=data_path)
314: dataset_root=data_path,
317: splitter = DatasetSplitter(split_cfg, dataset_root=data_path)
771: "dataset_root": str(data_path),


In [93]:
!python scripts/train.py \
    --mode sensor_only \
    --sensor_backbone tcn \
    --epochs 50 \
    --min_epochs 50 \
    --batch_size 8 \
    --lr 0.0003 \
    --weight_decay 0.0001 \
    --dropout 0.20 \
    --split_strategy subject \
    --data_root /kaggle/working/data/genuine_mmact/trimmed \
    --output_dir outputs/tcn_sensor_only \
    --checkpoint_name best_model_tcn_sensor.pth \
    --cache_dir /kaggle/working/har_mmact_cache_20261107_154134 \
    --benchmark_warmup

 HAR_MMAct: Multimodal Human Activity Recognition Training Pipeline
 Mode            : sensor_only
 Video Backbone  : cnn
 Sensor Backbone : tcn
 Split Strategy  : subject
 Epochs          : 50 | Batch Size: 8 | LR: 0.0003
 Random Seed     : 42

[Step 1] Indexing dataset at: /kaggle/working/data/genuine_mmact/trimmed
Indexed samples: 5431
Usable samples: 5427
Excluded samples: 4
  - Total Videos Scanned   : 17820
  - Total Sensors Scanned  : 35685
  - Paired Samples Created : 5431
  - Unknown Labels Filtered: 36388

[Step 2] Partitioning into subject-independent splits...
 SPLIT AUDIT & INTEGRITY REPORT
Strategy                : subject
Official Split Protocol : False
Custom Test Partition   : False
Train Source File       : N/A (Rule-based)
Val Source File         : N/A (Rule-based)
Train Samples / Records : 3795 samples (595 recordings)
Val Samples / Records   : 780 samples (124 recordings)
Test Samples / Records  : 852 samples (132 recordings)
---------------------------------------

In [94]:
import shutil

print(
    "Free disk:",
    round(shutil.disk_usage("/kaggle/working").free / 1024**3, 2),
    "GB"
)

Free disk: 8.81 GB


In [ ]:
# Section 9A: Accuracy Upgrade - TCN Sensor-Only (50 Epochs)
# New architecture candidate; baseline InceptionTime experiment above remains unchanged.
out_tcn_sensor = 'outputs/tcn_sensor_only'
Path(out_tcn_sensor).mkdir(parents=True, exist_ok=True)

print(f'=== Accuracy Upgrade A: TCN Sensor-Only ({EPOCHS} Epochs) ===')
!python scripts/train.py \
    --data_root /kaggle/working/data/genuine_mmact \
    --mode sensor_only \
    --sensor_backbone tcn \
    --epochs {EPOCHS} \
    --min_epochs 50 \
    --batch_size 8 \
    --lr 0.0003 \
    --weight_decay 0.0001 \
    --dropout 0.20 \
    --split_strategy subject \
    --checkpoint_name best_model_tcn_sensor.pth \
    --output_dir {out_tcn_sensor} \
    --cache_dir {CACHE_ROOT} \
    --benchmark_warmup



In [ ]:
!python scripts/train.py \
    --mode multimodal \
    --video_backbone mobilenet_v2 \
    --sensor_backbone tcn \
    --fusion_strategy gated \
    --epochs 50 \
    --min_epochs 50 \
    --batch_size 8 \
    --lr 0.0001 \
    --weight_decay 0.0005 \
    --dropout 0.30 \
    --split_strategy subject \
    --data_root /kaggle/working/data/genuine_mmact/trimmed \
    --output_dir outputs/tcn_gated_fusion \
    --checkpoint_name best_model_tcn_gated_fusion.pth \
    --cache_dir /kaggle/working/har_mmact_cache_20261107_154134 \
    --benchmark_warmup

 HAR_MMAct: Multimodal Human Activity Recognition Training Pipeline
 Mode            : multimodal
 Video Backbone  : mobilenet_v2
 Sensor Backbone : tcn
 Split Strategy  : subject
 Epochs          : 50 | Batch Size: 8 | LR: 0.0001
 Random Seed     : 42

[Step 1] Indexing dataset at: /kaggle/working/data/genuine_mmact/trimmed
Indexed samples: 5431
Usable samples: 5427
Excluded samples: 4
  - Total Videos Scanned   : 17820
  - Total Sensors Scanned  : 35685
  - Paired Samples Created : 5431
  - Unknown Labels Filtered: 36388

[Step 2] Partitioning into subject-independent splits...
 SPLIT AUDIT & INTEGRITY REPORT
Strategy                : subject
Official Split Protocol : False
Custom Test Partition   : False
Train Source File       : N/A (Rule-based)
Val Source File         : N/A (Rule-based)
Train Samples / Records : 3795 samples (595 recordings)
Val Samples / Records   : 780 samples (124 recordings)
Test Samples / Records  : 852 samples (132 recordings)
-------------------------------

In [96]:
from pathlib import Path
import torch

ckpt_dir = Path("outputs/tcn_gated_fusion/checkpoints")

print("Checkpoint directory exists:", ckpt_dir.exists())

if ckpt_dir.exists():
    for path in sorted(ckpt_dir.glob("*.pth")):
        print(f"\nFile: {path.name}")
        print(f"Size: {path.stat().st_size / 1024**2:.2f} MB")

        try:
            ckpt = torch.load(path, map_location="cpu", weights_only=False)
            if isinstance(ckpt, dict):
                print("Epoch:", ckpt.get("epoch", "Not found"))
                print("Has model state:", "model_state_dict" in ckpt)
                print("Has optimizer state:", "optimizer_state_dict" in ckpt)
                print("Has scheduler state:", "scheduler_state_dict" in ckpt)
                print("Best validation F1:", ckpt.get("val_f1", "Not found"))
        except Exception as e:
            print("Could not inspect:", str(e))

Checkpoint directory exists: True

File: best_model.pth
Size: 32.95 MB
Epoch: 31
Has model state: True
Has optimizer state: True
Has scheduler state: True
Best validation F1: 0.7643156391656183

File: best_model_tcn_gated_fusion.pth
Size: 32.96 MB
Epoch: 31
Has model state: True
Has optimizer state: True
Has scheduler state: True
Best validation F1: 0.7643156391656183

File: latest_model.pth
Size: 32.95 MB
Epoch: 50
Has model state: True
Has optimizer state: True
Has scheduler state: True
Best validation F1: 0.7405434800972401

File: latest_model_mobilenet_v2.pth
Size: 32.96 MB
Epoch: 50
Has model state: True
Has optimizer state: True
Has scheduler state: True
Best validation F1: 0.7405434800972401


In [97]:
from pathlib import Path
from datetime import datetime
import json

out = Path("outputs/tcn_gated_fusion")

print("OUTPUT DIRECTORY:", out.resolve())

for p in sorted(out.rglob("*")):
    if p.is_file():
        modified = datetime.fromtimestamp(p.stat().st_mtime)
        print(f"{p.relative_to(out)} | {p.stat().st_size / 1024**2:.2f} MB | {modified}")

for name in ["history.json", "training_history.json", "manifest.json"]:
    p = out / name
    if p.exists():
        print(f"\n--- {name} ---")
        data = json.loads(p.read_text())
        print(str(data)[:3000])

OUTPUT DIRECTORY: /kaggle/working/HAR-MMAct/outputs/tcn_gated_fusion
checkpoints/best_model.pth | 32.95 MB | 2026-11-08 00:05:41.827890
checkpoints/best_model_tcn_gated_fusion.pth | 32.96 MB | 2026-11-08 00:05:41.720881
checkpoints/latest_model.pth | 32.95 MB | 2026-11-08 00:25:41.788286
checkpoints/latest_model_mobilenet_v2.pth | 32.96 MB | 2026-11-08 00:25:41.680277
experiment_manifest.json | 0.00 MB | 2026-11-08 00:25:50.852549
experiment_manifest_mobilenet_v2.json | 0.00 MB | 2026-11-08 00:25:50.852347
history.json | 0.01 MB | 2026-11-08 00:25:41.797496
history_mobilenet_v2.csv | 0.00 MB | 2026-11-08 00:25:50.845051
history_mobilenet_v2.json | 0.01 MB | 2026-11-08 00:25:50.848051
plots/generalization_gap.png | 0.09 MB | 2026-11-08 00:25:46.410676
plots/generalization_gap_mobilenet_v2.png | 0.09 MB | 2026-11-08 00:25:46.106651
plots/learning_rate_schedule.png | 0.09 MB | 2026-11-08 00:25:45.770622
plots/learning_rate_schedule_mobilenet_v2.png | 0.09 MB | 2026-11-08 00:25:45.254579
p

In [98]:
import json
from pathlib import Path

out = Path("outputs/tcn_gated_fusion")

for name in [
    "training_summary.json",
    "training_summary_mobilenet_v2.json",
    "experiment_manifest.json",
    "runtime_report.json",
]:
    path = out / name
    print(f"\n--- {name} ---")

    if path.exists():
        data = json.loads(path.read_text())
        print(json.dumps(data, indent=2)[:2500])
    else:
        print("Not found")


--- training_summary.json ---
{
  "dataset_root": "/kaggle/working/data/genuine_mmact/trimmed",
  "is_synthetic": false,
  "mode": "multimodal",
  "video_backbone": "mobilenet_v2",
  "sensor_backbone": "tcn",
  "split_strategy": "subject",
  "epochs": 50,
  "runtime_seconds": 3505.82,
  "best_checkpoint_mb": 32.96,
  "best_metrics": {
    "accuracy": 0.7769230769230769,
    "macro_f1": 0.7643156391656183,
    "macro_precision": 0.8260875917541398,
    "macro_recall": 0.7710526315789472,
    "per_class": {
      "crouching": {
        "precision": 1.0,
        "recall": 1.0,
        "f1": 1.0
      },
      "jumping": {
        "precision": 0.5714285714285714,
        "recall": 0.2631578947368421,
        "f1": 0.36036036036036034
      },
      "kicking": {
        "precision": 0.6333333333333333,
        "recall": 0.25,
        "f1": 0.3584905660377358
      },
      "pointing": {
        "precision": 0.9137931034482759,
        "recall": 0.6973684210526315,
        "f1": 0.791044776

In [ ]:
# Section 9B: Accuracy Upgrade - TCN + Gated Multimodal Fusion (50 Epochs)
# Learns sample-dependent weights for video and sensor embeddings.
out_gated = 'outputs/tcn_gated_fusion'
Path(out_gated).mkdir(parents=True, exist_ok=True)

print(f'=== Accuracy Upgrade B: TCN + Gated Fusion ({EPOCHS} Epochs) ===')
!python scripts/train.py \
    --data_root /kaggle/working/data/genuine_mmact \
    --mode multimodal \
    --video_backbone mobilenet_v2 \
    --sensor_backbone tcn \
    --fusion_strategy gated \
    --epochs {EPOCHS} \
    --min_epochs 50 \
    --batch_size 8 \
    --lr 0.0001 \
    --weight_decay 0.0005 \
    --dropout 0.30 \
    --split_strategy subject \
    --checkpoint_name best_model_tcn_gated_fusion.pth \
    --output_dir {out_gated} \
    --cache_dir {CACHE_ROOT} \
    --benchmark_warmup



In [ ]:
from pathlib import Path

p = Path("outputs/multimodal_fusion/checkpoints/best_model_multimodal.pth")

print("Checkpoint exists:", p.exists())
print(
    "Checkpoint size (MB):",
    round(p.stat().st_size / (1024 * 1024), 2) if p.exists() else "Missing"
)

print("\nExperiment artifacts:")
for f in sorted(Path("outputs/multimodal_fusion").rglob("*")):
    if f.is_file():
        print(f, f.stat().st_size)

In [ ]:
print("EPOCHS:", EPOCHS)
print("Dataset exists:", Path("/kaggle/working/data/genuine_mmact").exists())
print("Training script exists:", Path("scripts/train.py").exists())

### MobileNetV2 resume-safety patch
Apply the optimizer-group reconstruction before training starts. This prevents a resumed staged-fine-tuning checkpoint from failing when the backbone has already been unfrozen.


In [99]:
from pathlib import Path

script_path = Path("scripts/train.py")
source = script_path.read_text()

old = '''    if args.resume and Path(args.resume).exists():
        print(f"\\n[Resume] Loading checkpoint from: {args.resume}")
        resume_ckpt = torch.load(args.resume, map_location=device)
        model.load_state_dict(resume_ckpt["model_state_dict"])
        if "optimizer_state_dict" in resume_ckpt:
            optimizer.load_state_dict(resume_ckpt["optimizer_state_dict"])
        if "scheduler_state_dict" in resume_ckpt and resume_ckpt["scheduler_state_dict"] and scheduler:
            scheduler.load_state_dict(resume_ckpt["scheduler_state_dict"])
'''

new = '''    if args.resume and Path(args.resume).exists():
        print(f"\\n[Resume] Loading checkpoint from: {args.resume}")
        resume_ckpt = torch.load(args.resume, map_location=device)
        model.load_state_dict(resume_ckpt["model_state_dict"])

        # Reconstruct the staged fine-tuning optimizer before restoring
        # a checkpoint saved after the backbone was unfrozen.
        resume_epoch = int(resume_ckpt.get("epoch", 0))

        if args.staged_finetune and resume_epoch >= args.unfreeze_epoch:
            print("[Resume] Reconstructing the two-group fine-tuning optimizer...")

            for p in model.video_branch.backbone.parameters():
                p.requires_grad = True

            backbone_params = list(model.video_branch.backbone.parameters())
            backbone_ids = set(id(p) for p in backbone_params)
            head_params = [
                p for p in model.parameters()
                if id(p) not in backbone_ids and p.requires_grad
            ]

            optimizer = torch.optim.AdamW([
                {
                    "params": backbone_params,
                    "lr": args.unfreeze_lr,
                    "weight_decay": args.weight_decay,
                },
                {
                    "params": head_params,
                    "lr": args.lr,
                    "weight_decay": args.weight_decay,
                },
            ])

            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                optimizer,
                T_max=max(1, args.epochs - args.unfreeze_epoch + 1),
                eta_min=1e-6,
            )

            print(
                f"[Resume] Optimizer groups reconstructed: "
                f"{len(optimizer.param_groups)} "
                f"(backbone={len(backbone_params)}, heads={len(head_params)})"
            )

        if "optimizer_state_dict" in resume_ckpt:
            optimizer.load_state_dict(resume_ckpt["optimizer_state_dict"])
            print("[Resume] Optimizer state restored successfully.")

        if "scheduler_state_dict" in resume_ckpt and resume_ckpt["scheduler_state_dict"] and scheduler:
            scheduler.load_state_dict(resume_ckpt["scheduler_state_dict"])
            print("[Resume] Scheduler state restored successfully.")
'''

if source.count(old) != 1:
    raise RuntimeError(
        f"Patch aborted: expected exactly one matching resume block, "
        f"found {source.count(old)}. No changes made."
    )

backup_path = Path("scripts/train.py.before_resume_fix")
backup_path.write_text(source)

script_path.write_text(source.replace(old, new, 1))

print("PATCH APPLIED")
print("Backup:", backup_path)
print("Updated:", script_path)

PATCH APPLIED
Backup: scripts/train.py.before_resume_fix
Updated: scripts/train.py


In [100]:
import py_compile
from pathlib import Path

script = Path("scripts/train.py")

py_compile.compile(str(script), doraise=True)

source = script.read_text()
checks = {
    "Resume optimizer reconstruction": "Reconstructing the two-group fine-tuning optimizer" in source,
    "Backbone parameter groups": "backbone_params = list(model.video_branch.backbone.parameters())" in source,
    "Optimizer state restoration": "Optimizer state restored successfully." in source,
    "Scheduler state restoration": "Scheduler state restored successfully." in source,
}

print("SYNTAX: PASS")
for name, passed in checks.items():
    print(f"{name}: {'PASS' if passed else 'FAIL'}")

SYNTAX: PASS
Resume optimizer reconstruction: PASS
Backbone parameter groups: PASS
Optimizer state restoration: PASS
Scheduler state restoration: PASS


## Section 10: Experiment 3 - MobileNetV2 Video-Only Staged Fine-Tuning (50 Epochs)
Phase 1 (Epochs 1-15): Pretrained MobileNetV2 backbone frozen, training classification head at $10^{-4}$.
Phase 2 (Epochs 16-50): Backbone unfrozen with differential learning rate ($10^{-5}$) for gentle fine-tuning.


In [101]:
from pathlib import Path

out_mob = "outputs/mobilenet_v2"
Path(out_mob).mkdir(parents=True, exist_ok=True)

print(f"=== Experiment 3: MobileNetV2 Video-Only Staged Fine-Tuning ({EPOCHS} Epochs) ===")

!python scripts/train.py \
    --data_root /kaggle/working/data/genuine_mmact/trimmed \
    --video_backbone mobilenet_v2 \
    --sensor_backbone inception_time \
    --mode video_only \
    --epochs {EPOCHS} \
    --min_epochs 50 \
    --batch_size 8 \
    --lr 0.0001 \
    --weight_decay 0.0005 \
    --dropout 0.35 \
    --staged_finetune \
    --unfreeze_epoch 15 \
    --unfreeze_lr 0.00001 \
    --split_strategy subject \
    --early_stopping_patience 15 \
    --checkpoint_name best_model_mobilenet_v2.pth \
    --output_dir {out_mob} \
    --cache_dir {CACHE_ROOT} \
    --benchmark_warmup \
    --auto_resume \
    --skip_if_completed

=== Experiment 3: MobileNetV2 Video-Only Staged Fine-Tuning (50 Epochs) ===
 HAR_MMAct: Multimodal Human Activity Recognition Training Pipeline
 Mode            : video_only
 Video Backbone  : mobilenet_v2
 Sensor Backbone : inception_time
 Split Strategy  : subject
 Epochs          : 50 | Batch Size: 8 | LR: 0.0001
 Random Seed     : 42

[Step 1] Indexing dataset at: /kaggle/working/data/genuine_mmact/trimmed
Indexed samples: 5431
Usable samples: 5427
Excluded samples: 4
  - Total Videos Scanned   : 17820
  - Total Sensors Scanned  : 35685
  - Paired Samples Created : 5431
  - Unknown Labels Filtered: 36388

[Step 2] Partitioning into subject-independent splits...
 SPLIT AUDIT & INTEGRITY REPORT
Strategy                : subject
Official Split Protocol : False
Custom Test Partition   : False
Train Source File       : N/A (Rule-based)
Val Source File         : N/A (Rule-based)
Train Samples / Records : 3795 samples (595 recordings)
Val Samples / Records   : 780 samples (124 recordings)

## Section 11: Experiment 4 - Improved Regularized CNN Video Baseline (50 Epochs)
Trains the improved custom video CNN baseline for 50 epochs using Spatial Dropout2d (0.15, 0.20), BatchNorm2d, Weight Decay ($10^{-3}$), Cosine Annealing, Automatic Mixed Precision, and auto-resume protection.


In [110]:
# Section 11: Experiment 4 - Improved Regularized CNN Video Baseline
out_imp_cnn = 'outputs/improved_cnn'
Path(out_imp_cnn).mkdir(parents=True, exist_ok=True)

print(f'=== Starting Experiment 4: Improved Regularized CNN ({EPOCHS} Epochs) ===')

!python scripts/train.py \
    --data_root /kaggle/working/data/genuine_mmact/trimmed \
    --video_backbone custom_cnn \
    --sensor_backbone inception_time \
    --mode video_only \
    --epochs {EPOCHS} \
    --min_epochs 50 \
    --batch_size 8 \
    --lr 0.0001 \
    --weight_decay 0.001 \
    --dropout 0.35 \
    --split_strategy subject \
    --early_stopping_patience 15 \
    --checkpoint_name best_model_cnn.pth \
    --output_dir {out_imp_cnn} \
    --cache_dir {CACHE_ROOT} \
    --benchmark_warmup \
    --auto_resume \
    --skip_if_completed

=== Starting Experiment 4: Improved Regularized CNN (50 Epochs) ===
 HAR_MMAct: Multimodal Human Activity Recognition Training Pipeline
 Mode            : video_only
 Video Backbone  : custom_cnn
 Sensor Backbone : inception_time
 Split Strategy  : subject
 Epochs          : 50 | Batch Size: 8 | LR: 0.0001
 Random Seed     : 42

[Step 1] Indexing dataset at: /kaggle/working/data/genuine_mmact/trimmed
Indexed samples: 5431
Usable samples: 5427
Excluded samples: 4
  - Total Videos Scanned   : 17820
  - Total Sensors Scanned  : 35685
  - Paired Samples Created : 5431
  - Unknown Labels Filtered: 36388

[Step 2] Partitioning into subject-independent splits...
 SPLIT AUDIT & INTEGRITY REPORT
Strategy                : subject
Official Split Protocol : False
Custom Test Partition   : False
Train Source File       : N/A (Rule-based)
Val Source File         : N/A (Rule-based)
Train Samples / Records : 3795 samples (595 recordings)
Val Samples / Records   : 780 samples (124 recordings)
Test Samp

In [112]:
from pathlib import Path

DATA_ROOT = Path('/kaggle/working/data/genuine_mmact/trimmed')
TUNE_SCRIPT = Path('scripts/tune.py')

print("Dataset exists:", DATA_ROOT.exists())
print("Dataset root:", DATA_ROOT)
print("Required folders:")
for name in ["video", "sensors", "splits"]:
    print(f"  {name}: {(DATA_ROOT / name).exists()}")

print("\nTuning script exists:", TUNE_SCRIPT.exists())

assert DATA_ROOT.exists(), "Correct MMAct dataset root not found"
assert all((DATA_ROOT / x).exists() for x in ["video", "sensors", "splits"])
assert TUNE_SCRIPT.exists(), "scripts/tune.py not found"

print("\nPREFLIGHT PASSED")

Dataset exists: True
Dataset root: /kaggle/working/data/genuine_mmact/trimmed
Required folders:
  video: True
  sensors: True
  splits: True

Tuning script exists: True

PREFLIGHT PASSED


In [113]:
from pathlib import Path
import sys

p = Path("scripts/tune.py")
source = p.read_text()

old = """    splitter = DatasetSplitter(SplitConfig(split_strategy=args.split_strategy))
    train_samples, val_samples, test_samples = splitter.split(report.paired_samples)
"""

new = """    # Remove samples with missing or empty required sensor files.
    required_modalities = ("acc_phone", "acc_watch", "gyro")
    original_count = len(report.paired_samples)
    valid_samples = []
    rejected_samples = []

    for sample in report.paired_samples:
        valid = True

        for modality in required_modalities:
            sensor_path = sample.sensor_paths.get(modality)

            if not sensor_path:
                valid = False
                break

            sensor_file = Path(sensor_path)
            if not sensor_file.is_file() or sensor_file.stat().st_size == 0:
                valid = False
                break

        if valid:
            valid_samples.append(sample)
        else:
            rejected_samples.append(sample.sample_id)

    print(f"  - Samples before sensor validation: {original_count}")
    print(f"  - Samples rejected (missing/empty sensors): {len(rejected_samples)}")
    print(f"  - Samples retained: {len(valid_samples)}")

    if not valid_samples:
        print("[ERROR] No valid samples remain after sensor validation.")
        sys.exit(1)

    splitter = DatasetSplitter(SplitConfig(split_strategy=args.split_strategy))
    train_samples, val_samples, test_samples = splitter.split(valid_samples)
"""

if old not in source:
    if "splitter.split(valid_samples)" in source:
        print("Patch already exists. No changes needed.")
    else:
        raise RuntimeError("Expected split code not found. No changes made.")
else:
    backup = Path("scripts/tune.py.before_sensor_filter_fix")
    if not backup.exists():
        backup.write_text(source)

    p.write_text(source.replace(old, new, 1))
    print("PATCH APPLIED")
    print("Backup:", backup)

updated = p.read_text()
compile(updated, str(p), "exec")

assert "required_modalities = (\"acc_phone\", \"acc_watch\", \"gyro\")" in updated
assert "splitter.split(valid_samples)" in updated

print("Python syntax validation: PASSED")

PATCH APPLIED
Backup: scripts/tune.py.before_sensor_filter_fix
Python syntax validation: PASSED


## Section 12: Experiment 5 - Hyperparameter Tuning (Train/Val Only) & Tuned CNN (50 Epochs)
Systematically explores learning rate, weight decay, and dropout across trials strictly on train/val splits. The held-out test split is NEVER touched. The best configuration is then trained for 50 epochs.


In [114]:
# Section 12A: Experiment 5 - Hyperparameter Tuning

print("=== Starting Experiment 5: Hyperparameter Tuning ===")

!python scripts/tune.py \
    --data_root /kaggle/working/data/genuine_mmact/trimmed \
    --split_strategy subject \
    --epochs_per_trial 10 \
    --batch_size 8 \
    --output_dir outputs/tuning \
    --plots_dir outputs/plots \
    --cache_dir {CACHE_ROOT} \
    --skip_if_completed

=== Starting Experiment 5: Hyperparameter Tuning ===
 HAR_MMAct: Hyperparameter Search Pipeline
 Split Strategy    : subject (Held-out Test Split ISOLATED)
 Trials Count      : 4
 Epochs per Trial  : 10
 Optimization Goal : Maximizing Validation Accuracy / Macro F1
  - Samples before sensor validation: 5431
  - Samples rejected (missing/empty sensors): 4
  - Samples retained: 5427

Dataset Partitions:
  - Train Partition (Used for fitting)     : 3795 samples
  - Val Partition   (Used for model selection): 780 samples
  - Test Partition  (STRICTLY UNTOUCHED)   : 852 samples
  - Tuning AMP Acceleration: ENABLED (float16)

--- Running Trial 1/4: Baseline Regularized ---
    Params: lr=0.0001, weight_decay=0.001, dropout=0.35
    Result: Best Val Acc=15.90%, Val Macro F1=0.0274 (349.4s)

--- Running Trial 2/4: Higher Learning Rate ---
    Params: lr=0.0003, weight_decay=0.001, dropout=0.35
    Result: Best Val Acc=15.51%, Val Macro F1=0.0815 (344.5s)

--- Running Trial 3/4: Heavy Regulariz

In [117]:
import json
from pathlib import Path

p = Path("outputs/tuning/best_hyperparameters.json")

assert p.exists(), "Best hyperparameters file missing"

with open(p) as f:
    best_cfg = json.load(f)

print("Selected configuration from tuning script:")
print(best_cfg)

print("\nAvailable tuning artifacts:")
for name in [
    "tuning_history.json",
    "tuning_history.csv",
    "trial_epoch_histories.csv",
    "best_hyperparameters.json"
]:
    f = p.parent / name
    print(name, "EXISTS" if f.exists() else "MISSING")

Selected configuration from tuning script:
{'trial_id': 1, 'name': 'Baseline Regularized', 'lr': 0.0001, 'weight_decay': 0.001, 'dropout': 0.35, 'val_accuracy': 15.9, 'val_macro_f1': 0.0274, 'duration_sec': 349.4, 'epoch_history': [{'epoch': 1, 'train_loss': 2.3858, 'train_acc': 11.44, 'val_loss': 2.2884, 'val_acc': 15.9, 'val_f1': 0.0274}, {'epoch': 2, 'train_loss': 2.3491, 'train_acc': 12.09, 'val_loss': 2.2814, 'val_acc': 15.9, 'val_f1': 0.0274}, {'epoch': 3, 'train_loss': 2.338, 'train_acc': 12.73, 'val_loss': 2.28, 'val_acc': 15.9, 'val_f1': 0.0274}, {'epoch': 4, 'train_loss': 2.321, 'train_acc': 12.96, 'val_loss': 2.2796, 'val_acc': 15.9, 'val_f1': 0.0555}, {'epoch': 5, 'train_loss': 2.2942, 'train_acc': 13.6, 'val_loss': 2.284, 'val_acc': 10.0, 'val_f1': 0.021}, {'epoch': 6, 'train_loss': 2.2827, 'train_acc': 14.44, 'val_loss': 2.2861, 'val_acc': 9.74, 'val_f1': 0.0193}, {'epoch': 7, 'train_loss': 2.2602, 'train_acc': 15.1, 'val_loss': 2.2777, 'val_acc': 14.23, 'val_f1': 0.0555}

In [118]:
from pathlib import Path

out_tuned = "outputs/tuned_cnn"
Path(out_tuned).mkdir(parents=True, exist_ok=True)

# Trial 2: Best validation Macro F1
BEST_LR = 0.0003
BEST_WEIGHT_DECAY = 0.001
BEST_DROPOUT = 0.35

print("=== Experiment 5: Tuned CNN Training ===")
print("Selected configuration: Trial 2")
print(f"Validation Macro F1: 0.0815")
print(f"Learning rate: {BEST_LR}")
print(f"Weight decay: {BEST_WEIGHT_DECAY}")
print(f"Dropout: {BEST_DROPOUT}")

!python scripts/train.py \
    --data_root /kaggle/working/data/genuine_mmact/trimmed \
    --video_backbone custom_cnn \
    --sensor_backbone inception_time \
    --mode video_only \
    --epochs 50 \
    --min_epochs 50 \
    --batch_size 8 \
    --lr {BEST_LR} \
    --weight_decay {BEST_WEIGHT_DECAY} \
    --dropout {BEST_DROPOUT} \
    --split_strategy subject \
    --early_stopping_patience 15 \
    --checkpoint_name best_model_tuned_cnn.pth \
    --output_dir {out_tuned} \
    --cache_dir {CACHE_ROOT} \
    --benchmark_warmup \
    --auto_resume \
    --skip_if_completed

=== Experiment 5: Tuned CNN Training ===
Selected configuration: Trial 2
Validation Macro F1: 0.0815
Learning rate: 0.0003
Weight decay: 0.001
Dropout: 0.35
 HAR_MMAct: Multimodal Human Activity Recognition Training Pipeline
 Mode            : video_only
 Video Backbone  : custom_cnn
 Sensor Backbone : inception_time
 Split Strategy  : subject
 Epochs          : 50 | Batch Size: 8 | LR: 0.0003
 Random Seed     : 42

[Step 1] Indexing dataset at: /kaggle/working/data/genuine_mmact/trimmed
Indexed samples: 5431
Usable samples: 5427
Excluded samples: 4
  - Total Videos Scanned   : 17820
  - Total Sensors Scanned  : 35685
  - Paired Samples Created : 5431
  - Unknown Labels Filtered: 36388

[Step 2] Partitioning into subject-independent splits...
 SPLIT AUDIT & INTEGRITY REPORT
Strategy                : subject
Official Split Protocol : False
Custom Test Partition   : False
Train Source File       : N/A (Rule-based)
Val Source File         : N/A (Rule-based)
Train Samples / Records : 3795 

## Section 13: Experiment 6 - Original Custom CNN Baseline (50 Epochs)
Trains the original unregularized custom CNN baseline representing the initial student model (without heavy spatial dropout or aggressive weight decay) for 50 epochs.


In [119]:
from pathlib import Path

print("=== Checking Original CNN Backbone ===")

!grep -Rni "original_cnn" src scripts

print("\n=== Checking Custom CNN Backbone ===")

!grep -Rni "custom_cnn" src scripts

=== Checking Original CNN Backbone ===
src/models/fusion_model.py:70:        elif video_backbone in ["original_cnn", "lightweight"]:
grep: src/models/__pycache__/fusion_model.cpython-313.pyc: binary file matches
scripts/train.py.indentation_broken.bak:85:    parser.add_argument("--video_backbone", type=str, default="cnn", choices=["cnn", "custom_cnn", "original_cnn", "improved_cnn", "mobilenet_v2", "resnet18", "lightweight"], help="Video backbone")
scripts/train.py:85:    parser.add_argument("--video_backbone", type=str, default="cnn", choices=["cnn", "custom_cnn", "original_cnn", "improved_cnn", "mobilenet_v2", "resnet18", "lightweight"], help="Video backbone")
scripts/train.py.before_indent_fix.bak:85:    parser.add_argument("--video_backbone", type=str, default="cnn", choices=["cnn", "custom_cnn", "original_cnn", "improved_cnn", "mobilenet_v2", "resnet18", "lightweight"], help="Video backbone")
scripts/train.py.before_resume_fix:85:    parser.add_argument("--video_backbone", type=st

In [ ]:
# Section 13: Experiment 6 - Original Custom CNN Baseline (50 Epochs)

out_orig_cnn = "outputs/original_cnn"
Path(out_orig_cnn).mkdir(parents=True, exist_ok=True)

print(f"=== Starting Experiment 6: Original Custom CNN ({EPOCHS} Epochs) ===")

!python scripts/train.py \
    --data_root /kaggle/working/data/genuine_mmact/trimmed \
    --video_backbone original_cnn \
    --sensor_backbone inception_time \
    --mode video_only \
    --epochs 50 \
    --min_epochs 50 \
    --batch_size 8 \
    --lr 0.0001 \
    --weight_decay 0.0001 \
    --dropout 0.20 \
    --split_strategy subject \
    --early_stopping_patience 15 \
    --checkpoint_name best_model_original_cnn.pth \
    --output_dir {out_orig_cnn} \
    --cache_dir {CACHE_ROOT} \
    --benchmark_warmup \
    --auto_resume \
    --skip_if_completed

=== Starting Experiment 6: Original Custom CNN (50 Epochs) ===
 HAR_MMAct: Multimodal Human Activity Recognition Training Pipeline
 Mode            : video_only
 Video Backbone  : original_cnn
 Sensor Backbone : inception_time
 Split Strategy  : subject
 Epochs          : 50 | Batch Size: 8 | LR: 0.0001
 Random Seed     : 42

[Step 1] Indexing dataset at: /kaggle/working/data/genuine_mmact/trimmed
Indexed samples: 5431
Usable samples: 5427
Excluded samples: 4
  - Total Videos Scanned   : 17820
  - Total Sensors Scanned  : 35685
  - Paired Samples Created : 5431
  - Unknown Labels Filtered: 36388

[Step 2] Partitioning into subject-independent splits...
 SPLIT AUDIT & INTEGRITY REPORT
Strategy                : subject
Official Split Protocol : False
Custom Test Partition   : False
Train Source File       : N/A (Rule-based)
Val Source File         : N/A (Rule-based)
Train Samples / Records : 3795 samples (595 recordings)
Val Samples / Records   : 780 samples (124 recordings)
Test Samples

In [20]:
import os
from pathlib import Path

os.chdir("/kaggle/working/HAR-MMAct")

files = [
    "scripts/train.py",
]

terms = [
    "auto_resume",
    "skip_if_completed",
    "checkpoint_name",
    "torch.save",
    "resume",
    "load_checkpoint",
    "best_model",
    "output_dir",
]

for filename in files:
    path = Path(filename)

    print(f"\n========== {filename} ==========")

    lines = path.read_text().splitlines()

    for i, line in enumerate(lines):
        if any(term in line.lower() for term in terms):
            start = max(0, i - 3)
            end = min(len(lines), i + 5)

            print(f"\n--- Lines {start+1}-{end} ---")

            for j in range(start, end):
                print(f"{j+1}: {lines[j]}")


========== scripts/train.py ==========

--- Lines 91-98 ---
91:     parser.add_argument("--unfreeze_lr", type=float, default=1e-5, help="Learning rate for unfrozen video backbone")
92:     parser.add_argument("--early_stopping_patience", type=int, default=0, help="Epochs without improvement before early stopping (0=disabled)")
93:     parser.add_argument("--min_epochs", type=int, default=50, help="Minimum number of epochs to train before early stopping can trigger (default: 50)")
94:     parser.add_argument("--resume", type=str, default=None, help="Path to checkpoint to resume training from")
95:     parser.add_argument("--auto_resume", action="store_true", help="Automatically resume from latest checkpoint in output_dir if present")
96:     parser.add_argument("--skip_if_completed", action="store_true", help="Skip training if experiment has already completed successfully")
97:     parser.add_argument("--checkpoint_name", type=str, default=None, help="Custom filename for saved checkpoi

In [21]:
from pathlib import Path

p = Path("outputs/original_cnn")

print("Directory:", p.resolve())
print("Exists:", p.exists())

for f in p.rglob("*"):
    if f.is_file():
        print(f, f.stat().st_size, "bytes")

Directory: /kaggle/working/HAR-MMAct/outputs/original_cnn
Exists: True


In [22]:
from pathlib import Path

root = Path("/kaggle/working/HAR-MMAct/outputs/original_cnn")
ckpt = root / "checkpoints"

print("=== CHECKPOINT FILES ===")

for f in ckpt.glob("*"):
    if f.is_file():
        print(f.name, f"{f.stat().st_size / 1024 / 1024:.2f} MB")

print("\n=== HISTORY FILES ===")

for f in root.glob("*"):
    if f.is_file():
        print(f.name, f.stat().st_size, "bytes")

print("\n=== RESUME CANDIDATES ===")

for name in [
    "latest_model_original_cnn.pth",
    "latest_model.pth",
    "best_model_original_cnn.pth",
    "best_model.pth"
]:
    f = ckpt / name
    print(name, "FOUND" if f.exists() else "MISSING")

=== CHECKPOINT FILES ===

=== HISTORY FILES ===

=== RESUME CANDIDATES ===
latest_model_original_cnn.pth MISSING
latest_model.pth MISSING
best_model_original_cnn.pth MISSING
best_model.pth MISSING


In [23]:
import torch
from pathlib import Path

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

dataset = Path("/kaggle/working/data/genuine_mmact/trimmed")
print("Dataset exists:", dataset.exists())

repo = Path("/kaggle/working/HAR-MMAct")
print("Repository exists:", repo.exists())

CUDA available: True
GPU: Tesla T4
Dataset exists: True
Repository exists: True


In [24]:
import os
from pathlib import Path

os.chdir("/kaggle/working/HAR-MMAct")

out_orig_cnn = "outputs/original_cnn_recovery"
Path(out_orig_cnn).mkdir(parents=True, exist_ok=True)

!python scripts/train.py \
    --data_root /kaggle/working/data/genuine_mmact/trimmed \
    --video_backbone original_cnn \
    --sensor_backbone inception_time \
    --mode video_only \
    --epochs 50 \
    --min_epochs 50 \
    --batch_size 8 \
    --lr 0.0001 \
    --weight_decay 0.0001 \
    --dropout 0.20 \
    --split_strategy subject \
    --early_stopping_patience 15 \
    --checkpoint_name best_model_original_cnn.pth \
    --output_dir outputs/original_cnn_recovery \
    --cache_dir {CACHE_ROOT} \
    --benchmark_warmup \
    --auto_resume

 HAR_MMAct: Multimodal Human Activity Recognition Training Pipeline
 Mode            : video_only
 Video Backbone  : original_cnn
 Sensor Backbone : inception_time
 Split Strategy  : subject
 Epochs          : 50 | Batch Size: 8 | LR: 0.0001
 Random Seed     : 42

[Step 1] Indexing dataset at: /kaggle/working/data/genuine_mmact/trimmed
Indexed samples: 5431
Usable samples: 5427
Excluded samples: 4
  - Total Videos Scanned   : 17820
  - Total Sensors Scanned  : 35685
  - Paired Samples Created : 5431
  - Unknown Labels Filtered: 36388

[Step 2] Partitioning into subject-independent splits...
 SPLIT AUDIT & INTEGRITY REPORT
Strategy                : subject
Official Split Protocol : False
Custom Test Partition   : False
Train Source File       : N/A (Rule-based)
Val Source File         : N/A (Rule-based)
Train Samples / Records : 3795 samples (595 recordings)
Val Samples / Records   : 780 samples (124 recordings)
Test Samples / Records  : 852 samples (132 recordings)
--------------------

In [6]:
from pathlib import Path

for root in [
    Path("/kaggle/input/datasets/vikramche/trimmed-scene-video"),
    Path("/kaggle/input/datasets/vikramche/mmact-trimmed"),
]:
    print(f"\n{'='*65}\n{root}")
    if root.exists():
        for p in list(root.iterdir())[:20]:
            print(" ", p.name, "[DIR]" if p.is_dir() else "[FILE]")


/kaggle/input/datasets/vikramche/trimmed-scene-video
  trimmed [DIR]

/kaggle/input/datasets/vikramche/mmact-trimmed
  trimmed_sensor [DIR]
  trimmed_splits [DIR]
  trimmed_x-scene_subject16 [DIR]
  trimmed_x-scene_subject8 [DIR]


In [7]:
from pathlib import Path

paths = [
    Path("/kaggle/input/datasets/vikramche/trimmed-scene-video/trimmed"),
    Path("/kaggle/input/datasets/vikramche/mmact-trimmed/trimmed_sensor"),
    Path("/kaggle/input/datasets/vikramche/mmact-trimmed/trimmed_splits"),
]

for root in paths:
    print(f"\n{root}")
    if root.exists():
        for p in list(root.iterdir())[:12]:
            print("  ", p.name, "[DIR]" if p.is_dir() else "[FILE]")
    else:
        print("MISSING")


/kaggle/input/datasets/vikramche/trimmed-scene-video/trimmed
   cross_scene_video [DIR]

/kaggle/input/datasets/vikramche/mmact-trimmed/trimmed_sensor
   sensor [DIR]

/kaggle/input/datasets/vikramche/mmact-trimmed/trimmed_splits
   splits [DIR]


In [9]:
import os
from pathlib import Path

REPO_URL = "https://github.com/CHENNAMSETTYVIKRAM/HAR-MMAct.git"
REPO = Path("/kaggle/working/HAR-MMAct")

if not REPO.exists():
    !git clone --branch kaggle-training-backup {REPO_URL} {REPO}

os.chdir(REPO)

print("Repository:", os.getcwd())
print("Training script:", (REPO / "scripts/train.py").exists())
print("Config directory:", (REPO / "configs").exists())
print("Git branch:")
!git branch --show-current

Cloning into '/kaggle/working/HAR-MMAct'...
remote: Enumerating objects: 529, done.
remote: Counting objects: 100% (2/2), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 529 (delta 0), reused 0 (delta 0), pack-reused 527 (from 1)
Receiving objects: 100% (529/529), 209.32 MiB | 33.78 MiB/s, done.
Resolving deltas: 100% (181/181), done.
Updating files: 100% (390/390), done.
Repository: /kaggle/working/HAR-MMAct
Training script: True
Config directory: False
Git branch:
kaggle-training-backup


In [10]:
from pathlib import Path
import os

REPO = Path("/kaggle/working/HAR-MMAct")
os.chdir(REPO)

print("=== CHECKPOINT RECOVERY ===")
for p in REPO.glob("outputs/**/*.pth"):
    print(f"{p.relative_to(REPO)} | {p.stat().st_size / 1024**2:.2f} MB")

print("\n=== TRAINING OUTPUT DIRECTORIES ===")
out = REPO / "outputs"
if out.exists():
    for p in sorted(out.iterdir()):
        if p.is_dir():
            print(p.name)

print("\n=== DATA ROOT REFERENCES ===")
!grep -R -n "cross_scene_video\|trimmed_sensor\|data_root" src scripts | head -50

print("\n=== REQUIRED SCRIPT CHECK ===")
for f in ["scripts/train.py", "scripts/tune.py"]:
    print(f, (REPO / f).exists())

=== CHECKPOINT RECOVERY ===
outputs/tcn_gated_fusion/checkpoints/best_model.pth | 32.95 MB
outputs/tcn_gated_fusion/checkpoints/best_model_tcn_gated_fusion.pth | 32.96 MB
outputs/exp1_multimodal/checkpoints/best_model.pth | 32.94 MB
outputs/improved_cnn/checkpoints/best_model.pth | 2.84 MB
outputs/improved_cnn/checkpoints/best_model_cnn.pth | 2.84 MB
outputs/tcn_sensor_only/checkpoints/best_model.pth | 3.36 MB
outputs/tcn_sensor_only/checkpoints/best_model_tcn_sensor.pth | 3.36 MB
outputs/multimodal_fusion/checkpoints/best_model.pth | 32.94 MB
outputs/multimodal_fusion/checkpoints/best_model_multimodal.pth | 32.95 MB
outputs/exp2_sensor_only/checkpoints/best_model.pth | 3.73 MB
outputs/exp2_sensor_only/checkpoints/best_model_sensor.pth | 3.73 MB
outputs/mobilenet_v2/checkpoints/best_model.pth | 30.95 MB
outputs/mobilenet_v2/checkpoints/best_model_mobilenet_v2.pth | 30.96 MB

=== TRAINING OUTPUT DIRECTORIES ===
exp1_multimodal
exp2_sensor_only
improved_cnn
mobilenet_v2
multimodal_fusion

In [11]:
from pathlib import Path

repo = Path("/kaggle/working/HAR-MMAct")

for filename, start, end in [
    ("src/data/indexer.py", 250, 320),
    ("src/data/stager.py", 580, 625),
    ("scripts/train.py", 205, 245),
]:
    path = repo / filename
    print(f"\n{'='*70}\n{filename}")
    lines = path.read_text().splitlines()
    for i in range(start - 1, min(end, len(lines))):
        print(f"{i+1}: {lines[i]}")


src/data/indexer.py
250: 
251:         # 1. Discover all videos (safely traversing symlinks)
252:         video_map: Dict[Tuple[str, str, str, str, str], Path] = {}
253:         for vid_file in self.find_files_recursively(self.dataset_root, {".mp4"}):
254:             report.total_video_files_found += 1
255:             parsed = self.parse_video_path(vid_file)
256:             if not parsed:
257:                 continue
258:             subj, cam, scene, session, action = parsed
259: 
260:             if action not in self.target_classes:
261:                 report.unknown_label_files.append(str(vid_file))
262:                 continue
263: 
264:             key = (subj, cam, scene, session, action)
265:             if key in video_map:
266:                 report.duplicate_ids.append(f"video:{key}")
267:             video_map[key] = vid_file
268: 
269:         # 2. Discover relevant sensors
270:         # Key: (subject, scene, session, action) -> {sensor_type: path}
271:         se

In [5]:
from pathlib import Path

root = Path("/kaggle/input/datasets")

for p in root.iterdir():
    print(f"\nDataset: {p.name}")
    if p.is_dir():
        for child in list(p.iterdir())[:15]:
            print("  ", child.name, "[DIR]" if child.is_dir() else "[FILE]")


Dataset: vikramche
   trimmed-scene-video [DIR]
   mmact-trimmed [DIR]


In [12]:
from pathlib import Path
import os

BASE = Path("/kaggle/working/data/genuine_mmact/trimmed")

SOURCES = {
    "video/cross_scene": Path(
        "/kaggle/input/datasets/vikramche/trimmed-scene-video/trimmed/cross_scene_video"
    ),
    "sensors": Path(
        "/kaggle/input/datasets/vikramche/mmact-trimmed/trimmed_sensor/sensor"
    ),
    "splits": Path(
        "/kaggle/input/datasets/vikramche/mmact-trimmed/trimmed_splits/splits"
    ),
}

BASE.mkdir(parents=True, exist_ok=True)

for relative, source in SOURCES.items():
    destination = BASE / relative
    destination.parent.mkdir(parents=True, exist_ok=True)

    if not source.exists():
        raise FileNotFoundError(f"Source missing: {source}")

    if destination.is_symlink():
        destination.unlink()
    elif destination.exists():
        raise FileExistsError(f"Destination already exists: {destination}")

    destination.symlink_to(source, target_is_directory=True)
    print(f"LINKED: {relative} -> {source}")

print("\nCanonical dataset root:", BASE)
print("Video:", (BASE / "video/cross_scene").exists())
print("Sensors:", (BASE / "sensors").exists())
print("Splits:", (BASE / "splits").exists())

LINKED: video/cross_scene -> /kaggle/input/datasets/vikramche/trimmed-scene-video/trimmed/cross_scene_video
LINKED: sensors -> /kaggle/input/datasets/vikramche/mmact-trimmed/trimmed_sensor/sensor
LINKED: splits -> /kaggle/input/datasets/vikramche/mmact-trimmed/trimmed_splits/splits

Canonical dataset root: /kaggle/working/data/genuine_mmact/trimmed
Video: True
Sensors: True
Splits: True


In [14]:
import os
import inspect
from pathlib import Path

os.chdir("/kaggle/working/HAR-MMAct")

from src.data.indexer import MMActIndexer

print("=== INDEXER METHODS ===")
for name in dir(MMActIndexer):
    if not name.startswith("_"):
        print(name)

print("\n=== TRAIN SCRIPT INDEXING CALL ===")
lines = Path("scripts/train.py").read_text().splitlines()

for i, line in enumerate(lines):
    if "indexer." in line or "usable_samples" in line:
        print(f"{i+1}: {line.strip()}")

print("\n=== EXPERIMENT CHECKPOINT ===")
p = Path("outputs/original_cnn/checkpoints/best_model_original_cnn.pth")
print("Existing checkpoint:", p.exists())

=== INDEXER METHODS ===
SENSOR_NAME_MAP
find_files_recursively
index
normalize_action_name
normalize_identifier
parse_sensor_path
parse_video_path

=== TRAIN SCRIPT INDEXING CALL ===
275: report = indexer.index()
278: usable_samples = []
291: usable_samples.append(sample)
293: print(f'Usable samples: {len(usable_samples)}')
294: print(f'Excluded samples: {len(report.paired_samples) - len(usable_samples)}')
318: train_samples, val_samples, test_samples = splitter.split(usable_samples)

=== EXPERIMENT CHECKPOINT ===
Existing checkpoint: False


## Section 14: Validation Benchmark and Single Final Test Evaluation
All available checkpoints are first evaluated on the validation split. The best checkpoint is selected by validation Macro-F1. Only that selected checkpoint is evaluated on the held-out test split. Evaluation uses the same required-sensor filter and split construction as training.


In [ ]:
# Section 14: Validation benchmark for all baselines and upgrade candidates.
import json
import subprocess
import sys
from pathlib import Path

candidate_models = [
    ("InceptionTime (Sensor-Only)", "outputs/sensor_only/checkpoints/best_model_sensor.pth", "outputs/sensor_only"),
    ("Multimodal Fusion (Concat)", "outputs/multimodal_fusion/checkpoints/best_model_multimodal.pth", "outputs/multimodal_fusion"),
    ("MobileNetV2 Fine-Tuned", "outputs/mobilenet_v2/checkpoints/best_model_mobilenet_v2.pth", "outputs/mobilenet_v2"),
    ("Improved Regularized CNN", "outputs/improved_cnn/checkpoints/best_model_cnn.pth", "outputs/improved_cnn"),
    ("Tuned Custom CNN", "outputs/tuned_cnn/checkpoints/best_model_tuned_cnn.pth", "outputs/tuned_cnn"),
    ("Original Custom CNN", "outputs/original_cnn/checkpoints/best_model_original_cnn.pth", "outputs/original_cnn"),
    ("TCN Sensor-Only", "outputs/tcn_sensor_only/checkpoints/best_model_tcn_sensor.pth", "outputs/tcn_sensor_only"),
    ("TCN + Gated Fusion", "outputs/tcn_gated_fusion/checkpoints/best_model_tcn_gated_fusion.pth", "outputs/tcn_gated_fusion"),
]

eval_results = {}
candidate_records = []

for model_name, checkpoint, output_dir in candidate_models:
    checkpoint_path = Path(checkpoint)
    if not checkpoint_path.is_file():
        print(f"[NOT RUN / MISSING CHECKPOINT] {model_name}: {checkpoint}")
        candidate_records.append({"model": model_name, "status": "MISSING_CHECKPOINT"})
        continue

    output_file = Path(output_dir) / "validation_metrics.json"
    command = [
        sys.executable, "scripts/evaluate.py",
        "--checkpoint", str(checkpoint_path),
        "--data_root", "/kaggle/working/data/genuine_mmact",
        "--eval_split", "val",
        "--output_file", str(output_file),
        "--plots_dir", str(Path(output_dir) / "validation_plots"),
        "--cache_dir", str(CACHE_ROOT),
    ]
    result = subprocess.run(command, capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        candidate_records.append({"model": model_name, "status": "EVALUATION_ERROR"})
        continue
    if not output_file.is_file():
        candidate_records.append({"model": model_name, "status": "MISSING_METRICS"})
        continue

    metrics_data = json.loads(output_file.read_text())
    active_mode = metrics_data.get("mode", "multimodal")
    metrics = metrics_data.get(active_mode, metrics_data.get("metrics"))
    if not isinstance(metrics, dict) or "macro_f1" not in metrics:
        candidate_records.append({"model": model_name, "status": "INVALID_METRICS"})
        continue

    metrics = dict(metrics)
    metrics.update({
        "_name": model_name,
        "_mode": active_mode,
        "_split": metrics_data.get("eval_split_description", "Validation Split"),
        "_samples": metrics_data.get("eval_samples_count", "N/A"),
        "_checkpoint": str(checkpoint_path),
        "_metrics_file": str(output_file),
    })
    eval_results[model_name] = metrics
    candidate_records.append({
        "model": model_name,
        "status": "VALIDATION_EVALUATED",
        "validation_macro_f1": float(metrics["macro_f1"]),
        "validation_accuracy": float(metrics.get("accuracy", 0.0)),
    })
    print(f"[VALIDATION] {model_name}: Macro-F1={metrics['macro_f1']:.4f}, "
          f"Accuracy={metrics.get('accuracy', 0.0)*100:.2f}%")

if not eval_results:
    raise RuntimeError("No model produced valid validation metrics; do not evaluate the test set.")

selected_name = max(eval_results, key=lambda name: eval_results[name]["macro_f1"])
selected_metrics = eval_results[selected_name]
selected_checkpoint = selected_metrics["_checkpoint"]
print("\nSELECTED USING VALIDATION MACRO-F1 ONLY")
print("Model:", selected_name)
print("Validation Macro-F1:", round(selected_metrics["macro_f1"], 4))

# Evaluate exactly one selected checkpoint on the held-out test split.
selected_output_dir = str(Path(selected_checkpoint).parents[1])
test_metrics_file = Path(selected_output_dir) / "test_metrics.json"
test_command = [
    sys.executable, "scripts/evaluate.py",
    "--checkpoint", selected_checkpoint,
    "--data_root", "/kaggle/working/data/genuine_mmact",
    "--eval_split", "test",
    "--output_file", str(test_metrics_file),
    "--plots_dir", str(Path(selected_output_dir) / "test_plots"),
    "--cache_dir", str(CACHE_ROOT),
]
test_result = subprocess.run(test_command, capture_output=True, text=True)
print(test_result.stdout)
if test_result.returncode != 0:
    print(test_result.stderr)
    raise RuntimeError("Final held-out test evaluation failed.")
if not test_metrics_file.is_file():
    raise FileNotFoundError("Final test metrics file was not created.")

test_data = json.loads(test_metrics_file.read_text())
test_mode = test_data.get("mode", "multimodal")
final_test_metrics = test_data.get(test_mode, test_data.get("metrics"))
if not isinstance(final_test_metrics, dict):
    raise RuntimeError("Final test metrics JSON has an unexpected structure.")

final_test_record = {
    "selected_model": selected_name,
    "selection_metric": "validation_macro_f1",
    "validation_macro_f1": float(selected_metrics["macro_f1"]),
    "test_metrics": final_test_metrics,
    "test_split": test_data.get("eval_split_description", "Held-Out Test Split"),
    "test_samples": test_data.get("eval_samples_count"),
    "checkpoint": selected_checkpoint,
    "metrics_file": str(test_metrics_file),
}
Path("outputs").mkdir(exist_ok=True)
Path("outputs/final_model_selection.json").write_text(json.dumps(final_test_record, indent=2))
Path("outputs/validation_candidate_results.json").write_text(json.dumps(candidate_records, indent=2))
print("\nFINAL HELD-OUT TEST RESULT — selected model only")
print("Model:", selected_name)
print("Test accuracy:", round(final_test_metrics.get("accuracy", 0.0)*100, 2), "%")
print("Test Macro-F1:", round(final_test_metrics.get("macro_f1", 0.0), 4))


In [ ]:
from pathlib import Path
import shutil
from datetime import datetime

# Project paths
PROJECT_ROOT = Path("/kaggle/working/HAR-MMAct")
OUTPUTS = PROJECT_ROOT / "outputs"

# Create timestamped baseline backup
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
BACKUP_ROOT = Path("/kaggle/working") / f"HAR_MMAct_baseline_backup_{timestamp}"
BACKUP_ROOT.mkdir(parents=True, exist_ok=True)

# Preserve all six experiment directories and plots
folders = [
    "sensor_only",
    "multimodal_fusion",
    "mobilenet_v2",
    "improved_cnn",
    "tuned_cnn",
    "original_cnn",
    "plots",
]

print("=== BASELINE BACKUP ===")

for folder in folders:
    source = OUTPUTS / folder
    destination = BACKUP_ROOT / folder

    if source.exists():
        shutil.copytree(source, destination, dirs_exist_ok=True)
        file_count = sum(1 for p in destination.rglob("*") if p.is_file())
        print(f"[BACKED UP] {folder}: {file_count} files")
    else:
        print(f"[WARNING] Missing folder: {source}")

print("\nBackup location:")
print(BACKUP_ROOT)

# Verify backup
print("\n=== BACKUP VERIFICATION ===")

for folder in folders:
    original = OUTPUTS / folder
    backup = BACKUP_ROOT / folder

    if original.exists() and backup.exists():
        original_files = {
            str(p.relative_to(original))
            for p in original.rglob("*") if p.is_file()
        }
        backup_files = {
            str(p.relative_to(backup))
            for p in backup.rglob("*") if p.is_file()
        }

        status = "PASSED" if original_files == backup_files else "FAILED"
        print(f"{folder}: {status} | Original: {len(original_files)}, Backup: {len(backup_files)}")
    else:
        print(f"{folder}: NOT VERIFIED")

## Section 15: Comprehensive Publication Visualizations & Galleries
Generates cross-model benchmark comparisons, modality ablation bar charts, annotated training trajectories, learning rate schedules, generalization gaps, confusion matrices, ROC curves, and PR curves across all 6 models.


In [ ]:
# Section 15: Comprehensive Publication Visualizations & Galleries
import json
from pathlib import Path
from IPython.display import Image, display
from src.utils.plots import (
    plot_cross_model_comparison,
    plot_modality_ablation,
)

plots_output = Path('outputs/plots')
plots_output.mkdir(parents=True, exist_ok=True)

# 1. Cross-Model Comparison Bar Chart
if eval_results:
    plot_cross_model_comparison(eval_results, plots_output / 'cross_model_comparison.png', split_name='Validation Split')
    print('Generated outputs/plots/cross_model_comparison.png')

# 2. Modality-Model Comparison (independently trained models; not a controlled input-ablation experiment)
ablation_summary = {}
if 'MobileNetV2 Fine-Tuned' in eval_results:
    ablation_summary['Video-Only (MobileNetV2)'] = eval_results['MobileNetV2 Fine-Tuned']
elif 'Improved Regularized CNN' in eval_results:
    ablation_summary['Video-Only (Improved CNN)'] = eval_results['Improved Regularized CNN']
elif 'Original Custom CNN' in eval_results:
    ablation_summary['Video-Only (Original CNN)'] = eval_results['Original Custom CNN']
if 'InceptionTime (Sensor-Only)' in eval_results:
    ablation_summary['Sensor-Only (InceptionTime)'] = eval_results['InceptionTime (Sensor-Only)']
if 'Multimodal Fusion (Video+Sensor)' in eval_results:
    ablation_summary['Multimodal Fusion'] = eval_results['Multimodal Fusion (Video+Sensor)']

if len(ablation_summary) >= 2:
    plot_modality_ablation(ablation_summary, plots_output / 'modality_ablation_comparison.png', split_name='Validation Split')
    print('Generated outputs/plots/modality_ablation_comparison.png')

# Display publication gallery
gallery_plots = [
    ('Validation Model Comparison (All Available Candidates)', 'outputs/plots/cross_model_comparison.png'),
    ('Modality-Model Comparison (Independently Trained Video, Sensor, and Multimodal Models)', 'outputs/plots/modality_ablation_comparison.png'),
    ('Hyperparameter Tuning Trials Comparison', 'outputs/plots/hyperparameter_trials.png'),
    ('MobileNetV2: Train vs Val Accuracy (Annotated 50 Epochs)', 'outputs/mobilenet_v2/plots/train_vs_val_accuracy_annotated_mobilenet_v2.png'),
    ('MobileNetV2: Train vs Val Loss (Annotated 50 Epochs)', 'outputs/mobilenet_v2/plots/train_vs_val_loss_annotated_mobilenet_v2.png'),
    ('MobileNetV2: Learning Rate Schedule Trajectory', 'outputs/mobilenet_v2/plots/learning_rate_schedule_mobilenet_v2.png'),
    ('MobileNetV2: Generalization Gap Across Epochs', 'outputs/mobilenet_v2/plots/generalization_gap_mobilenet_v2.png'),
    ('MobileNetV2: Test Confusion Matrix', 'outputs/mobilenet_v2/plots/test_confusion_matrix_mobilenet_v2.png'),
    ('MobileNetV2: Standalone Per-Class F1-Score', 'outputs/mobilenet_v2/plots/test_f1_bars_mobilenet_v2.png'),
    ('MobileNetV2: Multiclass OvR ROC Curves', 'outputs/mobilenet_v2/plots/test_roc_ovr_mobilenet_v2.png'),
    ('MobileNetV2: Multiclass Precision-Recall Curves', 'outputs/mobilenet_v2/plots/test_pr_curves_mobilenet_v2.png'),
    ('Original CNN: Train vs Val Accuracy (Annotated)', 'outputs/original_cnn/plots/train_vs_val_accuracy_annotated_original_cnn.png'),
    ('Improved CNN: Train vs Val Accuracy (Annotated)', 'outputs/improved_cnn/plots/train_vs_val_accuracy_annotated_custom_cnn.png'),
    ('Multimodal Fusion: Test Confusion Matrix', 'outputs/multimodal_fusion/plots/test_confusion_matrix_mobilenet_v2.png'),
]

for title, p in gallery_plots:
    if Path(p).exists():
        print('\n' + '=' * 75)
        print(title)
        print('=' * 75)
        display(Image(filename=p))


In [ ]:
# Section 15A: Gallery File Verification

from pathlib import Path

gallery_plots = [
    'outputs/plots/cross_model_comparison.png',
    'outputs/plots/modality_ablation_comparison.png',
    'outputs/plots/hyperparameter_trials.png',
    'outputs/mobilenet_v2/plots/train_vs_val_accuracy_annotated_mobilenet_v2.png',
    'outputs/mobilenet_v2/plots/train_vs_val_loss_annotated_mobilenet_v2.png',
    'outputs/mobilenet_v2/plots/learning_rate_schedule_mobilenet_v2.png',
    'outputs/mobilenet_v2/plots/generalization_gap_mobilenet_v2.png',
    'outputs/mobilenet_v2/plots/test_confusion_matrix_mobilenet_v2.png',
    'outputs/mobilenet_v2/plots/test_f1_bars_mobilenet_v2.png',
    'outputs/mobilenet_v2/plots/test_roc_ovr_mobilenet_v2.png',
    'outputs/mobilenet_v2/plots/test_pr_curves_mobilenet_v2.png',
    'outputs/original_cnn/plots/train_vs_val_accuracy_annotated_original_cnn.png',
    'outputs/improved_cnn/plots/train_vs_val_accuracy_annotated_custom_cnn.png',
    'outputs/multimodal_fusion/plots/test_confusion_matrix_mobilenet_v2.png',
]

for p in gallery_plots:
    status = "FOUND" if Path(p).is_file() else "MISSING"
    print(f"[{status}] {p}")

found = sum(Path(p).is_file() for p in gallery_plots)

print(f"\nGallery files: {found}/{len(gallery_plots)}")

## Section 16: Sample Multimodal Inference Demonstration
Runs standalone single-sample multimodal inference using the trained fusion model and prints predicted class probabilities.


In [ ]:
# Section 16: Sample Multimodal Inference Demonstration
from pathlib import Path
import subprocess
import sys
from src.data.indexer import MMActIndexer

print("Indexing dataset to locate a complete paired sample for a smoke test.")
indexer = MMActIndexer(
    dataset_root=canonical_data_dir,
    required_sensor_modalities=["acc_phone", "acc_watch", "gyro"],
)
demo_report = indexer.index()

demo_sample = None
for sample in demo_report.paired_samples:
    required = ("acc_phone", "acc_watch", "gyro")
    if not Path(sample.video_path).is_file():
        continue
    if not all(
        key in sample.sensor_paths
        and Path(sample.sensor_paths[key]).is_file()
        and Path(sample.sensor_paths[key]).stat().st_size > 10
        for key in required
    ):
        continue
    demo_sample = sample
    break

if demo_sample is None:
    raise FileNotFoundError(
        "No valid video + acc_phone + acc_watch + gyro sample was found."
    )

sample_video = str(demo_sample.video_path)
sample_phone = str(demo_sample.sensor_paths["acc_phone"])
sample_watch = str(demo_sample.sensor_paths["acc_watch"])
sample_gyro = str(demo_sample.sensor_paths["gyro"])

print("Smoke-test sample:", demo_sample.sample_id)
print("Ground-truth activity:", demo_sample.action)
print("Subject:", demo_sample.subject)
print("Note: this is an inference smoke test, not a held-out test metric.")

cmd = [
    sys.executable, "scripts/infer.py",
    "--video", sample_video,
    "--sensor_phone", sample_phone,
    "--sensor_watch", sample_watch,
    "--sensor_gyro", sample_gyro,
    "--checkpoint",
    selected_checkpoint,
]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("STDERR:")
    print(result.stderr)
print("Exit code:", result.returncode)
if result.returncode != 0:
    raise RuntimeError("Multimodal inference smoke test failed.")


In [ ]:
# Section 17A: Packaging Pre-flight Verification

from pathlib import Path

expected_models = [
    'InceptionTime (Sensor-Only)',
    'Multimodal Fusion (Video+Sensor)',
    'MobileNetV2 Fine-Tuned',
    'Improved Regularized CNN',
    'Tuned Custom CNN',
    'Original Custom CNN',
]

print("BENCHMARK RESULT VERIFICATION")
print("=" * 65)

missing = []

for name in expected_models:
    if name in eval_results:
        m = eval_results[name]
        print(
            f"[FOUND] {name}: "
            f"Accuracy={m.get('accuracy', 0)*100:.2f}% | "
            f"Macro F1={m.get('macro_f1', 0):.4f}"
        )
    else:
        missing.append(name)
        print(f"[MISSING] {name}")

print("\nCheckpoint verification")
print("=" * 65)

checkpoint_paths = [
    'outputs/sensor_only/checkpoints/best_model_sensor.pth',
    'outputs/multimodal_fusion/checkpoints/best_model_multimodal.pth',
    'outputs/mobilenet_v2/checkpoints/best_model_mobilenet_v2.pth',
    'outputs/improved_cnn/checkpoints/best_model_cnn.pth',
    'outputs/tuned_cnn/checkpoints/best_model_tuned_cnn.pth',
    'outputs/original_cnn/checkpoints/best_model_original_cnn.pth',
]

for p in checkpoint_paths:
    path = Path(p)
    print(f"[{'FOUND' if path.is_file() else 'MISSING'}] {p}")

print("\nMissing benchmark entries:", missing)

## Section 17: Validation-Driven Selection, One-Time Test Evaluation & Evidence-Based Packaging
Creates a validation benchmark for all available checkpoints, selects by validation Macro-F1, reports test metrics only for the selected checkpoint, and packages only artifacts that exist.


In [ ]:
# Section 17: Evidence-Based Benchmark Summary, Status Manifest & Packaging
import json
import shutil
import hashlib
import pandas as pd
from pathlib import Path

Path("outputs").mkdir(exist_ok=True)
validation_rows = []
for model_name, metrics in eval_results.items():
    validation_rows.append({
        "Model": model_name,
        "Status": "VALIDATION_EVALUATED",
        "Validation Accuracy (%)": round(metrics.get("accuracy", 0.0) * 100.0, 2),
        "Validation Macro Precision": round(metrics.get("macro_precision", 0.0), 4),
        "Validation Macro Recall": round(metrics.get("macro_recall", 0.0), 4),
        "Validation Macro F1": round(metrics.get("macro_f1", 0.0), 4),
        "Validation Samples": metrics.get("_samples", "N/A"),
        "Checkpoint": metrics.get("_checkpoint", "N/A"),
    })

validation_df = pd.DataFrame(validation_rows).sort_values(
    "Validation Macro F1", ascending=False
)
validation_df.to_csv("outputs/benchmark_validation.csv", index=False)
print("VALIDATION BENCHMARK — used for model selection")
print(validation_df.to_string(index=False))

final_test_df = pd.DataFrame([{
    "Selected Model": selected_name,
    "Selection Criterion": "Highest validation Macro-F1",
    "Validation Macro F1": round(selected_metrics["macro_f1"], 4),
    "Test Accuracy (%)": round(final_test_metrics.get("accuracy", 0.0) * 100.0, 2),
    "Test Macro Precision": round(final_test_metrics.get("macro_precision", 0.0), 4),
    "Test Macro Recall": round(final_test_metrics.get("macro_recall", 0.0), 4),
    "Test Macro F1": round(final_test_metrics.get("macro_f1", 0.0), 4),
    "Test Samples": test_data.get("eval_samples_count", "N/A"),
    "Checkpoint": selected_checkpoint,
}])
final_test_df.to_csv("outputs/benchmark_final_test.csv", index=False)
print("\nFINAL TEST — one selected checkpoint")
print(final_test_df.to_string(index=False))

status_manifest = {
    "selection_rule": "highest validation macro_f1",
    "selected_model": selected_name,
    "validation_candidates": candidate_records,
    "final_test_evaluation": {
        "status": "COMPLETED",
        "model": selected_name,
        "metrics_file": str(test_metrics_file),
        "split": test_data.get("eval_split_description"),
        "sample_count": test_data.get("eval_samples_count"),
    },
    "warning": "Do not report unmeasured models as completed. Test metrics are for the selected checkpoint only."
}
Path("outputs/experiment_status.json").write_text(json.dumps(status_manifest, indent=2))
Path("outputs/final_model_selection.json").write_text(json.dumps(final_test_record, indent=2))

# Package only artifacts that actually exist.
results_dir = Path("/kaggle/working/har_mmact_results")
results_dir.mkdir(parents=True, exist_ok=True)
for sub in [
    "sensor_only", "multimodal_fusion", "mobilenet_v2", "improved_cnn",
    "tuned_cnn", "original_cnn", "tcn_sensor_only", "tcn_gated_fusion",
    "tuning", "plots", "manifests", "reports",
]:
    source_dir = Path("outputs") / sub
    if source_dir.exists():
        shutil.copytree(source_dir, results_dir / sub, dirs_exist_ok=True)

for name in [
    "benchmark_validation.csv", "benchmark_final_test.csv",
    "experiment_status.json", "final_model_selection.json",
    "validation_candidate_results.json",
]:
    source_file = Path("outputs") / name
    if source_file.exists():
        shutil.copy2(source_file, results_dir / name)

archive_base = "/kaggle/working/har_mmact_results"
shutil.make_archive(archive_base, "zip", results_dir)
zip_path = Path(archive_base + ".zip")
sha256 = hashlib.sha256(zip_path.read_bytes()).hexdigest()
print("\nCreated:", zip_path)
print("Size (MB):", round(zip_path.stat().st_size / (1024 ** 2), 2))
print("SHA-256:", sha256)
print("The archive is valid only after this cell finishes successfully.")


## Section 18: (Optional) Extended Benchmarks (100 and 150 Epochs)
This section is **strictly optional** and intended for extended GPU quota or overnight execution.
Includes a runtime feasibility estimator to ensure sufficient Kaggle session time is available before starting extended runs.


In [ ]:
# Section 18: (Optional) Extended Benchmarks (100 and 150 Epochs)
# Set RUN_EXTENDED = True to enable extended training beyond the required 50-epoch baseline.
RUN_EXTENDED = False
EXTENDED_EPOCHS = 100  # Set to 100 or 150

if RUN_EXTENDED:
    # Check runtime feasibility based on previous 50-epoch multimodal run
    summary_candidates = list(Path('outputs/multimodal_fusion').glob('**/training_summary*.json'))
    summary_p = summary_candidates[0] if summary_candidates else None
    if summary_p is not None and summary_p.exists():
        with open(summary_p) as f:
            s_data = json.load(f)
        avg_ep_sec = s_data.get('avg_epoch_duration_seconds', 15.0)
        est_total_min = (avg_ep_sec * EXTENDED_EPOCHS) / 60.0
        print(f'Estimated training duration for {EXTENDED_EPOCHS} epochs: {est_total_min:.1f} minutes ({avg_ep_sec:.1f}s/epoch)')

    print(f'=== Starting Optional Extended Training ({EXTENDED_EPOCHS} Epochs) ===')
    out_extended = f'outputs/multimodal_extended_{EXTENDED_EPOCHS}ep'
    Path(out_extended).mkdir(parents=True, exist_ok=True)
    !python scripts/train.py \
        --data_root /kaggle/working/data/genuine_mmact \
        --mode multimodal \
        --video_backbone mobilenet_v2 \
        --sensor_backbone inception_time \
        --epochs {EXTENDED_EPOCHS} \
        --min_epochs {EXTENDED_EPOCHS} \
        --batch_size 8 \
        --lr 0.0001 \
        --weight_decay 0.0005 \
        --split_strategy subject \
        --checkpoint_name best_model_multimodal_extended.pth \
        --output_dir {out_extended} \
        --cache_dir {CACHE_ROOT} \
        --auto_resume \
        --skip_if_completed
else:
    print('Optional extended training skipped. Required 50-epoch benchmark is complete.')


In [102]:
from kaggle_secrets import UserSecretsClient
import subprocess
import os

token = UserSecretsClient().get_secret("GH_TOKEN")

if not token:
    raise RuntimeError("GH_TOKEN is empty or unavailable.")

result = subprocess.run(
    [
        "git", "ls-remote",
        "https://github.com/YOUR_USERNAME/HAR-MMAct.git"
    ],
    env={**os.environ, "GIT_ASKPASS": "echo"},
    capture_output=True,
    text=True
)

print("GitHub remote check completed.")
print("Return code:", result.returncode)

if result.returncode == 0:
    print("Repository is accessible.")
else:
    print("Repository access not confirmed.")
    print("Check repository URL and authentication settings.")

GitHub remote check completed.
Return code: 128
Repository access not confirmed.
Check repository URL and authentication settings.


In [103]:
from kaggle_secrets import UserSecretsClient
import subprocess
import os
import base64

token = UserSecretsClient().get_secret("GH_TOKEN")

repo_url = input("Enter GitHub repository URL: ").strip()

if not repo_url.startswith("https://github.com/"):
    raise ValueError("Use an HTTPS GitHub repository URL.")

# Configure authentication only for this subprocess.
auth = base64.b64encode(
    f"x-access-token:{token}".encode()
).decode()

env = os.environ.copy()
env["GIT_CONFIG_COUNT"] = "1"
env["GIT_CONFIG_KEY_0"] = "http.https://github.com/.extraheader"
env["GIT_CONFIG_VALUE_0"] = f"AUTHORIZATION: basic {auth}"

result = subprocess.run(
    ["git", "ls-remote", repo_url],
    env=env,
    capture_output=True,
    text=True
)

print("Return code:", result.returncode)

if result.returncode == 0:
    print("SUCCESS: GitHub repository access verified.")
else:
    print("FAILED: Repository access not verified.")
    print(result.stderr[:1000])

Enter GitHub repository URL:  https://github.com/CHENNAMSETTYVIKRAM/HAR-MMAct.git


Return code: 0
SUCCESS: GitHub repository access verified.


In [104]:
from pathlib import Path
import subprocess

repo = Path("/kaggle/working/HAR-MMAct")

print("Repository exists:", repo.exists())

if not repo.exists():
    raise RuntimeError("HAR-MMAct repository folder not found.")

print("\nGit branch and status:")
subprocess.run(["git", "branch", "--show-current"], cwd=repo)
subprocess.run(["git", "status", "--short"], cwd=repo)

print("\nFolder sizes:")
for name in ["src", "scripts", "configs", "outputs", "docs", "notebooks"]:
    folder = repo / name
    if folder.exists():
        total = sum(
            f.stat().st_size for f in folder.rglob("*")
            if f.is_file()
        )
        print(f"{name}: {total / (1024**2):.2f} MB")
    else:
        print(f"{name}: not present")

Repository exists: True

Git branch and status:
master
 D outputs/metrics.json
 D outputs/plots/test_confusion_matrix.png
 D outputs/plots/test_pr_curves.png
 D outputs/plots/test_roc_ovr.png
 D outputs/plots/training_curves.png
 D outputs/plots/val_confusion_matrix.png
 D outputs/plots/val_pr_curves.png
 D outputs/plots/val_roc_ovr.png
 D outputs/training_summary.json
 M scripts/evaluate.py
 M scripts/infer.py
 M scripts/train.py
 M src/data/sensor_loader.py
 M src/models/__init__.py
 M src/models/fusion_model.py
 M src/models/sensor_branch.py
?? outputs/exp1_multimodal/
?? outputs/exp2_sensor_only/
?? outputs/mobilenet_v2/
?? outputs/multimodal_fusion/
?? outputs/tcn_gated_fusion/
?? outputs/tcn_sensor_only/
?? scripts/train.py.bak
?? scripts/train.py.before_indent_fix.bak
?? scripts/train.py.before_resume_fix
?? scripts/train.py.indentation_broken.bak

Folder sizes:
src: 0.50 MB
scripts: 0.38 MB
configs: not present
outputs: 563.62 MB
docs: 0.05 MB
notebooks: 0.26 MB


In [105]:
from pathlib import Path
import subprocess

repo = Path("/kaggle/working/HAR-MMAct")

# Confirm remote
subprocess.run(["git", "remote", "-v"], cwd=repo)

# Check files larger than GitHub's 100 MB limit
print("\nFiles larger than 100 MB:")
for f in (repo / "outputs").rglob("*"):
    if f.is_file() and f.stat().st_size > 100 * 1024 * 1024:
        print(f, f"{f.stat().st_size / 1024**2:.1f} MB")

# Check ignored output files
print("\nGit ignore check:")
subprocess.run(
    ["git", "check-ignore", "-v",
     "outputs/exp1_multimodal/checkpoints/exp1_multimodal_best.pth"],
    cwd=repo
)

# Confirm current branch
print("\nBranch:")
subprocess.run(["git", "branch", "--show-current"], cwd=repo)

print("\nPreparation check complete. No files changed.")

origin	https://github.com/CHENNAMSETTYVIKRAM/HAR-MMAct.git (fetch)
origin	https://github.com/CHENNAMSETTYVIKRAM/HAR-MMAct.git (push)

Files larger than 100 MB:

Git ignore check:
.gitignore:20:*.pth	outputs/exp1_multimodal/checkpoints/exp1_multimodal_best.pth

Branch:
master

Preparation check complete. No files changed.


In [106]:
from pathlib import Path
import subprocess

repo = Path("/kaggle/working/HAR-MMAct")

# Protect existing tracked output files from accidental deletion.
subprocess.run(
    ["git", "restore", "--worktree", "--", "outputs/metrics.json",
     "outputs/training_summary.json", "outputs/plots"],
    cwd=repo,
    check=False
)

# Create a backup branch without changing master.
branch = "kaggle-training-backup"

result = subprocess.run(
    ["git", "branch", branch],
    cwd=repo,
    capture_output=True,
    text=True
)

if result.returncode != 0 and "already exists" not in result.stderr:
    raise RuntimeError(result.stderr)

print("Backup branch prepared:", branch)
print("\nCurrent branch:")
subprocess.run(["git", "branch", "--show-current"], cwd=repo)

print("\nStatus:")
subprocess.run(["git", "status", "--short"], cwd=repo)

Backup branch prepared: kaggle-training-backup

Current branch:
master

Status:
 M scripts/evaluate.py
 M scripts/infer.py
 M scripts/train.py
 M src/data/sensor_loader.py
 M src/models/__init__.py
 M src/models/fusion_model.py
 M src/models/sensor_branch.py
?? outputs/exp1_multimodal/
?? outputs/exp2_sensor_only/
?? outputs/mobilenet_v2/
?? outputs/multimodal_fusion/
?? outputs/tcn_gated_fusion/
?? outputs/tcn_sensor_only/
?? scripts/train.py.bak
?? scripts/train.py.before_indent_fix.bak
?? scripts/train.py.before_resume_fix
?? scripts/train.py.indentation_broken.bak


CompletedProcess(args=['git', 'status', '--short'], returncode=0)

In [108]:
import subprocess
from pathlib import Path

repo = Path("/kaggle/working/HAR-MMAct")

# Set Git identity for this repository only
subprocess.run(
    ["git", "config", "user.name", "CHENNAMSETTYVIKRAM"],
    cwd=repo,
    check=True
)

subprocess.run(
    ["git", "config", "user.email", "vikramchennams@gmail.com"],
    cwd=repo,
    check=True
)

# Retry commit
result = subprocess.run(
    ["git", "commit", "-m",
     "Backup completed HAR-MMAct experiments and source"],
    cwd=repo,
    capture_output=True,
    text=True
)

print(result.stdout)
if result.returncode:
    print(result.stderr)
else:
    print("COMMIT SUCCESSFUL")

[kaggle-training-backup bd44ce9] Backup completed HAR-MMAct experiments and source
 205 files changed, 14464 insertions(+), 23 deletions(-)
 create mode 100644 outputs/exp1_multimodal/experiment_manifest.json
 create mode 100644 outputs/exp1_multimodal/experiment_manifest_mobilenet_v2.json
 create mode 100644 outputs/exp1_multimodal/history.json
 create mode 100644 outputs/exp1_multimodal/history_mobilenet_v2.csv
 create mode 100644 outputs/exp1_multimodal/history_mobilenet_v2.json
 create mode 100644 outputs/exp1_multimodal/plots/generalization_gap.png
 create mode 100644 outputs/exp1_multimodal/plots/generalization_gap_mobilenet_v2.png
 create mode 100644 outputs/exp1_multimodal/plots/learning_rate_schedule.png
 create mode 100644 outputs/exp1_multimodal/plots/learning_rate_schedule_mobilenet_v2.png
 create mode 100644 outputs/exp1_multimodal/plots/train_vs_val_accuracy.png
 create mode 100644 outputs/exp1_multimodal/plots/train_vs_val_accuracy_annotated_mobilenet_v2.png
 create mode

In [109]:
from pathlib import Path
from kaggle_secrets import UserSecretsClient
import subprocess, os, base64

repo = Path("/kaggle/working/HAR-MMAct")
token = UserSecretsClient().get_secret("GH_TOKEN")

auth = base64.b64encode(
    f"x-access-token:{token}".encode()
).decode()

env = os.environ.copy()
env["GIT_CONFIG_COUNT"] = "1"
env["GIT_CONFIG_KEY_0"] = "http.https://github.com/.extraheader"
env["GIT_CONFIG_VALUE_0"] = f"AUTHORIZATION: basic {auth}"

r = subprocess.run(
    ["git", "push", "-u", "origin", "kaggle-training-backup"],
    cwd=repo,
    env=env,
    capture_output=True,
    text=True
)

print(r.stdout)
if r.returncode:
    raise RuntimeError(r.stderr)

print("GITHUB PUSH SUCCESSFUL")

branch 'kaggle-training-backup' set up to track 'origin/kaggle-training-backup'.

GITHUB PUSH SUCCESSFUL


In [115]:
from pathlib import Path

repo = Path("/kaggle/working/HAR-MMAct")

folders = [
    "src/models",
    "src/data",
    "scripts",
    "outputs"
]

for folder in folders:
    p = repo / folder
    files = [x for x in p.rglob("*") if x.is_file()] if p.exists() else []

    print(f"\n{folder}")
    print("Files:", len(files))

    for f in files[:8]:
        print(" -", f.relative_to(repo))

print("\n=== ALL BEST MODEL CHECKPOINTS ===")

for f in (repo / "outputs").rglob("best_model*.pth"):
    print(f"{f.relative_to(repo)} | {f.stat().st_size / (1024**2):.2f} MB")


src/models
Files: 8
 - src/models/video_branch.py
 - src/models/sensor_branch.py
 - src/models/fusion_model.py
 - src/models/__init__.py
 - src/models/__pycache__/video_branch.cpython-313.pyc
 - src/models/__pycache__/sensor_branch.cpython-313.pyc
 - src/models/__pycache__/__init__.cpython-313.pyc
 - src/models/__pycache__/fusion_model.cpython-313.pyc

src/data
Files: 17
 - src/data/dataset.py
 - src/data/video_loader.py
 - src/data/indexer.py
 - src/data/synthetic.py
 - src/data/stager.py
 - src/data/ingestion.py
 - src/data/splits.py
 - src/data/__init__.py

scripts
Files: 15
 - scripts/infer.py
 - scripts/tune.py
 - scripts/audit_dataset.py
 - scripts/train.py.indentation_broken.bak
 - scripts/generate_synthetic_data.py
 - scripts/evaluate.py
 - scripts/extract_genuine_data.py
 - scripts/train.py

outputs
Files: 274
 - outputs/training_summary.json
 - outputs/metrics.json
 - outputs/exp1_multimodal/history.json
 - outputs/exp1_multimodal/history_mobilenet_v2.json
 - outputs/exp1_mu

In [116]:
from pathlib import Path
import os
import shutil
import subprocess
import base64
import hashlib
from kaggle_secrets import UserSecretsClient

# ============================================================
# 1. PATHS AND CONFIGURATION
# ============================================================

WORK = Path("/kaggle/working")
REPO = WORK / "HAR-MMAct"
BRANCH = "kaggle-training-backup"

assert REPO.exists(), f"Repository not found: {REPO}"

# ============================================================
# 2. GIT HELPER
# ============================================================

def git(*args, check=True):
    r = subprocess.run(
        ["git", *args],
        cwd=REPO,
        capture_output=True,
        text=True
    )
    if check and r.returncode:
        raise RuntimeError(r.stderr or r.stdout)
    return r

# ============================================================
# 3. EXCLUSION RULES
# ============================================================

EXCLUDED_DIRS = {
    "__pycache__", ".ipynb_checkpoints",
    ".git", ".venv", "venv", "env",
    "cache", "caches", "dataset", "datasets",
    "genuine_mmact", ".virtual_documents"
}

EXCLUDED_FILES = {
    ".DS_Store", "Thumbs.db"
}

# ============================================================
# 4. COLLECT ROOT-LEVEL PROJECT OUTPUTS
# ============================================================

def copy_project_files(source, destination):
    source = Path(source)
    destination = Path(destination)

    if not source.exists() or source.resolve() == destination.resolve():
        return 0

    copied = 0

    for item in source.rglob("*"):
        if not item.is_file():
            continue

        relative = item.relative_to(source)

        if any(part in EXCLUDED_DIRS for part in relative.parts):
            continue

        if item.name in EXCLUDED_FILES:
            continue

        # Never copy non-best checkpoints.
        if item.suffix == ".pth" and not item.name.startswith("best_model"):
            continue

        target = destination / relative
        target.parent.mkdir(parents=True, exist_ok=True)

        # Preserve an existing destination unless source is newer.
        if target.exists() and target.stat().st_mtime >= item.stat().st_mtime:
            continue

        shutil.copy2(item, target)
        copied += 1

    return copied


print("=== COLLECTING PROJECT OUTPUTS ===")

root_outputs = WORK / "outputs"
repo_outputs = REPO / "outputs"

copied = copy_project_files(root_outputs, repo_outputs)

print("Root output files copied:", copied)

# ============================================================
# 5. COLLECT IMPORTANT ROOT-LEVEL PROJECT FILES
# ============================================================

root_files = [
    "README.md",
    "requirements.txt",
    "requirements-kaggle.txt",
    "sources.txt"
]

for name in root_files:
    src = WORK / name
    dst = REPO / name

    if src.is_file() and not dst.exists():
        shutil.copy2(src, dst)
        print("Collected:", name)

# ============================================================
# 6. ENSURE GIT IDENTITY
# ============================================================

if not git("config", "user.name", check=False).stdout.strip():
    git("config", "user.name", "HAR-MMAct Kaggle Backup")

if not git("config", "user.email", check=False).stdout.strip():
    git("config", "user.email", "kaggle-backup@users.noreply.github.com")

# ============================================================
# 7. UPDATE BRANCH
# ============================================================

git("fetch", "origin")
git("checkout", BRANCH)

# ============================================================
# 8. STAGE IMPORTANT PROJECT DIRECTORIES
# ============================================================

allowed_paths = [
    "src",
    "scripts",
    "docs",
    "notebooks",
    "outputs",
    "README.md",
    "requirements.txt",
    "requirements-kaggle.txt",
    "sources.txt",
    "tests",
    "configs"
]

for path in allowed_paths:
    if (REPO / path).exists():
        git("add", path)

# ============================================================
# 9. FORCE-ADD SELECTED BEST CHECKPOINTS
# ============================================================

checkpoint_count = 0

for checkpoint in (REPO / "outputs").rglob("best_model*.pth"):
    relative = str(checkpoint.relative_to(REPO))
    git("add", "-f", relative)
    checkpoint_count += 1

print("Selected best checkpoints staged:", checkpoint_count)

# ============================================================
# 10. VERIFY STAGED FILES AND EXCLUSIONS
# ============================================================

staged = git("diff", "--cached", "--name-only").stdout.splitlines()

print("\n=== STAGED FILE AUDIT ===")
print("Total staged files:", len(staged))

for f in staged:
    print(f)

# Refuse obvious dataset/cache uploads.
bad_terms = (
    "/genuine_mmact/",
    "/cache/",
    "/caches/",
    "/.virtual_documents/",
    "/__pycache__/"
)

bad_files = [
    f for f in staged
    if any(term in "/" + f for term in bad_terms)
]

if bad_files:
    raise RuntimeError(
        "Excluded data/cache paths were staged:\n" + "\n".join(bad_files)
    )

# ============================================================
# 11. COMMIT
# ============================================================

if staged:
    commit = git(
        "commit",
        "-m",
        "Sync HAR-MMAct project artifacts and selected checkpoints"
    )
    print("\n", commit.stdout)
else:
    print("\nNo new changes to commit.")

# ============================================================
# 12. PUSH TO GITHUB
# ============================================================

token = UserSecretsClient().get_secret("GH_TOKEN")

auth = base64.b64encode(
    f"x-access-token:{token}".encode()
).decode()

env = os.environ.copy()
env["GIT_CONFIG_COUNT"] = "1"
env["GIT_CONFIG_KEY_0"] = "http.https://github.com/.extraheader"
env["GIT_CONFIG_VALUE_0"] = f"AUTHORIZATION: basic {auth}"

push = subprocess.run(
    ["git", "push", "-u", "origin", BRANCH],
    cwd=REPO,
    env=env,
    capture_output=True,
    text=True
)

if push.returncode:
    raise RuntimeError(push.stderr or push.stdout)

print("\n========================================")
print("GITHUB SYNCHRONIZATION SUCCESSFUL")
print("========================================")
print("Branch:", BRANCH)
print("Files staged:", len(staged))
print("Best checkpoints staged:", checkpoint_count)
print(push.stdout)

# ============================================================
# 13. FINAL STATUS
# ============================================================

print("\n=== REMAINING GIT CHANGES ===")
print(git("status", "--short").stdout or "Working tree clean")

=== COLLECTING PROJECT OUTPUTS ===
Root output files copied: 2
Selected best checkpoints staged: 13

=== STAGED FILE AUDIT ===
Total staged files: 57
outputs/exp1_multimodal/checkpoints/best_model.pth
outputs/exp2_sensor_only/checkpoints/best_model.pth
outputs/exp2_sensor_only/checkpoints/best_model_sensor.pth
outputs/improved_cnn/checkpoints/best_model.pth
outputs/improved_cnn/checkpoints/best_model_cnn.pth
outputs/improved_cnn/experiment_manifest.json
outputs/improved_cnn/experiment_manifest_custom_cnn.json
outputs/improved_cnn/history.json
outputs/improved_cnn/history_custom_cnn.csv
outputs/improved_cnn/history_custom_cnn.json
outputs/improved_cnn/plots/generalization_gap.png
outputs/improved_cnn/plots/generalization_gap_custom_cnn.png
outputs/improved_cnn/plots/learning_rate_schedule.png
outputs/improved_cnn/plots/learning_rate_schedule_custom_cnn.png
outputs/improved_cnn/plots/train_vs_val_accuracy.png
outputs/improved_cnn/plots/train_vs_val_accuracy_annotated_custom_cnn.png
outpu

In [27]:
import os
import subprocess

os.chdir("/kaggle/working/HAR-MMAct")

subprocess.run(["git", "status", "--short"], check=True)
subprocess.run(["git", "pull", "origin", "master"], check=True)

print("\nLatest commit:")
subprocess.run(["git", "log", "-1", "--oneline"], check=True)

?? outputs/original_cnn_recovery/
Updating 2ccccfd..fe8aad4
Fast-forward
 src/config.py                |   5 +-
 src/data/sensor_loader.py    |  19 +++
 tests/test_accuracy_fixes.py | 354 +++++++++++++++++++++++++++++++++++++++++++
 3 files changed, 377 insertions(+), 1 deletion(-)
 create mode 100644 tests/test_accuracy_fixes.py

Latest commit:
fe8aad4 fix(data): preserve MMAct sensor alignment and add regression tests


From https://github.com/CHENNAMSETTYVIKRAM/HAR-MMAct
 * branch            master     -> FETCH_HEAD
   25d5f59..fe8aad4  master     -> origin/master


CompletedProcess(args=['git', 'log', '-1', '--oneline'], returncode=0)

In [ ]:
import os
from pathlib import Path

os.chdir("/kaggle/working/HAR-MMAct")

OUT = "outputs/mobilenet_v2_staged_v2"
CACHE = "/kaggle/working/har_mmact_cache"

Path(OUT).mkdir(parents=True, exist_ok=True)

!python scripts/train.py \
    --data_root /kaggle/working/data/genuine_mmact/trimmed \
    --video_backbone mobilenet_v2 \
    --sensor_backbone inception_time \
    --mode video_only \
    --epochs 50 \
    --min_epochs 50 \
    --batch_size 8 \
    --lr 0.0002 \
    --weight_decay 0.0001 \
    --dropout 0.20 \
    --staged_finetune \
    --unfreeze_epoch 5 \
    --unfreeze_lr 0.00005 \
    --split_strategy subject \
    --early_stopping_patience 0 \
    --checkpoint_name best_model_mobilenet_v2_staged.pth \
    --output_dir outputs/mobilenet_v2_staged_v2 \
    --cache_dir /kaggle/working/har_mmact_cache \
    --benchmark_warmup

 HAR_MMAct: Multimodal Human Activity Recognition Training Pipeline
 Mode            : video_only
 Video Backbone  : mobilenet_v2
 Sensor Backbone : inception_time
 Split Strategy  : subject
 Epochs          : 50 | Batch Size: 8 | LR: 0.0002
 Random Seed     : 42

[Step 1] Indexing dataset at: /kaggle/working/data/genuine_mmact/trimmed
Indexed samples: 5431
Usable samples: 5427
Excluded samples: 4
  - Total Videos Scanned   : 17820
  - Total Sensors Scanned  : 35685
  - Paired Samples Created : 5431
  - Unknown Labels Filtered: 36388

[Step 2] Partitioning into subject-independent splits...
 SPLIT AUDIT & INTEGRITY REPORT
Strategy                : subject
Official Split Protocol : False
Custom Test Partition   : False
Train Source File       : N/A (Rule-based)
Val Source File         : N/A (Rule-based)
Train Samples / Records : 3795 samples (595 recordings)
Val Samples / Records   : 780 samples (124 recordings)
Test Samples / Records  : 852 samples (132 recordings)
--------------------

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

print("=== DISK SPACE ===")
subprocess.run(["df", "-h", "/kaggle"])

print("\n=== TOP-LEVEL FOLDER SIZES ===")
for path in [
    Path("/kaggle/working"),
    Path("/kaggle/working/HAR-MMAct"),
    Path("/kaggle/working/data"),
    Path("/kaggle/working/har_mmact_cache"),
    Path("/kaggle/temp"),
]:
    if path.exists():
        result = subprocess.run(
            ["du", "-sh", str(path)],
            capture_output=True, text=True
        )
        print(result.stdout.strip())

print("\n=== LARGEST DIRECTORIES IN WORKING ===")
subprocess.run(
    "du -h --max-depth=2 /kaggle/working 2>/dev/null | sort -hr | head -35",
    shell=True
)

print("\n=== LARGEST FILES (TOP 30) ===")
subprocess.run(
    "find /kaggle/working -type f -printf '%s %p\\n' 2>/dev/null | sort -nr | head -30",
    shell=True
)